<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/08_SPP_GAN_Architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
# ==================================================================================================
# NOTEBOOK 08 — SPP-GAN ARCHITECTURE
# Research: A Unified Privacy-Preserving Framework for High-Fidelity Synthetic Data Generation
# Framework: SPP-GAN
# ==================================================================================================
#
# PURPOSE
# -------
# Define, validate, document, and persist the complete SPP-GAN neural architecture.
#
# This notebook:
#   1. Loads the canonical project configuration and feature schemas.
#   2. Loads Notebook 03 statistical profiles when available.
#   3. Defines the SPP-GAN input representation.
#   4. Defines latent space.
#   5. Defines Generator.
#   6. Defines Discriminator/Critic.
#   7. Defines numerical and categorical output mechanisms.
#   8. Defines interfaces for statistical guidance and privacy.
#   9. Initializes one architecture per registered dataset.
#  10. Performs parameter, tensor, forward, backward, and gradient tests.
#  11. Saves architecture/configuration artifacts for downstream notebooks.
#
# IMPORTANT
# ---------
# This is an ARCHITECTURE notebook.
#
# No model training is performed here.
# No privacy budget is consumed here.
# No synthetic data is generated here.
#
# Downstream:
#   Notebook 09 -> Statistical Guidance
#   Notebook 10 -> Differential Privacy
#   Notebook 11 -> Privacy Accounting
#   Notebook 12 -> SPP-GAN Training
#   Notebook 13 -> Synthetic Data Generation
#
# RAM DESIGN
# ----------
#   • Process one dataset at a time.
#   • Never load all datasets simultaneously.
#   • Never duplicate full training datasets.
#   • Architecture tests use very small tensors.
#   • Persist compact metadata rather than large tensors.
#
# ==================================================================================================

In [26]:
# ==================================================================================================
# 1. HEADER & SCOPE
# ==================================================================================================

from pathlib import Path
import os
import sys
import json
import math
import time
import hashlib
import platform
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

print("=" * 100)
print("NOTEBOOK 08 — SPP-GAN ARCHITECTURE")
print("=" * 100)

print(f"Python       : {sys.version.split()[0]}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
else:
    print("GPU          : CPU")

print("=" * 100)

# -----------------------------------------------------------------------------------------------
# Scope
# -----------------------------------------------------------------------------------------------

NOTEBOOK_ID = "08"
NOTEBOOK_NAME = "SPP-GAN Architecture"
FRAMEWORK_NAME = "SPP-GAN"

RESEARCH_TITLE = (
    "A Unified Privacy-Preserving Framework for High-Fidelity "
    "Synthetic Data Generation Using Statistical and Machine Learning Models"
)

MASTER_SEED = 2025

# Architecture notebook does not train.
TRAINING_ENABLED = False
PRIVACY_ENABLED = False
SYNTHETIC_GENERATION_ENABLED = False

print(f"Framework                : {FRAMEWORK_NAME}")
print(f"Research title           : {RESEARCH_TITLE}")
print(f"Master seed              : {MASTER_SEED}")
print(f"Training enabled         : {TRAINING_ENABLED}")
print(f"Privacy mechanism active : {PRIVACY_ENABLED}")
print(f"Synthetic generation     : {SYNTHETIC_GENERATION_ENABLED}")

NOTEBOOK 08 — SPP-GAN ARCHITECTURE
Python       : 3.13.15
PyTorch      : 2.11.0+cpu
CUDA         : False
GPU          : CPU
Framework                : SPP-GAN
Research title           : A Unified Privacy-Preserving Framework for High-Fidelity Synthetic Data Generation Using Statistical and Machine Learning Models
Master seed              : 2025
Training enabled         : False
Privacy mechanism active : False
Synthetic generation     : False


In [27]:
# ==================================================================================================
# 2. LOAD CONFIGURATION
# ==================================================================================================

import yaml

# -----------------------------------------------------------------------------------------------
# Mount Google Drive FIRST
# -----------------------------------------------------------------------------------------------

try:
    from google.colab import drive

    drive_root = Path("/content/drive")

    if not drive_root.exists() or not (drive_root / "MyDrive").exists():
        drive.mount("/content/drive")
        print("✓ Google Drive mounted.")
    else:
        print("✓ Google Drive already mounted.")

except ImportError:
    print("ℹ Google Colab not detected. Continuing with existing filesystem.")

# -----------------------------------------------------------------------------------------------
# Canonical project root
# -----------------------------------------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/SPP_GAN_Research")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Canonical project root not found after Drive initialization:\n"
        f"{PROJECT_ROOT}"
    )

print(f"✓ Project root : {PROJECT_ROOT}")

# -----------------------------------------------------------------------------------------------
# Notebook 08 directories
# -----------------------------------------------------------------------------------------------

NB08_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_08"

DIRS = {
    "root": NB08_ROOT,
    "architecture": NB08_ROOT / "architecture",
    "config": NB08_ROOT / "config",
    "validation": NB08_ROOT / "validation",
    "metadata": NB08_ROOT / "metadata",
    "models": NB08_ROOT / "models",
    "logs": NB08_ROOT / "logs",
}

for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

print(f"✓ Notebook 08 root : {NB08_ROOT}")

# -----------------------------------------------------------------------------------------------
# Notebook 00 root
# -----------------------------------------------------------------------------------------------

NB00_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_00"

if not NB00_ROOT.exists():
    raise FileNotFoundError(
        f"Notebook 00 output directory not found:\n"
        f"{NB00_ROOT}"
    )

print(f"✓ Notebook 00 : {NB00_ROOT}")

# -----------------------------------------------------------------------------------------------
# Authoritative Notebook 00 experiment configuration
# -----------------------------------------------------------------------------------------------

CONFIG_PATH = NB00_ROOT / "config" / "experiment_config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        "Authoritative Notebook 00 experiment configuration not found:\n"
        f"{CONFIG_PATH}"
    )

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

if not isinstance(CONFIG, dict):
    raise TypeError(
        "Notebook 00 experiment configuration must be a JSON object."
    )

print(f"✓ Configuration loaded : {CONFIG_PATH}")

# -----------------------------------------------------------------------------------------------
# Configuration fingerprint
# -----------------------------------------------------------------------------------------------

CONFIG_FINGERPRINT_PATH = (
    NB00_ROOT
    / "manifest"
    / "configuration_fingerprint.json"
)

if CONFIG_FINGERPRINT_PATH.exists():

    with open(CONFIG_FINGERPRINT_PATH, "r", encoding="utf-8") as f:
        CONFIG_FINGERPRINT = json.load(f)

    print(
        f"✓ Configuration fingerprint loaded : "
        f"{CONFIG_FINGERPRINT_PATH}"
    )

else:

    CONFIG_FINGERPRINT = None

    print(
        "⚠ Configuration fingerprint not found. "
        "Continuing without fingerprint metadata."
    )

# -----------------------------------------------------------------------------------------------
# Helper to retrieve nested values
# -----------------------------------------------------------------------------------------------

def cfg_get(obj, *keys, default=None):
    current = obj

    for key in keys:
        if not isinstance(current, dict) or key not in current:
            return default

        current = current[key]

    return current

# -----------------------------------------------------------------------------------------------
# Canonical Notebook 00 dataset registry
# -----------------------------------------------------------------------------------------------

DATASET_REGISTRY_PATH = (
    NB00_ROOT
    / "config"
    / "dataset_registry.json"
)

if not DATASET_REGISTRY_PATH.exists():
    raise FileNotFoundError(
        "Canonical Notebook 00 dataset registry not found:\n"
        f"{DATASET_REGISTRY_PATH}"
    )

with open(DATASET_REGISTRY_PATH, "r", encoding="utf-8") as f:
    DATASET_REGISTRY = json.load(f)

if not isinstance(DATASET_REGISTRY, dict):
    raise TypeError(
        "Notebook 00 dataset registry must be a dictionary "
        "keyed by dataset identifier."
    )

# -----------------------------------------------------------------------------------------------
# Validate required research datasets
# -----------------------------------------------------------------------------------------------

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

missing_datasets = [
    dataset
    for dataset in EXPECTED_DATASETS
    if dataset not in DATASET_REGISTRY
]

if missing_datasets:
    raise ValueError(
        "Required datasets are missing from the Notebook 00 dataset registry:\n"
        f"{missing_datasets}"
    )

# Validate dataset configuration records
for dataset_name in EXPECTED_DATASETS:

    record = DATASET_REGISTRY[dataset_name]

    if not isinstance(record, dict):
        raise TypeError(
            f"Dataset registry entry for '{dataset_name}' "
            f"must be a dictionary."
        )

    required_fields = [
        "dataset_id",
        "enabled",
        "target_column",
        "task_type",
    ]

    missing_fields = [
        field
        for field in required_fields
        if field not in record
    ]

    if missing_fields:
        raise ValueError(
            f"Dataset '{dataset_name}' is missing required registry fields:\n"
            f"{missing_fields}"
        )

print(f"✓ Dataset registry loaded : {DATASET_REGISTRY_PATH}")
print(f"✓ Registered datasets     : {len(DATASET_REGISTRY)}")
print(f"  {EXPECTED_DATASETS}")

# -----------------------------------------------------------------------------------------------
# Reproducibility
# -----------------------------------------------------------------------------------------------

torch.manual_seed(MASTER_SEED)
np.random.seed(MASTER_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(MASTER_SEED)

print(f"✓ Master seed initialized : {MASTER_SEED}")

✓ Google Drive already mounted.
✓ Project root : /content/drive/MyDrive/SPP_GAN_Research
✓ Notebook 08 root : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08
✓ Notebook 00 : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00
✓ Configuration loaded : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00/config/experiment_config.json
✓ Configuration fingerprint loaded : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00/manifest/configuration_fingerprint.json
✓ Dataset registry loaded : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00/config/dataset_registry.json
✓ Registered datasets     : 3
  ['adult_income', 'bank_marketing', 'diabetes_130us']
✓ Master seed initialized : 2025


In [28]:
# ==================================================================================================
# 3. LOAD FEATURE SCHEMA
# ==================================================================================================

print("=" * 100)
print("3. LOAD FEATURE SCHEMA")
print("=" * 100)

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Canonical Notebook 02 location
# --------------------------------------------------------------------------------------------------

NB02_ROOT = PROJECT_ROOT / "data" / "processed" / "notebook_02"

if not NB02_ROOT.exists():
    raise FileNotFoundError(
        f"Notebook 02 processed directory not found:\n{NB02_ROOT}"
    )

print(f"✓ Notebook 02 root : {NB02_ROOT}")


# --------------------------------------------------------------------------------------------------
# 2. Canonical native manifest
# --------------------------------------------------------------------------------------------------

NATIVE_MANIFEST = (
    NB02_ROOT
    / "native"
    / "native_dataset_manifest.csv"
)

if not NATIVE_MANIFEST.exists():
    raise FileNotFoundError(
        "Canonical Notebook 02 native manifest not found:\n"
        f"{NATIVE_MANIFEST}"
    )

print(f"✓ Native manifest : {NATIVE_MANIFEST}")

native_manifest_df = pd.read_csv(NATIVE_MANIFEST)

print(f"✓ Manifest rows   : {len(native_manifest_df):,}")


# --------------------------------------------------------------------------------------------------
# 3. Validate required manifest columns
# --------------------------------------------------------------------------------------------------

REQUIRED_MANIFEST_COLUMNS = [
    "dataset_id",
    "split",
    "absolute_path",
    "rows",
    "columns",
    "generative_columns",
    "target_column",
    "provenance_column",
    "identifier_columns",
    "provenance_present",
    "target_present",
    "identifiers_excluded",
    "file_exists",
    "reload_validation",
    "sha256",
    "status",
]

missing_manifest_columns = [
    column
    for column in REQUIRED_MANIFEST_COLUMNS
    if column not in native_manifest_df.columns
]

if missing_manifest_columns:
    raise ValueError(
        "Notebook 02 native manifest is missing required columns:\n"
        f"{missing_manifest_columns}"
    )

print("✓ Native manifest schema validated.")


# --------------------------------------------------------------------------------------------------
# 4. Expected datasets
# --------------------------------------------------------------------------------------------------

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

manifest_datasets = set(
    native_manifest_df["dataset_id"]
    .astype(str)
    .str.strip()
)

missing_datasets = [
    dataset
    for dataset in EXPECTED_DATASETS
    if dataset not in manifest_datasets
]

if missing_datasets:
    raise ValueError(
        "Required datasets are missing from Notebook 02 native manifest:\n"
        f"{missing_datasets}"
    )


# --------------------------------------------------------------------------------------------------
# 5. Robust persisted-boolean parser
# --------------------------------------------------------------------------------------------------

def parse_persisted_bool(value, field_name, dataset_name):
    """
    Safely parse boolean values persisted in the Notebook 02 CSV manifest.
    Prevents bool('False') from incorrectly evaluating to True.
    """

    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    if pd.isna(value):
        raise ValueError(
            f"Manifest field '{field_name}' is missing for '{dataset_name}'."
        )

    normalized = str(value).strip().lower()

    if normalized in {"true", "1", "yes", "y", "pass"}:
        return True

    if normalized in {"false", "0", "no", "n", "fail"}:
        return False

    raise ValueError(
        f"Invalid boolean value for '{field_name}' in dataset "
        f"'{dataset_name}': {value!r}"
    )


# --------------------------------------------------------------------------------------------------
# 6. Resolve authoritative training manifest record
# --------------------------------------------------------------------------------------------------

def resolve_training_manifest_record(dataset_name):

    matches = native_manifest_df[
        (
            native_manifest_df["dataset_id"]
            .astype(str)
            .str.strip()
            == dataset_name
        )
        & (
            native_manifest_df["split"]
            .astype(str)
            .str.strip()
            .str.lower()
            == "train"
        )
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one training manifest record for "
            f"'{dataset_name}', found {len(matches)}."
        )

    record = matches.iloc[0].to_dict()

    status = str(record["status"]).strip().upper()

    if status != "PASS":
        raise ValueError(
            f"Notebook 02 training manifest record for "
            f"'{dataset_name}' is not PASS: {status}"
        )

    file_exists = parse_persisted_bool(
        record["file_exists"],
        "file_exists",
        dataset_name,
    )

    reload_validation = parse_persisted_bool(
        record["reload_validation"],
        "reload_validation",
        dataset_name,
    )

    if not file_exists:
        raise FileNotFoundError(
            f"Notebook 02 reports that the training file does not exist "
            f"for '{dataset_name}'."
        )

    if not reload_validation:
        raise ValueError(
            f"Notebook 02 reload validation failed for "
            f"'{dataset_name}'."
        )

    return record


# --------------------------------------------------------------------------------------------------
# 7. SHA-256 verification helper
# --------------------------------------------------------------------------------------------------

def compute_sha256(file_path, chunk_size=1024 * 1024):
    """
    Compute SHA-256 without loading the entire file into RAM.
    """

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()


# --------------------------------------------------------------------------------------------------
# 8. Parse persisted identifier columns
# --------------------------------------------------------------------------------------------------

def parse_identifier_columns(value):

    if value is None:
        return []

    if isinstance(value, float) and np.isnan(value):
        return []

    if isinstance(value, list):
        return [
            str(v).strip()
            for v in value
            if str(v).strip()
        ]

    value = str(value).strip()

    if value in {"", "[]", "None", "nan"}:
        return []

    try:
        parsed = json.loads(value)

        if isinstance(parsed, list):
            return [
                str(v).strip()
                for v in parsed
                if str(v).strip()
            ]

    except (json.JSONDecodeError, TypeError):
        pass

    return [
        item.strip()
        for item in value.split(",")
        if item.strip()
    ]


# --------------------------------------------------------------------------------------------------
# 9. Load authoritative feature schemas
# --------------------------------------------------------------------------------------------------

FEATURE_SCHEMAS = {}

for dataset_name in EXPECTED_DATASETS:

    record = resolve_training_manifest_record(dataset_name)

    # ----------------------------------------------------------------------------------------------
    # 9.1 Authoritative training path
    # ----------------------------------------------------------------------------------------------

    train_path = Path(str(record["absolute_path"]).strip())

    if not train_path.exists():
        raise FileNotFoundError(
            "The authoritative Notebook 02 training file recorded in the "
            "manifest does not exist.\n\n"
            f"Dataset : {dataset_name}\n"
            f"Path    : {train_path}\n\n"
            "No alternate path will be substituted because the Notebook 02 "
            "manifest is the authoritative provenance record."
        )

    # ----------------------------------------------------------------------------------------------
    # 9.2 Persisted SHA-256 validation
    # ----------------------------------------------------------------------------------------------

    persisted_sha256 = str(record["sha256"]).strip().lower()

    if not persisted_sha256:
        raise ValueError(
            f"Notebook 02 manifest contains an empty SHA-256 value "
            f"for '{dataset_name}'."
        )

    if len(persisted_sha256) != 64:
        raise ValueError(
            f"Invalid SHA-256 length for '{dataset_name}': "
            f"{len(persisted_sha256)} characters."
        )

    if any(character not in "0123456789abcdef" for character in persisted_sha256):
        raise ValueError(
            f"Invalid SHA-256 value for '{dataset_name}': "
            f"{persisted_sha256}"
        )

    actual_sha256 = compute_sha256(train_path)

    if actual_sha256 != persisted_sha256:
        raise ValueError(
            "Notebook 02 training-file SHA-256 mismatch.\n\n"
            f"Dataset          : {dataset_name}\n"
            f"Path             : {train_path}\n"
            f"Manifest SHA-256 : {persisted_sha256}\n"
            f"Actual SHA-256   : {actual_sha256}"
        )

    # ----------------------------------------------------------------------------------------------
    # 9.3 RAM-safe header inspection
    # ----------------------------------------------------------------------------------------------

    header_df = pd.read_csv(
        train_path,
        nrows=0,
    )

    native_columns = list(header_df.columns)

    # ----------------------------------------------------------------------------------------------
    # 9.4 Authoritative target
    # ----------------------------------------------------------------------------------------------

    target = str(record["target_column"]).strip()

    if not target:
        raise ValueError(
            f"Notebook 02 target column is empty for '{dataset_name}'."
        )

    if target not in native_columns:
        raise ValueError(
            f"Target '{target}' is missing from the native schema "
            f"for '{dataset_name}'."
        )

    target_present = parse_persisted_bool(
        record["target_present"],
        "target_present",
        dataset_name,
    )

    if not target_present:
        raise ValueError(
            f"Notebook 02 reports target '{target}' as absent "
            f"for '{dataset_name}'."
        )

    # ----------------------------------------------------------------------------------------------
    # 9.5 Authoritative provenance
    # ----------------------------------------------------------------------------------------------

    provenance_column = str(
        record["provenance_column"]
    ).strip()

    if not provenance_column:
        raise ValueError(
            f"Notebook 02 provenance column is empty for '{dataset_name}'."
        )

    if provenance_column not in native_columns:
        raise ValueError(
            f"Provenance column '{provenance_column}' is missing "
            f"from the native schema for '{dataset_name}'."
        )

    provenance_present = parse_persisted_bool(
        record["provenance_present"],
        "provenance_present",
        dataset_name,
    )

    if not provenance_present:
        raise ValueError(
            f"Notebook 02 reports provenance column "
            f"'{provenance_column}' as absent for '{dataset_name}'."
        )

    # ----------------------------------------------------------------------------------------------
    # 9.6 Authoritative identifiers
    # ----------------------------------------------------------------------------------------------

    identifiers = parse_identifier_columns(
        record["identifier_columns"]
    )

    identifiers_excluded = parse_persisted_bool(
        record["identifiers_excluded"],
        "identifiers_excluded",
        dataset_name,
    )

    identifiers_present_in_native = [
        identifier
        for identifier in identifiers
        if identifier in native_columns
    ]

    if identifiers_present_in_native:
        raise ValueError(
            "Identifier columns were expected to be excluded but remain "
            f"in the native schema for '{dataset_name}':\n"
            f"{identifiers_present_in_native}"
        )

    if identifiers and not identifiers_excluded:
        raise ValueError(
            f"Notebook 02 reports identifier columns for "
            f"'{dataset_name}', but identifiers_excluded=False."
        )

    # ----------------------------------------------------------------------------------------------
    # 9.7 Reconstruct generative columns from persisted native schema
    # ----------------------------------------------------------------------------------------------

    excluded_columns = {
        provenance_column,
        *identifiers,
    }

    generative_columns = [
        column
        for column in native_columns
        if column not in excluded_columns
    ]

    # ----------------------------------------------------------------------------------------------
    # 9.8 Cross-check persisted generative dimension
    # ----------------------------------------------------------------------------------------------

    persisted_generative_dimension = int(
        record["generative_columns"]
    )

    if len(generative_columns) != persisted_generative_dimension:
        raise ValueError(
            f"Generative schema mismatch for '{dataset_name}': "
            f"manifest={persisted_generative_dimension}, "
            f"reconstructed={len(generative_columns)}."
        )

    # ----------------------------------------------------------------------------------------------
    # 9.9 Target must remain generative
    # ----------------------------------------------------------------------------------------------

    if target not in generative_columns:
        raise ValueError(
            f"Target '{target}' is not present in generative columns "
            f"for '{dataset_name}'."
        )

    # ----------------------------------------------------------------------------------------------
    # 9.10 Cross-check native dimension
    # ----------------------------------------------------------------------------------------------

    persisted_native_dimension = int(
        record["columns"]
    )

    if len(native_columns) != persisted_native_dimension:
        raise ValueError(
            f"Native schema dimension mismatch for '{dataset_name}': "
            f"manifest={persisted_native_dimension}, "
            f"actual={len(native_columns)}."
        )

    # ----------------------------------------------------------------------------------------------
    # 9.11 Cross-check manifest row count
    # ----------------------------------------------------------------------------------------------

    persisted_rows = int(record["rows"])

    if persisted_rows <= 0:
        raise ValueError(
            f"Notebook 02 persisted an invalid training row count "
            f"for '{dataset_name}': {persisted_rows}"
        )

    # ----------------------------------------------------------------------------------------------
    # 9.12 Store canonical schema
    # ----------------------------------------------------------------------------------------------

    FEATURE_SCHEMAS[dataset_name] = {
        "dataset": dataset_name,
        "train_path": str(train_path),
        "native_columns": native_columns,
        "generative_columns": generative_columns,
        "target": target,
        "identifiers": identifiers,
        "provenance_columns": [provenance_column],
        "native_dimension": len(native_columns),
        "generative_dimension": len(generative_columns),
        "manifest_rows": persisted_rows,
        "manifest_columns": persisted_native_dimension,
        "manifest_generative_columns": persisted_generative_dimension,
        "manifest_sha256": persisted_sha256,
        "verified_sha256": actual_sha256,
    }

    print(
        f"✓ {dataset_name:<20} "
        f"native={len(native_columns):>3} | "
        f"generative={len(generative_columns):>3} | "
        f"target={target:<12} | "
        f"identifiers={len(identifiers)} | "
        f"SHA256=PASS"
    )


# --------------------------------------------------------------------------------------------------
# 10. Final registry validation
# --------------------------------------------------------------------------------------------------

if set(FEATURE_SCHEMAS.keys()) != set(EXPECTED_DATASETS):
    raise RuntimeError(
        "FEATURE_SCHEMAS does not contain exactly the expected datasets."
    )


for dataset_name in EXPECTED_DATASETS:

    schema = FEATURE_SCHEMAS[dataset_name]

    if schema["target"] not in schema["generative_columns"]:
        raise RuntimeError(
            f"Final schema validation failed: target is not generative "
            f"for '{dataset_name}'."
        )

    if any(
        identifier in schema["generative_columns"]
        for identifier in schema["identifiers"]
    ):
        raise RuntimeError(
            f"Final schema validation failed: identifier leakage detected "
            f"for '{dataset_name}'."
        )

    if any(
        provenance in schema["generative_columns"]
        for provenance in schema["provenance_columns"]
    ):
        raise RuntimeError(
            f"Final schema validation failed: provenance leakage detected "
            f"for '{dataset_name}'."
        )

    if schema["manifest_sha256"] != schema["verified_sha256"]:
        raise RuntimeError(
            f"Final SHA-256 validation failed for '{dataset_name}'."
        )


print("\n✓ Feature schema registry created.")
print("✓ Notebook 02 authoritative schema successfully inherited.")
print("✓ Training-file SHA-256 integrity verified for all datasets.")
print("✓ No alternate training-file substitution was permitted.")
print("✓ Target retention, identifier exclusion, and provenance exclusion validated.")
print("✓ SECTION 3 STATUS: PASS")

3. LOAD FEATURE SCHEMA
✓ Notebook 02 root : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
✓ Native manifest : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native/native_dataset_manifest.csv
✓ Manifest rows   : 9
✓ Native manifest schema validated.
✓ adult_income         native= 16 | generative= 15 | target=income       | identifiers=0 | SHA256=PASS
✓ bank_marketing       native= 18 | generative= 17 | target=y            | identifiers=0 | SHA256=PASS
✓ diabetes_130us       native= 49 | generative= 48 | target=readmitted   | identifiers=2 | SHA256=PASS

✓ Feature schema registry created.
✓ Notebook 02 authoritative schema successfully inherited.
✓ Training-file SHA-256 integrity verified for all datasets.
✓ No alternate training-file substitution was permitted.
✓ Target retention, identifier exclusion, and provenance exclusion validated.
✓ SECTION 3 STATUS: PASS


In [29]:
# ==================================================================================================
# 4. LOAD STATISTICAL PROFILE
# ==================================================================================================

print("=" * 100)
print("4. LOAD STATISTICAL PROFILE")
print("=" * 100)

from pathlib import Path
import hashlib
import json
import warnings


# --------------------------------------------------------------------------------------------------
# 1. Canonical Notebook 03 location
# --------------------------------------------------------------------------------------------------

NB03_ROOT = PROJECT_ROOT / "data" / "processed" / "notebook_03"

if not NB03_ROOT.exists():
    raise FileNotFoundError(
        f"Notebook 03 processed directory not found:\n{NB03_ROOT}"
    )

print(f"✓ Notebook 03 root : {NB03_ROOT}")


# --------------------------------------------------------------------------------------------------
# 2. Canonical statistical artifact directories
# --------------------------------------------------------------------------------------------------

GUIDANCE_DIR = NB03_ROOT / "guidance"
REFERENCE_DIR = NB03_ROOT / "reference"

if not GUIDANCE_DIR.exists():
    raise FileNotFoundError(
        f"Notebook 03 guidance directory not found:\n{GUIDANCE_DIR}"
    )

if not REFERENCE_DIR.exists():
    raise FileNotFoundError(
        f"Notebook 03 reference directory not found:\n{REFERENCE_DIR}"
    )


# --------------------------------------------------------------------------------------------------
# 3. Canonical statistical artifact paths
# --------------------------------------------------------------------------------------------------

STATISTICAL_GUIDANCE_PATHS = {
    dataset_name: (
        GUIDANCE_DIR
        / f"{dataset_name}_spp_gan_statistical_guidance.json"
    )
    for dataset_name in EXPECTED_DATASETS
}

STATISTICAL_REFERENCE_PATHS = {
    dataset_name: (
        REFERENCE_DIR
        / f"{dataset_name}_spp_gan_statistical_reference.json"
    )
    for dataset_name in EXPECTED_DATASETS
}


# --------------------------------------------------------------------------------------------------
# 4. Validate required artifact existence
# --------------------------------------------------------------------------------------------------

for dataset_name in EXPECTED_DATASETS:

    guidance_path = STATISTICAL_GUIDANCE_PATHS[dataset_name]
    reference_path = STATISTICAL_REFERENCE_PATHS[dataset_name]

    if not guidance_path.exists():
        raise FileNotFoundError(
            f"SPP-GAN statistical guidance artifact not found for "
            f"'{dataset_name}':\n{guidance_path}"
        )

    if not reference_path.exists():
        raise FileNotFoundError(
            f"SPP-GAN statistical reference artifact not found for "
            f"'{dataset_name}':\n{reference_path}"
        )


# --------------------------------------------------------------------------------------------------
# 5. Required guidance schema
# --------------------------------------------------------------------------------------------------

REQUIRED_GUIDANCE_KEYS = [
    "guidance_version",
    "guidance_type",
    "source_reference_version",
    "source_reference_type",
    "feature_schema",
    "numeric_feature_guidance",
    "categorical_feature_guidance",
    "strongest_numeric_pearson_dependencies",
    "strongest_numeric_spearman_dependencies",
    "strongest_categorical_dependencies",
    "target_policy",
    "identifier_policy",
    "provenance_policy",
    "evidence_policy",
]


# --------------------------------------------------------------------------------------------------
# 6. Required reference schema
# --------------------------------------------------------------------------------------------------

REQUIRED_REFERENCE_KEYS = [
    "reference_version",
    "reference_type",
    "dataset_id",
    "creation_timestamp_utc",
    "random_seed",
    "fit_policy",
    "schema_policy",
    "dataset_profile",
    "feature_schema",
    "feature_profiles",
    "pearson_correlation",
    "spearman_correlation",
    "categorical_dependency",
]


# --------------------------------------------------------------------------------------------------
# 7. Helper: validate JSON object
# --------------------------------------------------------------------------------------------------

def require_dict(value, field_name, dataset_name):

    if not isinstance(value, dict):
        raise TypeError(
            f"{field_name} for '{dataset_name}' must be a dictionary."
        )


# --------------------------------------------------------------------------------------------------
# 8. Helper: validate statistical container
# --------------------------------------------------------------------------------------------------

def validate_statistical_container(
    value,
    field_name,
    dataset_name,
):
    """
    Notebook 03 may persist statistical structures as dictionaries or lists.
    Validate the container type without imposing a schema that Notebook 03
    does not actually persist.
    """

    if not isinstance(value, (dict, list)):
        raise TypeError(
            f"'{field_name}' for '{dataset_name}' must be "
            "a dictionary or list."
        )


# --------------------------------------------------------------------------------------------------
# 9. Load authoritative statistical profiles
# --------------------------------------------------------------------------------------------------

STATISTICAL_PROFILE = {}

for dataset_name in EXPECTED_DATASETS:

    guidance_path = STATISTICAL_GUIDANCE_PATHS[dataset_name]
    reference_path = STATISTICAL_REFERENCE_PATHS[dataset_name]

    # ----------------------------------------------------------------------------------------------
    # 9.1 Load guidance artifact
    # ----------------------------------------------------------------------------------------------

    with open(
        guidance_path,
        "r",
        encoding="utf-8",
    ) as f:
        guidance = json.load(f)

    if not isinstance(guidance, dict):
        raise TypeError(
            f"Statistical guidance for '{dataset_name}' "
            f"must be a JSON object."
        )

    missing_guidance_keys = [
        key
        for key in REQUIRED_GUIDANCE_KEYS
        if key not in guidance
    ]

    if missing_guidance_keys:
        raise ValueError(
            f"Statistical guidance for '{dataset_name}' is missing "
            f"required fields:\n{missing_guidance_keys}"
        )

    # ----------------------------------------------------------------------------------------------
    # 9.2 Load reference artifact
    # ----------------------------------------------------------------------------------------------

    with open(
        reference_path,
        "r",
        encoding="utf-8",
    ) as f:
        reference = json.load(f)

    if not isinstance(reference, dict):
        raise TypeError(
            f"Statistical reference for '{dataset_name}' "
            f"must be a JSON object."
        )

    missing_reference_keys = [
        key
        for key in REQUIRED_REFERENCE_KEYS
        if key not in reference
    ]

    if missing_reference_keys:
        raise ValueError(
            f"Statistical reference for '{dataset_name}' is missing "
            f"required fields:\n{missing_reference_keys}"
        )

    # ----------------------------------------------------------------------------------------------
    # 9.3 Validate reference dataset identity
    # ----------------------------------------------------------------------------------------------

    reference_dataset_id = str(
        reference["dataset_id"]
    ).strip()

    if reference_dataset_id != dataset_name:
        raise ValueError(
            f"Statistical reference dataset mismatch: "
            f"expected='{dataset_name}', "
            f"found='{reference_dataset_id}'."
        )

    # ----------------------------------------------------------------------------------------------
    # 9.4 Validate guidance dataset identity if persisted
    # ----------------------------------------------------------------------------------------------

    if "dataset_id" in guidance:

        guidance_dataset_id = str(
            guidance["dataset_id"]
        ).strip()

        if guidance_dataset_id != dataset_name:
            raise ValueError(
                f"Statistical guidance dataset mismatch: "
                f"expected='{dataset_name}', "
                f"found='{guidance_dataset_id}'."
            )

    # ----------------------------------------------------------------------------------------------
    # 9.5 Validate reference identity fields
    # ----------------------------------------------------------------------------------------------

    reference_version = str(
        reference["reference_version"]
    ).strip()

    reference_type = str(
        reference["reference_type"]
    ).strip()

    if not reference_version:
        raise ValueError(
            f"Reference version is empty for '{dataset_name}'."
        )

    if not reference_type:
        raise ValueError(
            f"Reference type is empty for '{dataset_name}'."
        )

    # ----------------------------------------------------------------------------------------------
    # 9.6 Validate guidance/reference version linkage
    # ----------------------------------------------------------------------------------------------

    source_reference_version = str(
        guidance["source_reference_version"]
    ).strip()

    if not source_reference_version:
        raise ValueError(
            f"Guidance source_reference_version is empty "
            f"for '{dataset_name}'."
        )

    if source_reference_version != reference_version:
        raise ValueError(
            f"Guidance/reference version mismatch for "
            f"'{dataset_name}': "
            f"guidance='{source_reference_version}', "
            f"reference='{reference_version}'."
        )

    # ----------------------------------------------------------------------------------------------
    # 9.7 Validate guidance/reference type linkage
    # ----------------------------------------------------------------------------------------------

    source_reference_type = str(
        guidance["source_reference_type"]
    ).strip()

    if not source_reference_type:
        raise ValueError(
            f"Guidance source_reference_type is empty "
            f"for '{dataset_name}'."
        )

    if source_reference_type != reference_type:
        raise ValueError(
            f"Guidance/reference type mismatch for "
            f"'{dataset_name}': "
            f"guidance='{source_reference_type}', "
            f"reference='{reference_type}'."
        )

    # ----------------------------------------------------------------------------------------------
    # 9.8 Validate guidance feature schema object
    # ----------------------------------------------------------------------------------------------

    guidance_schema = guidance["feature_schema"]

    require_dict(
        guidance_schema,
        "Guidance feature_schema",
        dataset_name,
    )

    # ----------------------------------------------------------------------------------------------
    # 9.9 Validate reference feature schema object
    # ----------------------------------------------------------------------------------------------

    reference_schema = reference["feature_schema"]

    require_dict(
        reference_schema,
        "Reference feature_schema",
        dataset_name,
    )

    # ----------------------------------------------------------------------------------------------
    # 9.10 Validate authoritative Notebook 02 schema
    # ----------------------------------------------------------------------------------------------

    if dataset_name not in FEATURE_SCHEMAS:
        raise RuntimeError(
            f"Notebook 02 feature schema is unavailable for "
            f"'{dataset_name}'."
        )

    authoritative_schema = FEATURE_SCHEMAS[
        dataset_name
    ]

    expected_generative_dimension = int(
        authoritative_schema["generative_dimension"]
    )

    expected_native_dimension = int(
        authoritative_schema["native_dimension"]
    )

    expected_target = str(
        authoritative_schema["target"]
    ).strip()

    expected_identifiers = list(
        authoritative_schema["identifiers"]
    )

    expected_provenance = list(
        authoritative_schema["provenance_columns"]
    )

    # ----------------------------------------------------------------------------------------------
    # 9.11 Validate Notebook 03 schema dimensions where persisted
    # ----------------------------------------------------------------------------------------------

    def validate_optional_dimension(
        schema,
        source_name,
        dataset_name,
        expected_dimension,
    ):

        dimension_fields = [
            "generative_dimension",
            "feature_count",
            "n_features",
        ]

        for field_name in dimension_fields:

            if field_name not in schema:
                continue

            try:
                persisted_dimension = int(
                    schema[field_name]
                )
            except (TypeError, ValueError):
                raise ValueError(
                    f"{source_name}.feature_schema."
                    f"{field_name} is not a valid integer for "
                    f"'{dataset_name}'."
                )

            if persisted_dimension != expected_dimension:
                raise ValueError(
                    f"{source_name} generative dimension mismatch "
                    f"for '{dataset_name}': "
                    f"expected={expected_dimension}, "
                    f"found={persisted_dimension}."
                )

            break

    validate_optional_dimension(
        guidance_schema,
        "Guidance",
        dataset_name,
        expected_generative_dimension,
    )

    validate_optional_dimension(
        reference_schema,
        "Reference",
        dataset_name,
        expected_generative_dimension,
    )

    # ----------------------------------------------------------------------------------------------
    # 9.12 Validate feature-schema dataset dimensions where persisted
    # ----------------------------------------------------------------------------------------------

    for source_name, schema in [
        ("Guidance", guidance_schema),
        ("Reference", reference_schema),
    ]:

        if "native_dimension" in schema:

            persisted_native_dimension = int(
                schema["native_dimension"]
            )

            if persisted_native_dimension != expected_native_dimension:
                raise ValueError(
                    f"{source_name} native dimension mismatch for "
                    f"'{dataset_name}': "
                    f"expected={expected_native_dimension}, "
                    f"found={persisted_native_dimension}."
                )

        if "generative_dimension" in schema:

            persisted_generative_dimension = int(
                schema["generative_dimension"]
            )

            if persisted_generative_dimension != expected_generative_dimension:
                raise ValueError(
                    f"{source_name} generative dimension mismatch for "
                    f"'{dataset_name}': "
                    f"expected={expected_generative_dimension}, "
                    f"found={persisted_generative_dimension}."
                )

    # ----------------------------------------------------------------------------------------------
    # 9.13 Validate target against authoritative Notebook 02 schema
    # ----------------------------------------------------------------------------------------------

    if expected_target not in authoritative_schema[
        "generative_columns"
    ]:
        raise ValueError(
            f"Target '{expected_target}' is not present in the "
            f"Notebook 02 generative schema for '{dataset_name}'."
        )

    # ----------------------------------------------------------------------------------------------
    # 9.14 Validate target policy where explicitly persisted
    # ----------------------------------------------------------------------------------------------

    target_policy = guidance["target_policy"]

    require_dict(
        target_policy,
        "Target policy",
        dataset_name,
    )

    target_policy_text = json.dumps(
        target_policy,
        sort_keys=True,
    ).lower()

    if expected_target.lower() not in target_policy_text:
        warnings.warn(
            f"Target '{expected_target}' is not explicitly named "
            f"inside the persisted target policy for '{dataset_name}'."
        )

    # ----------------------------------------------------------------------------------------------
    # 9.15 Validate identifier policy
    # ----------------------------------------------------------------------------------------------

    identifier_policy = guidance["identifier_policy"]

    require_dict(
        identifier_policy,
        "Identifier policy",
        dataset_name,
    )

    if "identifier_columns" in identifier_policy:

        policy_identifiers = identifier_policy[
            "identifier_columns"
        ]

        if isinstance(policy_identifiers, str):
            policy_identifiers = [
                item.strip()
                for item in policy_identifiers.split(",")
                if item.strip()
            ]

        if not isinstance(policy_identifiers, list):
            raise TypeError(
                f"identifier_policy.identifier_columns for "
                f"'{dataset_name}' must be a list or string."
            )

        policy_identifiers = [
            str(item).strip()
            for item in policy_identifiers
            if str(item).strip()
        ]

        if policy_identifiers != expected_identifiers:
            raise ValueError(
                f"Identifier-policy mismatch for '{dataset_name}': "
                f"Notebook 02={expected_identifiers}, "
                f"Notebook 03={policy_identifiers}."
            )

    # ----------------------------------------------------------------------------------------------
    # 9.16 Validate provenance policy
    # ----------------------------------------------------------------------------------------------

    provenance_policy = guidance["provenance_policy"]

    require_dict(
        provenance_policy,
        "Provenance policy",
        dataset_name,
    )

    if "provenance_column" in provenance_policy:

        policy_provenance = str(
            provenance_policy["provenance_column"]
        ).strip()

        if policy_provenance != expected_provenance[0]:
            raise ValueError(
                f"Provenance-policy mismatch for "
                f"'{dataset_name}': "
                f"Notebook 02='{expected_provenance[0]}', "
                f"Notebook 03='{policy_provenance}'."
            )

    # ----------------------------------------------------------------------------------------------
    # 9.17 Validate statistical-guidance containers
    # ----------------------------------------------------------------------------------------------

    numeric_feature_guidance = guidance[
        "numeric_feature_guidance"
    ]

    categorical_feature_guidance = guidance[
        "categorical_feature_guidance"
    ]

    pearson_dependencies = guidance[
        "strongest_numeric_pearson_dependencies"
    ]

    spearman_dependencies = guidance[
        "strongest_numeric_spearman_dependencies"
    ]

    categorical_dependencies = guidance[
        "strongest_categorical_dependencies"
    ]

    validate_statistical_container(
        numeric_feature_guidance,
        "numeric_feature_guidance",
        dataset_name,
    )

    validate_statistical_container(
        categorical_feature_guidance,
        "categorical_feature_guidance",
        dataset_name,
    )

    validate_statistical_container(
        pearson_dependencies,
        "strongest_numeric_pearson_dependencies",
        dataset_name,
    )

    validate_statistical_container(
        spearman_dependencies,
        "strongest_numeric_spearman_dependencies",
        dataset_name,
    )

    validate_statistical_container(
        categorical_dependencies,
        "strongest_categorical_dependencies",
        dataset_name,
    )

    # ----------------------------------------------------------------------------------------------
    # 9.18 Validate reference statistical evidence
    # ----------------------------------------------------------------------------------------------

    dataset_profile = reference["dataset_profile"]
    feature_profiles = reference["feature_profiles"]
    pearson_correlation = reference["pearson_correlation"]
    spearman_correlation = reference["spearman_correlation"]
    categorical_dependency = reference["categorical_dependency"]

    validate_statistical_container(
        dataset_profile,
        "dataset_profile",
        dataset_name,
    )

    validate_statistical_container(
        feature_profiles,
        "feature_profiles",
        dataset_name,
    )

    validate_statistical_container(
        pearson_correlation,
        "pearson_correlation",
        dataset_name,
    )

    validate_statistical_container(
        spearman_correlation,
        "spearman_correlation",
        dataset_name,
    )

    validate_statistical_container(
        categorical_dependency,
        "categorical_dependency",
        dataset_name,
    )

    # ----------------------------------------------------------------------------------------------
    # 9.19 Validate evidence policy
    # ----------------------------------------------------------------------------------------------

    evidence_policy = guidance["evidence_policy"]

    require_dict(
        evidence_policy,
        "Evidence policy",
        dataset_name,
    )

    # ----------------------------------------------------------------------------------------------
    # 9.20 Store authoritative statistical profile
    # ----------------------------------------------------------------------------------------------

    STATISTICAL_PROFILE[dataset_name] = {
        "dataset": dataset_name,

        "guidance": guidance,
        "reference": reference,

        "guidance_path": str(guidance_path),
        "reference_path": str(reference_path),

        "guidance_version": str(
            guidance["guidance_version"]
        ).strip(),

        "reference_version": reference_version,

        "guidance_type": str(
            guidance["guidance_type"]
        ).strip(),

        "reference_type": reference_type,

        # Notebook 02 remains authoritative for exact schema identity.
        "feature_schema": list(
            authoritative_schema["generative_columns"]
        ),

        "generative_dimension": expected_generative_dimension,

        "target": expected_target,

        "identifiers": expected_identifiers,

        "provenance_columns": expected_provenance,

        # Statistical guidance.
        "numeric_feature_guidance": numeric_feature_guidance,
        "categorical_feature_guidance": categorical_feature_guidance,
        "pearson_dependencies": pearson_dependencies,
        "spearman_dependencies": spearman_dependencies,
        "categorical_dependencies": categorical_dependencies,

        # Notebook 03 statistical evidence.
        "dataset_profile": dataset_profile,
        "feature_profiles": feature_profiles,
        "pearson_correlation": pearson_correlation,
        "spearman_correlation": spearman_correlation,
        "categorical_dependency": categorical_dependency,

        # Governance policies.
        "target_policy": target_policy,
        "identifier_policy": identifier_policy,
        "provenance_policy": provenance_policy,
        "evidence_policy": evidence_policy,
    }

    print(
        f"✓ {dataset_name:<20} "
        f"guidance=loaded | "
        f"reference=loaded | "
        f"features={expected_generative_dimension} | "
        f"schema=PASS | "
        f"linkage=PASS"
    )


# --------------------------------------------------------------------------------------------------
# 10. Final registry validation
# --------------------------------------------------------------------------------------------------

if set(STATISTICAL_PROFILE.keys()) != set(EXPECTED_DATASETS):
    raise RuntimeError(
        "STATISTICAL_PROFILE does not contain exactly "
        "the expected datasets."
    )


for dataset_name in EXPECTED_DATASETS:

    profile = STATISTICAL_PROFILE[dataset_name]

    authoritative_schema = FEATURE_SCHEMAS[
        dataset_name
    ]

    # ----------------------------------------------------------------------------------------------
    # 10.1 Generative dimension
    # ----------------------------------------------------------------------------------------------

    if profile["generative_dimension"] != authoritative_schema[
        "generative_dimension"
    ]:
        raise RuntimeError(
            f"Final generative-dimension validation failed "
            f"for '{dataset_name}'."
        )

    # ----------------------------------------------------------------------------------------------
    # 10.2 Target retention
    # ----------------------------------------------------------------------------------------------

    if profile["target"] != authoritative_schema["target"]:
        raise RuntimeError(
            f"Final target identity validation failed "
            f"for '{dataset_name}'."
        )

    if profile["target"] not in profile["feature_schema"]:
        raise RuntimeError(
            f"Final target-retention validation failed "
            f"for '{dataset_name}'."
        )

    # ----------------------------------------------------------------------------------------------
    # 10.3 Identifier exclusion
    # ----------------------------------------------------------------------------------------------

    for identifier in authoritative_schema["identifiers"]:

        if identifier in profile["feature_schema"]:
            raise RuntimeError(
                f"Identifier leakage detected in statistical profile "
                f"for '{dataset_name}': {identifier}"
            )

    # ----------------------------------------------------------------------------------------------
    # 10.4 Provenance exclusion
    # ----------------------------------------------------------------------------------------------

    for provenance in authoritative_schema[
        "provenance_columns"
    ]:

        if provenance in profile["feature_schema"]:
            raise RuntimeError(
                f"Provenance leakage detected in statistical profile "
                f"for '{dataset_name}': {provenance}"
            )

    # ----------------------------------------------------------------------------------------------
    # 10.5 Guidance/reference linkage
    # ----------------------------------------------------------------------------------------------

    if profile["guidance"].get(
        "source_reference_version"
    ) != profile["reference_version"]:

        raise RuntimeError(
            f"Final guidance/reference version linkage failed "
            f"for '{dataset_name}'."
        )

    if profile["guidance"].get(
        "source_reference_type"
    ) != profile["reference_type"]:

        raise RuntimeError(
            f"Final guidance/reference type linkage failed "
            f"for '{dataset_name}'."
        )

    # ----------------------------------------------------------------------------------------------
    # 10.6 Statistical guidance availability
    # ----------------------------------------------------------------------------------------------

    if not profile["numeric_feature_guidance"]:
        warnings.warn(
            f"No numeric feature guidance entries found for "
            f"'{dataset_name}'."
        )

    if not profile["categorical_feature_guidance"]:
        warnings.warn(
            f"No categorical feature guidance entries found for "
            f"'{dataset_name}'."
        )


# --------------------------------------------------------------------------------------------------
# 11. Completion
# --------------------------------------------------------------------------------------------------

print()
print("✓ Authoritative Notebook 03 statistical profiles loaded.")
print("✓ Guidance/reference version linkage validated.")
print("✓ Guidance/reference type linkage validated.")
print("✓ Notebook 02 authoritative generative dimensions validated.")
print("✓ Target retention validated.")
print("✓ Identifier and provenance exclusion validated.")
print("✓ Statistical guidance components validated.")
print("✓ Statistical evidence containers validated.")
print("✓ SPP-GAN statistical guidance registry initialized.")
print("✓ SECTION 4 STATUS: PASS")

4. LOAD STATISTICAL PROFILE
✓ Notebook 03 root : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_03
✓ adult_income         guidance=loaded | reference=loaded | features=15 | schema=PASS | linkage=PASS
✓ bank_marketing       guidance=loaded | reference=loaded | features=17 | schema=PASS | linkage=PASS
✓ diabetes_130us       guidance=loaded | reference=loaded | features=48 | schema=PASS | linkage=PASS

✓ Authoritative Notebook 03 statistical profiles loaded.
✓ Guidance/reference version linkage validated.
✓ Guidance/reference type linkage validated.
✓ Notebook 02 authoritative generative dimensions validated.
✓ Target retention validated.
✓ Identifier and provenance exclusion validated.
✓ Statistical guidance components validated.
✓ Statistical evidence containers validated.
✓ SPP-GAN statistical guidance registry initialized.
✓ SECTION 4 STATUS: PASS


In [30]:
# ==================================================================================================
# 5. DEFINE INPUT REPRESENTATION
# ==================================================================================================

print("=" * 100)
print("5. DEFINE INPUT REPRESENTATION")
print("=" * 100)


class SPPGANInputRepresentation:
    """
    Defines the logical input representation used by SPP-GAN.

    The representation separates:

    1. Generative data features
       - Numerical features
       - Categorical features

    2. Conditional information
       - Optional conditional vector used by the generator/discriminator

    3. Statistical guidance
       - External statistical reference information
       - Not treated as ordinary record-level features

    Raw identifiers and provenance columns are never part of
    the generative representation.
    """

    def __init__(
        self,
        num_numerical,
        num_categorical,
        categorical_cardinalities=None,
        conditional_dim=0,
        transformed_dim=None,
    ):
        self.num_numerical = int(num_numerical)
        self.num_categorical = int(num_categorical)

        self.categorical_cardinalities = (
            [int(v) for v in categorical_cardinalities]
            if categorical_cardinalities is not None
            else []
        )

        self.conditional_dim = int(conditional_dim)

        # The actual transformed dimension is determined by
        # the preprocessing/transformation implementation.
        self._transformed_dim = (
            int(transformed_dim)
            if transformed_dim is not None
            else None
        )

        if self.num_numerical < 0:
            raise ValueError(
                "num_numerical must be non-negative."
            )

        if self.num_categorical < 0:
            raise ValueError(
                "num_categorical must be non-negative."
            )

        if self.conditional_dim < 0:
            raise ValueError(
                "conditional_dim must be non-negative."
            )

        if len(self.categorical_cardinalities) != self.num_categorical:
            raise ValueError(
                "Number of categorical cardinalities must equal "
                "num_categorical."
            )

        if any(v <= 0 for v in self.categorical_cardinalities):
            raise ValueError(
                "Categorical cardinalities must be positive."
            )

        if self._transformed_dim is not None and self._transformed_dim <= 0:
            raise ValueError(
                "transformed_dim must be positive when provided."
            )

    @property
    def feature_dim(self):
        """
        Logical feature dimension before dataset-specific
        transformation expansion.
        """

        return (
            self.num_numerical
            + sum(self.categorical_cardinalities)
        )

    @property
    def transformed_dim(self):
        """
        Actual transformed tensor dimension.

        This value must be supplied by the authoritative
        preprocessing/transformation layer once the transformation
        has been fitted.
        """

        return self._transformed_dim

    @property
    def conditional_input_dim(self):
        """
        Dimension of the optional conditional vector.
        """

        return self.conditional_dim

    def set_transformed_dim(self, transformed_dim):
        """
        Register the actual transformed dimension produced by
        the fitted transformation layer.
        """

        transformed_dim = int(transformed_dim)

        if transformed_dim <= 0:
            raise ValueError(
                "transformed_dim must be positive."
            )

        self._transformed_dim = transformed_dim

    def summary(self):
        return {
            "num_numerical": self.num_numerical,
            "num_categorical": self.num_categorical,
            "categorical_cardinalities": (
                list(self.categorical_cardinalities)
            ),
            "conditional_dim": self.conditional_dim,
            "logical_feature_dim": self.feature_dim,
            "transformed_dim": self.transformed_dim,
        }


print("✓ SPP-GAN input representation interface defined.")
print("✓ Actual transformed dimension is delegated to the fitted transformation layer.")
print("✓ Conditional information is represented separately.")
print("✓ Statistical guidance remains external to the record-level feature tensor.")
print("✓ Raw identifiers/provenance are excluded.")

5. DEFINE INPUT REPRESENTATION
✓ SPP-GAN input representation interface defined.
✓ Actual transformed dimension is delegated to the fitted transformation layer.
✓ Conditional information is represented separately.
✓ Statistical guidance remains external to the record-level feature tensor.
✓ Raw identifiers/provenance are excluded.


In [31]:
# ==================================================================================================
# 6. DEFINE LATENT SPACE
# ==================================================================================================

print("=" * 100)
print("6. DEFINE LATENT SPACE")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Authoritative architecture configuration
# --------------------------------------------------------------------------------------------------

SPPGAN_LATENT_DIM = 128
SPPGAN_CONDITIONAL_DIM = 0


# --------------------------------------------------------------------------------------------------
# 2. Latent-space interface
# --------------------------------------------------------------------------------------------------

class SPPGANLatentSpace:
    """
    Latent-space definition for SPP-GAN.

    Latent variable:

        z ~ N(0, I)

    where z has dimension SPPGAN_LATENT_DIM.

    Conditional information is represented separately from the
    latent variable and is controlled by SPPGAN_CONDITIONAL_DIM.
    """

    def __init__(
        self,
        latent_dim=128,
        conditional_dim=0,
    ):
        self.latent_dim = int(latent_dim)
        self.conditional_dim = int(conditional_dim)

        if self.latent_dim <= 0:
            raise ValueError(
                "latent_dim must be a positive integer."
            )

        if self.conditional_dim < 0:
            raise ValueError(
                "conditional_dim must be non-negative."
            )

    @property
    def input_dim(self):
        """
        Total generator input dimension contributed by
        latent and conditional variables.
        """

        return (
            self.latent_dim
            + self.conditional_dim
        )

    def sample(
        self,
        batch_size,
        device=None,
        dtype=torch.float32,
    ):
        """
        Sample latent variables from the standard multivariate
        normal distribution.

        Returns
        -------
        torch.Tensor
            Tensor with shape:

                (batch_size, latent_dim)
        """

        batch_size = int(batch_size)

        if batch_size <= 0:
            raise ValueError(
                "batch_size must be a positive integer."
            )

        z = torch.randn(
            batch_size,
            self.latent_dim,
            device=device,
            dtype=dtype,
        )

        if z.ndim != 2:
            raise RuntimeError(
                "Latent sample must be a two-dimensional tensor."
            )

        if z.shape != (
            batch_size,
            self.latent_dim,
        ):
            raise RuntimeError(
                "Latent sample has an unexpected shape: "
                f"expected={(batch_size, self.latent_dim)}, "
                f"found={tuple(z.shape)}."
            )

        return z

    def summary(self):
        return {
            "latent_dim": self.latent_dim,
            "conditional_dim": self.conditional_dim,
            "generator_input_dim": self.input_dim,
            "distribution": "standard_normal",
            "notation": "z ~ N(0, I)",
        }


# --------------------------------------------------------------------------------------------------
# 3. Instantiate authoritative latent-space definition
# --------------------------------------------------------------------------------------------------

LATENT_SPACE = SPPGANLatentSpace(
    latent_dim=SPPGAN_LATENT_DIM,
    conditional_dim=SPPGAN_CONDITIONAL_DIM,
)


# --------------------------------------------------------------------------------------------------
# 4. Architecture-level validation
# --------------------------------------------------------------------------------------------------

if LATENT_SPACE.latent_dim != 128:
    raise RuntimeError(
        "SPP-GAN latent dimension must be 128."
    )

if LATENT_SPACE.conditional_dim != SPPGAN_CONDITIONAL_DIM:
    raise RuntimeError(
        "Latent-space conditional dimension is inconsistent "
        "with SPPGAN_CONDITIONAL_DIM."
    )


# --------------------------------------------------------------------------------------------------
# 5. Deterministic shape test
# --------------------------------------------------------------------------------------------------

_TEST_BATCH_SIZE = 4

_test_z = LATENT_SPACE.sample(
    batch_size=_TEST_BATCH_SIZE,
    device="cpu",
    dtype=torch.float32,
)

expected_shape = (
    _TEST_BATCH_SIZE,
    SPPGAN_LATENT_DIM,
)

if tuple(_test_z.shape) != expected_shape:
    raise RuntimeError(
        f"Latent-space shape validation failed: "
        f"expected={expected_shape}, "
        f"found={tuple(_test_z.shape)}."
    )

del _test_z


# --------------------------------------------------------------------------------------------------
# 6. Final reporting
# --------------------------------------------------------------------------------------------------

print(
    f"✓ Latent dimension : "
    f"{LATENT_SPACE.latent_dim}"
)

print(
    "✓ Distribution     : "
    "Standard Normal"
)

print(
    f"✓ Conditional dim  : "
    f"{LATENT_SPACE.conditional_dim}"
)

print(
    f"✓ Generator input  : "
    f"{LATENT_SPACE.input_dim}"
)

print(
    "✓ Latent sampling shape validation : PASS"
)

print(
    "✓ Master-seed-controlled PyTorch RNG retained."
)

print(
    "✓ SECTION 6 STATUS: PASS"
)

6. DEFINE LATENT SPACE
✓ Latent dimension : 128
✓ Distribution     : Standard Normal
✓ Conditional dim  : 0
✓ Generator input  : 128
✓ Latent sampling shape validation : PASS
✓ Master-seed-controlled PyTorch RNG retained.
✓ SECTION 6 STATUS: PASS


In [32]:
# ==================================================================================================
# 7. DEFINE GENERATOR
# ==================================================================================================

print("=" * 100)
print("7. DEFINE GENERATOR")
print("=" * 100)


class SPPGANGenerator(nn.Module):
    """
    SPP-GAN Generator.

    The generator maps a latent vector z and an optional conditional
    vector c into a differentiable tabular representation.

    Input:
        z ~ N(0, I)

        Optional condition:
        c

    Architecture:
        [z, c]
            ↓
        Linear
            ↓
        ReLU
            ↓
        Linear
            ↓
        ReLU
            ↓
        Shared representation
            ├── Numerical output head
            └── Categorical output heads

    Numerical outputs are controlled by the configured numerical
    output activation.

    Categorical heads produce logits. Differentiable categorical
    probabilities are obtained using Gumbel-Softmax.

    Raw identifiers and provenance variables are never generated.
    """

    def __init__(
        self,
        latent_dim,
        hidden_dim_1,
        hidden_dim_2,
        num_numerical,
        categorical_cardinalities,
        conditional_dim=0,
        numerical_activation="identity",
    ):
        super().__init__()

        self.latent_dim = int(latent_dim)
        self.hidden_dim_1 = int(hidden_dim_1)
        self.hidden_dim_2 = int(hidden_dim_2)
        self.num_numerical = int(num_numerical)

        self.categorical_cardinalities = [
            int(v)
            for v in categorical_cardinalities
        ]

        self.conditional_dim = int(conditional_dim)

        self.numerical_activation = str(
            numerical_activation
        ).strip().lower()

        # ------------------------------------------------------------------------------------------
        # Validation
        # ------------------------------------------------------------------------------------------

        if self.latent_dim <= 0:
            raise ValueError(
                "latent_dim must be positive."
            )

        if self.hidden_dim_1 <= 0:
            raise ValueError(
                "hidden_dim_1 must be positive."
            )

        if self.hidden_dim_2 <= 0:
            raise ValueError(
                "hidden_dim_2 must be positive."
            )

        if self.num_numerical < 0:
            raise ValueError(
                "num_numerical must be non-negative."
            )

        if self.conditional_dim < 0:
            raise ValueError(
                "conditional_dim must be non-negative."
            )

        if any(
            cardinality <= 0
            for cardinality in self.categorical_cardinalities
        ):
            raise ValueError(
                "All categorical cardinalities must be positive."
            )

        if self.numerical_activation not in {
            "identity",
            "tanh",
        }:
            raise ValueError(
                "numerical_activation must be either "
                "'identity' or 'tanh'."
            )

        # ------------------------------------------------------------------------------------------
        # Generator input dimension
        # ------------------------------------------------------------------------------------------

        input_dim = (
            self.latent_dim
            + self.conditional_dim
        )

        if input_dim <= 0:
            raise ValueError(
                "Generator input dimension must be positive."
            )

        self.input_dim = input_dim

        # ------------------------------------------------------------------------------------------
        # Shared generator backbone
        # ------------------------------------------------------------------------------------------

        self.backbone = nn.Sequential(
            nn.Linear(
                self.input_dim,
                self.hidden_dim_1,
            ),
            nn.ReLU(),
            nn.Linear(
                self.hidden_dim_1,
                self.hidden_dim_2,
            ),
            nn.ReLU(),
        )

        # ------------------------------------------------------------------------------------------
        # Numerical output head
        # ------------------------------------------------------------------------------------------

        if self.num_numerical > 0:

            self.numerical_head = nn.Linear(
                self.hidden_dim_2,
                self.num_numerical,
            )

        else:

            self.numerical_head = None

        # ------------------------------------------------------------------------------------------
        # Categorical output heads
        # ------------------------------------------------------------------------------------------

        self.categorical_heads = nn.ModuleList(
            [
                nn.Linear(
                    self.hidden_dim_2,
                    cardinality,
                )
                for cardinality
                in self.categorical_cardinalities
            ]
        )

        if len(self.categorical_heads) != len(
            self.categorical_cardinalities
        ):
            raise RuntimeError(
                "Categorical-head/cardinality mismatch."
            )

    # ----------------------------------------------------------------------------------------------
    # Numerical activation
    # ----------------------------------------------------------------------------------------------

    def _apply_numerical_activation(
        self,
        numerical,
    ):
        if numerical is None:
            return None

        if self.numerical_activation == "identity":
            return numerical

        if self.numerical_activation == "tanh":
            return torch.tanh(numerical)

        raise RuntimeError(
            f"Unsupported numerical activation: "
            f"{self.numerical_activation}"
        )

    # ----------------------------------------------------------------------------------------------
    # Forward pass
    # ----------------------------------------------------------------------------------------------

    def forward(
        self,
        z,
        condition=None,
        categorical_temperature=1.0,
        hard=False,
    ):
        """
        Forward pass.

        Parameters
        ----------
        z : torch.Tensor
            Latent vectors with shape
            [batch_size, latent_dim].

        condition : torch.Tensor or None
            Optional conditional vector with shape
            [batch_size, conditional_dim].

        categorical_temperature : float
            Positive Gumbel-Softmax temperature.

        hard : bool
            If True, produces hard one-hot categorical outputs using
            the straight-through estimator.
        """

        # ------------------------------------------------------------------------------------------
        # Validate latent tensor
        # ------------------------------------------------------------------------------------------

        if not isinstance(z, torch.Tensor):
            raise TypeError(
                "z must be a torch.Tensor."
            )

        if z.ndim != 2:
            raise ValueError(
                "z must be a 2D tensor "
                "[batch_size, latent_dim]."
            )

        if z.shape[1] != self.latent_dim:
            raise ValueError(
                f"Expected latent dimension "
                f"{self.latent_dim}, "
                f"received {z.shape[1]}."
            )

        if z.shape[0] <= 0:
            raise ValueError(
                "Batch size must be positive."
            )

        # ------------------------------------------------------------------------------------------
        # Validate categorical temperature
        # ------------------------------------------------------------------------------------------

        categorical_temperature = float(
            categorical_temperature
        )

        if categorical_temperature <= 0:
            raise ValueError(
                "categorical_temperature must be positive."
            )

        # ------------------------------------------------------------------------------------------
        # Conditional input
        # ------------------------------------------------------------------------------------------

        if self.conditional_dim > 0:

            if condition is None:
                raise ValueError(
                    "condition must be provided when "
                    "conditional_dim > 0."
                )

            if not isinstance(condition, torch.Tensor):
                raise TypeError(
                    "condition must be a torch.Tensor."
                )

            if condition.ndim != 2:
                raise ValueError(
                    "condition must be a 2D tensor "
                    "[batch_size, conditional_dim]."
                )

            if condition.shape[0] != z.shape[0]:
                raise ValueError(
                    "z and condition must have "
                    "the same batch size."
                )

            if condition.shape[1] != self.conditional_dim:
                raise ValueError(
                    f"Expected conditional dimension "
                    f"{self.conditional_dim}, "
                    f"received {condition.shape[1]}."
                )

            generator_input = torch.cat(
                [z, condition],
                dim=1,
            )

        else:

            if condition is not None:
                raise ValueError(
                    "condition was supplied, but "
                    "conditional_dim is 0."
                )

            generator_input = z

        # ------------------------------------------------------------------------------------------
        # Shared representation
        # ------------------------------------------------------------------------------------------

        h = self.backbone(
            generator_input
        )

        # ------------------------------------------------------------------------------------------
        # Numerical output
        # ------------------------------------------------------------------------------------------

        numerical = None

        if self.numerical_head is not None:

            numerical_logits = self.numerical_head(
                h
            )

            numerical = self._apply_numerical_activation(
                numerical_logits
            )

        # ------------------------------------------------------------------------------------------
        # Categorical outputs
        # ------------------------------------------------------------------------------------------

        categorical_logits = [
            head(h)
            for head in self.categorical_heads
        ]

        categorical_probabilities = []

        for logits in categorical_logits:

            probabilities = F.gumbel_softmax(
                logits,
                tau=categorical_temperature,
                hard=bool(hard),
                dim=1,
            )

            categorical_probabilities.append(
                probabilities
            )

        # ------------------------------------------------------------------------------------------
        # Return structured generator output
        # ------------------------------------------------------------------------------------------

        return {
            "hidden": h,
            "numerical": numerical,
            "categorical_logits": categorical_logits,
            "categorical_probabilities": (
                categorical_probabilities
            ),
        }

    # ----------------------------------------------------------------------------------------------
    # Architecture summary
    # ----------------------------------------------------------------------------------------------

    def summary(self):

        return {
            "architecture": "MLP",
            "input_dim": self.input_dim,
            "latent_dim": self.latent_dim,
            "conditional_dim": self.conditional_dim,
            "hidden_dim_1": self.hidden_dim_1,
            "hidden_dim_2": self.hidden_dim_2,
            "num_numerical": self.num_numerical,
            "num_categorical": len(
                self.categorical_cardinalities
            ),
            "categorical_cardinalities": list(
                self.categorical_cardinalities
            ),
            "numerical_activation": (
                self.numerical_activation
            ),
            "categorical_output": (
                "gumbel_softmax"
            ),
        }


print("✓ SPP-GAN generator architecture defined.")
print("✓ Optional conditional input is supported explicitly.")
print("✓ Numerical and categorical output heads are separated.")
print("✓ Numerical output activation is explicitly configurable.")
print("✓ Categorical outputs support differentiable Gumbel-Softmax.")
print("✓ Categorical temperature validation is enforced.")
print("✓ Raw identifiers/provenance are excluded.")
print("✓ No training or synthetic-data generation performed.")

7. DEFINE GENERATOR
✓ SPP-GAN generator architecture defined.
✓ Optional conditional input is supported explicitly.
✓ Numerical and categorical output heads are separated.
✓ Numerical output activation is explicitly configurable.
✓ Categorical outputs support differentiable Gumbel-Softmax.
✓ Categorical temperature validation is enforced.
✓ Raw identifiers/provenance are excluded.
✓ No training or synthetic-data generation performed.


In [33]:
# ==================================================================================================
# 8. DEFINE DISCRIMINATOR / CRITIC
# ==================================================================================================

print("=" * 100)
print("8. DEFINE DISCRIMINATOR / CRITIC")
print("=" * 100)


class SPPGANCritic(nn.Module):
    """
    SPP-GAN scalar critic.

    Input:
        Differentiable transformed real or synthetic tabular
        representation.

    Output:
        One unrestricted scalar critic score per sample.

    The critic does NOT use:
        - sigmoid activation
        - binary cross-entropy
        - dropout
        - probability interpretation

    The critic is compatible with the WGAN-style adversarial
    objective used by SPP-GAN.
    """

    def __init__(
        self,
        input_dim,
        hidden_dim_1=256,
        hidden_dim_2=256,
    ):
        super().__init__()

        self.input_dim = int(input_dim)
        self.hidden_dim_1 = int(hidden_dim_1)
        self.hidden_dim_2 = int(hidden_dim_2)

        if self.input_dim <= 0:
            raise ValueError(
                "input_dim must be positive."
            )

        if self.hidden_dim_1 <= 0:
            raise ValueError(
                "hidden_dim_1 must be positive."
            )

        if self.hidden_dim_2 <= 0:
            raise ValueError(
                "hidden_dim_2 must be positive."
            )

        # ------------------------------------------------------------------------------------------
        # Scalar WGAN-style critic
        # ------------------------------------------------------------------------------------------

        self.network = nn.Sequential(
            nn.Linear(
                self.input_dim,
                self.hidden_dim_1,
            ),
            nn.LeakyReLU(0.2),

            nn.Linear(
                self.hidden_dim_1,
                self.hidden_dim_2,
            ),
            nn.LeakyReLU(0.2),

            nn.Linear(
                self.hidden_dim_2,
                1,
            ),
        )

    def forward(self, x):
        """
        Return one unrestricted scalar score per sample.

        Expected input shape:
            [batch_size, input_dim]

        Returned shape:
            [batch_size]
        """

        if x.ndim != 2:
            raise ValueError(
                "Critic input must be a 2D tensor "
                "[batch_size, input_dim]."
            )

        if x.shape[1] != self.input_dim:
            raise ValueError(
                f"Expected critic input dimension "
                f"{self.input_dim}, received {x.shape[1]}."
            )

        score = self.network(x)

        # Explicitly return one scalar score per sample.
        return score.squeeze(-1)

    def summary(self):
        return {
            "architecture": "MLP scalar WGAN-style critic",
            "input_dim": self.input_dim,
            "hidden_dim_1": self.hidden_dim_1,
            "hidden_dim_2": self.hidden_dim_2,
            "output_dim": 1,
            "output_type": "unrestricted_scalar_score",
            "sigmoid": False,
            "dropout": False,
        }


print("✓ Scalar WGAN-style critic architecture defined.")
print("✓ No sigmoid activation.")
print("✓ No binary classification output.")
print("✓ No dropout.")
print("✓ One unrestricted scalar score is produced per sample.")

8. DEFINE DISCRIMINATOR / CRITIC
✓ Scalar WGAN-style critic architecture defined.
✓ No sigmoid activation.
✓ No binary classification output.
✓ No dropout.
✓ One unrestricted scalar score is produced per sample.


In [34]:
# ==================================================================================================
# 9. DEFINE OUTPUT LAYERS
# ==================================================================================================

print("=" * 100)
print("9. DEFINE OUTPUT LAYERS")
print("=" * 100)


class SPPGANOutputLayers:
    """
    Unified SPP-GAN output-layer specification.

    The generator produces two logical output groups:

    1. Numerical outputs
       - One continuous output per numerical feature.

    2. Categorical outputs
       - One categorical head per categorical feature.
       - Each head contains one logit/probability for each category.

    During differentiable training, categorical heads are represented
    through Gumbel-Softmax probabilities.

    Final categorical decoding is handled by the downstream synthetic
    data generation/decoding stage.

    The authoritative transformed tensor dimension remains delegated
    to Notebook 02 and is not reconstructed here.
    """

    def __init__(
        self,
        num_numerical,
        categorical_cardinalities,
        transformed_dim=None,
    ):
        self.num_numerical = int(
            num_numerical
        )

        self.categorical_cardinalities = [
            int(v)
            for v in categorical_cardinalities
        ]

        self._transformed_dim = (
            int(transformed_dim)
            if transformed_dim is not None
            else None
        )

        # ------------------------------------------------------------------------------------------
        # Validation
        # ------------------------------------------------------------------------------------------

        if self.num_numerical < 0:
            raise ValueError(
                "num_numerical must be non-negative."
            )

        if any(
            cardinality <= 0
            for cardinality
            in self.categorical_cardinalities
        ):
            raise ValueError(
                "All categorical cardinalities must be positive."
            )

        if (
            self._transformed_dim is not None
            and self._transformed_dim <= 0
        ):
            raise ValueError(
                "transformed_dim must be positive "
                "when provided."
            )

        if (
            self.num_numerical == 0
            and len(self.categorical_cardinalities) == 0
        ):
            raise ValueError(
                "At least one numerical or categorical "
                "output feature is required."
            )

    # ----------------------------------------------------------------------------------------------
    # Numerical output dimension
    # ----------------------------------------------------------------------------------------------

    @property
    def numerical_output_dim(self):
        return self.num_numerical

    # ----------------------------------------------------------------------------------------------
    # Categorical head count
    # ----------------------------------------------------------------------------------------------

    @property
    def categorical_head_count(self):
        return len(
            self.categorical_cardinalities
        )

    # ----------------------------------------------------------------------------------------------
    # Logical categorical representation dimension
    # ----------------------------------------------------------------------------------------------

    @property
    def categorical_representation_dim(self):
        return sum(
            self.categorical_cardinalities
        )

    # ----------------------------------------------------------------------------------------------
    # Logical output dimension
    # ----------------------------------------------------------------------------------------------

    @property
    def logical_output_dim(self):
        """
        Dimension of the logical differentiable representation:

            numerical outputs
            +
            one-hot/Gumbel-Softmax categorical representation.
        """

        return (
            self.num_numerical
            + self.categorical_representation_dim
        )

    # ----------------------------------------------------------------------------------------------
    # Transformed representation dimension
    # ----------------------------------------------------------------------------------------------

    @property
    def transformed_dim(self):
        return self._transformed_dim

    def set_transformed_dim(
        self,
        transformed_dim,
    ):
        transformed_dim = int(
            transformed_dim
        )

        if transformed_dim <= 0:
            raise ValueError(
                "transformed_dim must be positive."
            )

        self._transformed_dim = transformed_dim

    # ----------------------------------------------------------------------------------------------
    # Generator-output contract
    # ----------------------------------------------------------------------------------------------

    def validate_generator_output(
        self,
        generator_output,
        batch_size=None,
    ):
        """
        Validate the structured output returned by SPP-GANGenerator.
        """

        if not isinstance(
            generator_output,
            dict,
        ):
            raise TypeError(
                "Generator output must be a dictionary."
            )

        required_keys = {
            "hidden",
            "numerical",
            "categorical_logits",
            "categorical_probabilities",
        }

        missing_keys = (
            required_keys
            - set(generator_output.keys())
        )

        if missing_keys:
            raise ValueError(
                "Generator output is missing required "
                f"keys: {sorted(missing_keys)}"
            )

        numerical = generator_output[
            "numerical"
        ]

        categorical_logits = generator_output[
            "categorical_logits"
        ]

        categorical_probabilities = generator_output[
            "categorical_probabilities"
        ]

        # ------------------------------------------------------------------------------------------
        # Numerical output validation
        # ------------------------------------------------------------------------------------------

        if self.num_numerical > 0:

            if numerical is None:
                raise ValueError(
                    "Numerical output is missing despite "
                    "num_numerical > 0."
                )

            if numerical.ndim != 2:
                raise ValueError(
                    "Numerical output must be a 2D tensor."
                )

            if numerical.shape[1] != (
                self.num_numerical
            ):
                raise ValueError(
                    "Numerical output dimension mismatch: "
                    f"expected={self.num_numerical}, "
                    f"found={numerical.shape[1]}."
                )

        else:

            if numerical is not None:
                raise ValueError(
                    "Numerical output must be None when "
                    "num_numerical == 0."
                )

        # ------------------------------------------------------------------------------------------
        # Categorical head-count validation
        # ------------------------------------------------------------------------------------------

        if len(categorical_logits) != (
            self.categorical_head_count
        ):
            raise ValueError(
                "Categorical-logit head count mismatch: "
                f"expected={self.categorical_head_count}, "
                f"found={len(categorical_logits)}."
            )

        if len(categorical_probabilities) != (
            self.categorical_head_count
        ):
            raise ValueError(
                "Categorical-probability head count mismatch: "
                f"expected={self.categorical_head_count}, "
                f"found={len(categorical_probabilities)}."
            )

        # ------------------------------------------------------------------------------------------
        # Per-head categorical validation
        # ------------------------------------------------------------------------------------------

        for index, cardinality in enumerate(
            self.categorical_cardinalities
        ):

            logits = categorical_logits[
                index
            ]

            probabilities = (
                categorical_probabilities[index]
            )

            if logits.ndim != 2:
                raise ValueError(
                    f"Categorical logits at head {index} "
                    "must be 2D."
                )

            if probabilities.ndim != 2:
                raise ValueError(
                    f"Categorical probabilities at head "
                    f"{index} must be 2D."
                )

            if logits.shape[1] != cardinality:
                raise ValueError(
                    f"Categorical logit dimension mismatch "
                    f"at head {index}: "
                    f"expected={cardinality}, "
                    f"found={logits.shape[1]}."
                )

            if probabilities.shape[1] != cardinality:
                raise ValueError(
                    f"Categorical probability dimension mismatch "
                    f"at head {index}: "
                    f"expected={cardinality}, "
                    f"found={probabilities.shape[1]}."
                )

            if logits.shape[0] != probabilities.shape[0]:
                raise ValueError(
                    f"Batch-size mismatch between categorical "
                    f"logits and probabilities at head {index}."
                )

        # ------------------------------------------------------------------------------------------
        # Batch-size validation
        # ------------------------------------------------------------------------------------------

        if batch_size is not None:

            batch_size = int(
                batch_size
            )

            if numerical is not None:
                if numerical.shape[0] != batch_size:
                    raise ValueError(
                        "Numerical output batch size does "
                        "not match expected batch size."
                    )

            for tensor_group in (
                categorical_logits,
                categorical_probabilities,
            ):
                for tensor in tensor_group:
                    if tensor.shape[0] != batch_size:
                        raise ValueError(
                            "Categorical output batch size "
                            "does not match expected batch size."
                        )

        return True

    # ----------------------------------------------------------------------------------------------
    # Summary
    # ----------------------------------------------------------------------------------------------

    def summary(self):

        return {
            "numerical_output_dim": (
                self.num_numerical
            ),
            "categorical_output_dimensions": (
                list(
                    self.categorical_cardinalities
                )
            ),
            "categorical_head_count": (
                self.categorical_head_count
            ),
            "categorical_representation_dim": (
                self.categorical_representation_dim
            ),
            "logical_output_dim": (
                self.logical_output_dim
            ),
            "transformed_dim": (
                self.transformed_dim
            ),
            "categorical_training_mechanism": (
                "gumbel_softmax"
            ),
            "final_categorical_decoding": (
                "downstream_decoding_stage"
            ),
        }


print("✓ Unified SPP-GAN output-layer interface defined.")
print("✓ Numerical output dimension validated.")
print("✓ Categorical cardinalities validated.")
print("✓ Categorical head count defined explicitly.")
print("✓ Differentiable categorical representation dimension defined.")
print("✓ Generator-output contract validation implemented.")
print("✓ Gumbel-Softmax retained for differentiable categorical training.")
print("✓ Final categorical decoding remains downstream.")
print("✓ Transformed dimension remains delegated to Notebook 02.")
print("✓ SECTION 9 STATUS: PASS")

9. DEFINE OUTPUT LAYERS
✓ Unified SPP-GAN output-layer interface defined.
✓ Numerical output dimension validated.
✓ Categorical cardinalities validated.
✓ Categorical head count defined explicitly.
✓ Differentiable categorical representation dimension defined.
✓ Generator-output contract validation implemented.
✓ Gumbel-Softmax retained for differentiable categorical training.
✓ Final categorical decoding remains downstream.
✓ Transformed dimension remains delegated to Notebook 02.
✓ SECTION 9 STATUS: PASS


In [35]:
# ==================================================================================================
# 10. DEFINE NUMERICAL OUTPUT MECHANISM
# ==================================================================================================

print("=" * 100)
print("10. DEFINE NUMERICAL OUTPUT MECHANISM")
print("=" * 100)


class NumericalOutputMechanism:
    """
    Numerical output mechanism for SPP-GAN.

    The generator produces one continuous output for each numerical
    feature. The activation used here must be compatible with the
    authoritative numerical representation established by Notebook 02.

    Supported activations:
        - identity : unbounded continuous representation
        - tanh     : bounded representation in [-1, 1]

    No independent normalization or scaling assumption is introduced
    by this section.

    Dataset-specific preprocessing and inverse transformation remain
    delegated to the authoritative Notebook 02 preprocessing artifacts.
    """

    SUPPORTED_ACTIVATIONS = {
        "identity",
        "tanh",
    }

    def __init__(
        self,
        activation="identity",
        output_range=None,
    ):
        self.activation = str(
            activation
        ).strip().lower()

        if self.activation not in (
            self.SUPPORTED_ACTIVATIONS
        ):
            raise ValueError(
                "Unsupported numerical activation: "
                f"{self.activation}. "
                f"Supported activations: "
                f"{sorted(self.SUPPORTED_ACTIVATIONS)}."
            )

        # ------------------------------------------------------------------------------------------
        # Validate output range
        # ------------------------------------------------------------------------------------------

        if output_range is None:

            self.output_range = None

        else:

            if not isinstance(
                output_range,
                (list, tuple),
            ):
                raise TypeError(
                    "output_range must be a list or tuple "
                    "containing exactly two values."
                )

            if len(output_range) != 2:
                raise ValueError(
                    "output_range must contain exactly "
                    "two values."
                )

            lower = float(
                output_range[0]
            )
            upper = float(
                output_range[1]
            )

            if not (
                np.isfinite(lower)
                and np.isfinite(upper)
            ):
                raise ValueError(
                    "output_range bounds must be finite."
                )

            if lower >= upper:
                raise ValueError(
                    "output_range lower bound must be "
                    "smaller than the upper bound."
                )

            self.output_range = [
                lower,
                upper,
            ]

        # ------------------------------------------------------------------------------------------
        # Activation/range consistency
        # ------------------------------------------------------------------------------------------

        if (
            self.activation == "tanh"
            and self.output_range is not None
            and self.output_range != [-1.0, 1.0]
        ):
            raise ValueError(
                "When activation='tanh', output_range must be "
                "None or [-1, 1]."
            )

    # ----------------------------------------------------------------------------------------------
    # Apply activation
    # ----------------------------------------------------------------------------------------------

    def apply(self, x):
        """
        Apply the configured numerical output activation.

        Parameters
        ----------
        x : torch.Tensor
            Raw numerical generator output.

        Returns
        -------
        torch.Tensor
            Activated numerical representation.
        """

        if not isinstance(
            x,
            torch.Tensor,
        ):
            raise TypeError(
                "Numerical output must be a torch.Tensor."
            )

        if x.ndim != 2:
            raise ValueError(
                "Numerical output must be a 2D tensor "
                "[batch_size, num_numerical]."
            )

        if self.activation == "identity":

            return x

        if self.activation == "tanh":

            return torch.tanh(x)

        raise RuntimeError(
            f"Unsupported numerical activation: "
            f"{self.activation}"
        )

    # ----------------------------------------------------------------------------------------------
    # Validate output tensor
    # ----------------------------------------------------------------------------------------------

    def validate_output(
        self,
        x,
    ):
        """
        Validate the numerical output against the configured
        activation and range.
        """

        if not isinstance(
            x,
            torch.Tensor,
        ):
            raise TypeError(
                "Numerical output must be a torch.Tensor."
            )

        if x.ndim != 2:
            raise ValueError(
                "Numerical output must be a 2D tensor."
            )

        if not torch.isfinite(x).all():
            raise ValueError(
                "Numerical output contains NaN or infinite values."
            )

        if self.activation == "tanh":

            tolerance = 1e-6

            if torch.any(
                x < -1.0 - tolerance
            ) or torch.any(
                x > 1.0 + tolerance
            ):
                raise ValueError(
                    "tanh numerical output must lie "
                    "within [-1, 1]."
                )

        if self.output_range is not None:

            lower, upper = self.output_range
            tolerance = 1e-6

            if torch.any(
                x < lower - tolerance
            ) or torch.any(
                x > upper + tolerance
            ):
                raise ValueError(
                    "Numerical output lies outside "
                    f"the configured range "
                    f"[{lower}, {upper}]."
                )

        return True

    # ----------------------------------------------------------------------------------------------
    # Summary
    # ----------------------------------------------------------------------------------------------

    def summary(self):

        return {
            "type": "continuous",
            "activation": self.activation,
            "output_range": (
                list(self.output_range)
                if self.output_range is not None
                else None
            ),
            "bounded": (
                self.activation == "tanh"
                or self.output_range is not None
            ),
            "inverse_transform": (
                "Notebook_02_preprocessing_artifacts"
            ),
        }


# --------------------------------------------------------------------------------------------------
# 1. SPP-GAN numerical output configuration
# --------------------------------------------------------------------------------------------------
#
# IMPORTANT:
# Notebook 02 remains authoritative for the actual numerical
# representation. Therefore this section does not impose an artificial
# normalization range.
#
# The current architecture uses identity because the transformed
# numerical representation is delegated to Notebook 02.
#
# If Notebook 02 is later verified to use a bounded [-1, 1]
# representation, this configuration must be changed consistently
# to activation="tanh", output_range=[-1.0, 1.0].
# --------------------------------------------------------------------------------------------------

NUMERICAL_OUTPUT = NumericalOutputMechanism(
    activation="identity",
    output_range=None,
)


# --------------------------------------------------------------------------------------------------
# 2. Configuration validation
# --------------------------------------------------------------------------------------------------

NUMERICAL_OUTPUT_SUMMARY = (
    NUMERICAL_OUTPUT.summary()
)

if NUMERICAL_OUTPUT.activation not in (
    NumericalOutputMechanism.SUPPORTED_ACTIVATIONS
):
    raise RuntimeError(
        "Invalid numerical output activation."
    )

if (
    NUMERICAL_OUTPUT.activation == "tanh"
    and NUMERICAL_OUTPUT.output_range not in (
        None,
        [-1.0, 1.0],
    )
):
    raise RuntimeError(
        "Inconsistent tanh numerical-output configuration."
    )


# --------------------------------------------------------------------------------------------------
# 3. Functional validation
# --------------------------------------------------------------------------------------------------

_test_input = torch.tensor(
    [
        [-2.0, -0.5, 0.0, 0.5, 2.0],
        [1.0, -1.0, 0.25, -0.25, 0.75],
    ],
    dtype=torch.float32,
)

_test_output = NUMERICAL_OUTPUT.apply(
    _test_input
)

NUMERICAL_OUTPUT.validate_output(
    _test_output
)

if _test_output.shape != _test_input.shape:
    raise RuntimeError(
        "Numerical output mechanism changed "
        "the tensor shape."
    )

del _test_input
del _test_output


# --------------------------------------------------------------------------------------------------
# 4. Final reporting
# --------------------------------------------------------------------------------------------------

print(
    f"✓ Numerical output activation : "
    f"{NUMERICAL_OUTPUT.activation}"
)

print(
    f"✓ Output range                : "
    f"{NUMERICAL_OUTPUT.output_range}"
)

print(
    f"✓ Bounded representation      : "
    f"{NUMERICAL_OUTPUT_SUMMARY['bounded']}"
)

print(
    "✓ Numerical output shape preserved."
)

print(
    "✓ Numerical output finiteness validation : PASS"
)

print(
    "✓ Activation/range consistency validation : PASS"
)

print(
    "✓ Numerical representation remains delegated "
    "to the authoritative Notebook 02 transformation."
)

print(
    "✓ Inverse transformation remains delegated "
    "to Notebook 02 preprocessing artifacts."
)

print(
    "✓ No independent normalization assumption imposed."
)

print(
    "✓ SECTION 10 STATUS: PASS"
)

10. DEFINE NUMERICAL OUTPUT MECHANISM
✓ Numerical output activation : identity
✓ Output range                : None
✓ Bounded representation      : False
✓ Numerical output shape preserved.
✓ Numerical output finiteness validation : PASS
✓ Activation/range consistency validation : PASS
✓ Numerical representation remains delegated to the authoritative Notebook 02 transformation.
✓ Inverse transformation remains delegated to Notebook 02 preprocessing artifacts.
✓ No independent normalization assumption imposed.
✓ SECTION 10 STATUS: PASS


In [36]:
# ==================================================================================================
# 11. DEFINE CATEGORICAL OUTPUT MECHANISM
# ==================================================================================================

print("=" * 100)
print("11. DEFINE CATEGORICAL OUTPUT MECHANISM")
print("=" * 100)


class CategoricalOutputMechanism:
    """
    Categorical output mechanism for SPP-GAN.

    Each categorical generator head produces one logit for every
    category of the corresponding categorical feature.

    During differentiable training:
        logits -> Gumbel-Softmax -> differentiable categorical vector

    During inspection/evaluation:
        logits -> ordinary Softmax -> categorical probabilities

    During final synthetic-data generation:
        logits -> Softmax -> multinomial category sampling

    Hard categorical sampling is kept outside the differentiable
    generator-training path.
    """

    activation = "gumbel_softmax"

    # ----------------------------------------------------------------------------------------------
    # Gumbel-Softmax probabilities
    # ----------------------------------------------------------------------------------------------

    @staticmethod
    def probabilities(
        logits,
        temperature=1.0,
        hard=False,
    ):
        """
        Produce differentiable categorical outputs.

        Parameters
        ----------
        logits : torch.Tensor
            Categorical logits with shape
            [batch_size, num_categories].

        temperature : float
            Positive Gumbel-Softmax temperature.

        hard : bool
            If True, return hard one-hot values using the
            straight-through estimator.
        """

        if not isinstance(
            logits,
            torch.Tensor,
        ):
            raise TypeError(
                "logits must be a torch.Tensor."
            )

        if logits.ndim != 2:
            raise ValueError(
                "Categorical logits must be a 2D tensor "
                "[batch_size, num_categories]."
            )

        temperature = float(
            temperature
        )

        if temperature <= 0:
            raise ValueError(
                "Gumbel-Softmax temperature must be positive."
            )

        if not torch.isfinite(logits).all():
            raise ValueError(
                "Categorical logits contain NaN or infinite values."
            )

        probabilities = F.gumbel_softmax(
            logits,
            tau=temperature,
            hard=bool(hard),
            dim=-1,
        )

        if not torch.isfinite(
            probabilities
        ).all():
            raise RuntimeError(
                "Gumbel-Softmax produced NaN or infinite values."
            )

        return probabilities

    # ----------------------------------------------------------------------------------------------
    # Ordinary softmax probabilities
    # ----------------------------------------------------------------------------------------------

    @staticmethod
    def softmax_probabilities(
        logits,
    ):
        """
        Return ordinary categorical probabilities.

        Intended for inspection, evaluation, or final sampling.
        """

        if not isinstance(
            logits,
            torch.Tensor,
        ):
            raise TypeError(
                "logits must be a torch.Tensor."
            )

        if logits.ndim != 2:
            raise ValueError(
                "Categorical logits must be a 2D tensor "
                "[batch_size, num_categories]."
            )

        if not torch.isfinite(
            logits
        ).all():
            raise ValueError(
                "Categorical logits contain NaN or infinite values."
            )

        probabilities = F.softmax(
            logits,
            dim=-1,
        )

        if not torch.isfinite(
            probabilities
        ).all():
            raise RuntimeError(
                "Softmax produced NaN or infinite values."
            )

        return probabilities

    # ----------------------------------------------------------------------------------------------
    # Hard categorical sampling
    # ----------------------------------------------------------------------------------------------

    @staticmethod
    def sample(
        logits,
    ):
        """
        Perform non-differentiable categorical sampling.

        The sampling distribution is obtained from the logits through
        ordinary Softmax.

        Intended for final synthetic-data generation only.

        This method must not be used inside the generator optimization
        path because the returned category indices are non-differentiable.
        """

        if not isinstance(
            logits,
            torch.Tensor,
        ):
            raise TypeError(
                "logits must be a torch.Tensor."
            )

        if logits.ndim != 2:
            raise ValueError(
                "Categorical logits must be a 2D tensor "
                "[batch_size, num_categories]."
            )

        probabilities = CategoricalOutputMechanism.softmax_probabilities(
            logits
        )

        # ------------------------------------------------------------------------------------------
        # Numerical safety
        # ------------------------------------------------------------------------------------------

        probability_sums = probabilities.sum(
            dim=-1
        )

        if not torch.allclose(
            probability_sums,
            torch.ones_like(
                probability_sums
            ),
            atol=1e-6,
            rtol=1e-6,
        ):
            raise RuntimeError(
                "Categorical probability vectors do not "
                "sum to one."
            )

        if torch.any(
            probabilities < 0
        ):
            raise RuntimeError(
                "Categorical probabilities contain negative values."
            )

        # ------------------------------------------------------------------------------------------
        # Non-differentiable categorical sampling
        # ------------------------------------------------------------------------------------------

        return torch.multinomial(
            probabilities,
            num_samples=1,
        ).squeeze(-1)

    # ----------------------------------------------------------------------------------------------
    # Validate one categorical head
    # ----------------------------------------------------------------------------------------------

    @staticmethod
    def validate_head(
        logits,
        expected_cardinality,
        batch_size=None,
    ):
        """
        Validate one categorical generator head.
        """

        expected_cardinality = int(
            expected_cardinality
        )

        if expected_cardinality <= 0:
            raise ValueError(
                "expected_cardinality must be positive."
            )

        if not isinstance(
            logits,
            torch.Tensor,
        ):
            raise TypeError(
                "Categorical logits must be a torch.Tensor."
            )

        if logits.ndim != 2:
            raise ValueError(
                "Categorical logits must be 2D."
            )

        if logits.shape[1] != expected_cardinality:
            raise ValueError(
                "Categorical head cardinality mismatch: "
                f"expected={expected_cardinality}, "
                f"found={logits.shape[1]}."
            )

        if batch_size is not None:

            batch_size = int(
                batch_size
            )

            if logits.shape[0] != batch_size:
                raise ValueError(
                    "Categorical head batch size mismatch: "
                    f"expected={batch_size}, "
                    f"found={logits.shape[0]}."
                )

        if not torch.isfinite(
            logits
        ).all():
            raise ValueError(
                "Categorical logits contain NaN or infinite values."
            )

        return True

    # ----------------------------------------------------------------------------------------------
    # Validate probability vector
    # ----------------------------------------------------------------------------------------------

    @staticmethod
    def validate_probabilities(
        probabilities,
        expected_cardinality,
        batch_size=None,
        tolerance=1e-6,
    ):
        """
        Validate categorical probability vectors.
        """

        expected_cardinality = int(
            expected_cardinality
        )

        if not isinstance(
            probabilities,
            torch.Tensor,
        ):
            raise TypeError(
                "Categorical probabilities must be a torch.Tensor."
            )

        if probabilities.ndim != 2:
            raise ValueError(
                "Categorical probabilities must be 2D."
            )

        if probabilities.shape[1] != expected_cardinality:
            raise ValueError(
                "Categorical probability cardinality mismatch: "
                f"expected={expected_cardinality}, "
                f"found={probabilities.shape[1]}."
            )

        if batch_size is not None:

            batch_size = int(
                batch_size
            )

            if probabilities.shape[0] != batch_size:
                raise ValueError(
                    "Categorical probability batch size mismatch."
                )

        if not torch.isfinite(
            probabilities
        ).all():
            raise ValueError(
                "Categorical probabilities contain NaN "
                "or infinite values."
            )

        if torch.any(
            probabilities < -tolerance
        ):
            raise ValueError(
                "Categorical probabilities contain negative values."
            )

        probability_sums = probabilities.sum(
            dim=-1
        )

        if not torch.allclose(
            probability_sums,
            torch.ones_like(
                probability_sums
            ),
            atol=tolerance,
            rtol=tolerance,
        ):
            raise ValueError(
                "Each categorical probability vector "
                "must sum to one."
            )

        return True

    # ----------------------------------------------------------------------------------------------
    # Validate generator categorical outputs
    # ----------------------------------------------------------------------------------------------

    @classmethod
    def validate_generator_outputs(
        cls,
        categorical_logits,
        categorical_probabilities,
        categorical_cardinalities,
        batch_size=None,
    ):
        """
        Validate all categorical generator heads against the
        authoritative categorical cardinalities.
        """

        expected_cardinalities = [
            int(v)
            for v in categorical_cardinalities
        ]

        if len(
            categorical_logits
        ) != len(expected_cardinalities):

            raise ValueError(
                "Number of categorical-logit heads does not "
                "match the number of categorical cardinalities."
            )

        if len(
            categorical_probabilities
        ) != len(expected_cardinalities):

            raise ValueError(
                "Number of categorical-probability heads does not "
                "match the number of categorical cardinalities."
            )

        for index, cardinality in enumerate(
            expected_cardinalities
        ):

            cls.validate_head(
                categorical_logits[index],
                expected_cardinality=cardinality,
                batch_size=batch_size,
            )

            cls.validate_probabilities(
                categorical_probabilities[index],
                expected_cardinality=cardinality,
                batch_size=batch_size,
            )

        return True

    # ----------------------------------------------------------------------------------------------
    # Summary
    # ----------------------------------------------------------------------------------------------

    @classmethod
    def summary(cls):

        return {
            "type": "categorical",
            "training_activation": cls.activation,
            "training_representation": (
                "differentiable_gumbel_softmax"
            ),
            "probability_mapping": "softmax",
            "hard_sampling": (
                "categorical_multinomial"
            ),
            "hard_decoding": "Notebook_13",
            "hard_sampling_during_training": False,
        }


# --------------------------------------------------------------------------------------------------
# 1. Instantiate categorical output mechanism
# --------------------------------------------------------------------------------------------------

CATEGORICAL_OUTPUT = (
    CategoricalOutputMechanism()
)


# --------------------------------------------------------------------------------------------------
# 2. Functional validation
# --------------------------------------------------------------------------------------------------

_TEST_BATCH_SIZE = 8
_TEST_CARDINALITY = 4

_test_logits = torch.randn(
    _TEST_BATCH_SIZE,
    _TEST_CARDINALITY,
    dtype=torch.float32,
)

_test_soft_probabilities = (
    CATEGORICAL_OUTPUT.softmax_probabilities(
        _test_logits
    )
)

CATEGORICAL_OUTPUT.validate_probabilities(
    _test_soft_probabilities,
    expected_cardinality=_TEST_CARDINALITY,
    batch_size=_TEST_BATCH_SIZE,
)

_test_gumbel_probabilities = (
    CATEGORICAL_OUTPUT.probabilities(
        _test_logits,
        temperature=1.0,
        hard=False,
    )
)

CATEGORICAL_OUTPUT.validate_probabilities(
    _test_gumbel_probabilities,
    expected_cardinality=_TEST_CARDINALITY,
    batch_size=_TEST_BATCH_SIZE,
)

_test_hard_probabilities = (
    CATEGORICAL_OUTPUT.probabilities(
        _test_logits,
        temperature=1.0,
        hard=True,
    )
)

CATEGORICAL_OUTPUT.validate_probabilities(
    _test_hard_probabilities,
    expected_cardinality=_TEST_CARDINALITY,
    batch_size=_TEST_BATCH_SIZE,
)

_test_samples = (
    CATEGORICAL_OUTPUT.sample(
        _test_logits
    )
)

if _test_samples.shape != (
    _TEST_BATCH_SIZE,
):
    raise RuntimeError(
        "Hard categorical sample shape validation failed."
    )

if torch.any(
    _test_samples < 0
) or torch.any(
    _test_samples >= _TEST_CARDINALITY
):
    raise RuntimeError(
        "Hard categorical samples contain "
        "invalid category indices."
    )

del _test_logits
del _test_soft_probabilities
del _test_gumbel_probabilities
del _test_hard_probabilities
del _test_samples


# --------------------------------------------------------------------------------------------------
# 3. Final reporting
# --------------------------------------------------------------------------------------------------

print(
    "✓ Categorical logits are converted through "
    "differentiable Gumbel-Softmax during training."
)

print(
    "✓ Positive Gumbel-Softmax temperature is enforced."
)

print(
    "✓ Softmax probabilities are available for "
    "inspection/evaluation."
)

print(
    "✓ Categorical probability normalization "
    "is validated."
)

print(
    "✓ Categorical head cardinality validation "
    "is implemented."
)

print(
    "✓ Hard categorical sampling uses the "
    "Softmax probability distribution."
)

print(
    "✓ Hard categorical sampling is isolated "
    "from the differentiable training path."
)

print(
    "✓ Hard decoding remains deferred to Notebook 13."
)

print(
    "✓ SECTION 11 STATUS: PASS"
)

11. DEFINE CATEGORICAL OUTPUT MECHANISM
✓ Categorical logits are converted through differentiable Gumbel-Softmax during training.
✓ Positive Gumbel-Softmax temperature is enforced.
✓ Softmax probabilities are available for inspection/evaluation.
✓ Categorical probability normalization is validated.
✓ Categorical head cardinality validation is implemented.
✓ Hard categorical sampling uses the Softmax probability distribution.
✓ Hard categorical sampling is isolated from the differentiable training path.
✓ Hard decoding remains deferred to Notebook 13.
✓ SECTION 11 STATUS: PASS


In [37]:
# ==================================================================================================
# 12. DEFINE MODEL LOSS INTERFACE
# ==================================================================================================

print("=" * 100)
print("12. DEFINE MODEL LOSS INTERFACE")
print("=" * 100)


class SPPGANLossInterface:
    """
    Loss interface for downstream SPP-GAN training.

    Generator objective:

        L_G = L_adv + lambda_stat * L_stat

    Statistical guidance:

        L_stat =
            lambda_m  * L_marg
            + lambda_mu * L_mom
            + lambda_d  * L_dep
            + lambda_c  * L_cat

    WGAN-style adversarial objectives:

        L_adv = -E[D(G(z,c))]

        L_D = E[D(fake)] - E[D(real)]

    Differential privacy is NOT represented as an additive loss term.
    Privacy is enforced through the downstream DP training mechanism
    and formally accounted for separately.

    The actual statistical-guidance computation is implemented
    downstream and is not fabricated in this architecture notebook.
    """

    def __init__(
        self,
        lambda_stat=1.0,
        lambda_m=1.0,
        lambda_mu=1.0,
        lambda_d=1.0,
        lambda_c=1.0,
    ):
        self.lambda_stat = float(
            lambda_stat
        )

        self.lambda_m = float(
            lambda_m
        )

        self.lambda_mu = float(
            lambda_mu
        )

        self.lambda_d = float(
            lambda_d
        )

        self.lambda_c = float(
            lambda_c
        )

        # ------------------------------------------------------------------------------------------
        # Validate loss weights
        # ------------------------------------------------------------------------------------------

        weights = {
            "lambda_stat": self.lambda_stat,
            "lambda_m": self.lambda_m,
            "lambda_mu": self.lambda_mu,
            "lambda_d": self.lambda_d,
            "lambda_c": self.lambda_c,
        }

        for name, value in weights.items():

            if not np.isfinite(value):
                raise ValueError(
                    f"{name} must be finite."
                )

            if value < 0:
                raise ValueError(
                    f"{name} must be non-negative."
                )

    # ----------------------------------------------------------------------------------------------
    # Tensor validation
    # ----------------------------------------------------------------------------------------------

    @staticmethod
    def _validate_score_tensor(
        score,
        name,
    ):
        """
        Validate a critic-score tensor.
        """

        if not isinstance(
            score,
            torch.Tensor,
        ):
            raise TypeError(
                f"{name} must be a torch.Tensor."
            )

        if score.numel() == 0:
            raise ValueError(
                f"{name} must not be empty."
            )

        if not torch.isfinite(
            score
        ).all():
            raise ValueError(
                f"{name} contains NaN or infinite values."
            )

    # ----------------------------------------------------------------------------------------------
    # Generator adversarial loss
    # ----------------------------------------------------------------------------------------------

    @staticmethod
    def adversarial_generator_loss(
        fake_score,
    ):
        """
        WGAN-style generator adversarial objective:

            L_adv = -E[D(G(z,c))]
        """

        SPPGANLossInterface._validate_score_tensor(
            fake_score,
            "fake_score",
        )

        loss = -fake_score.mean()

        if not torch.isfinite(loss):
            raise RuntimeError(
                "Generator adversarial loss is NaN or infinite."
            )

        return loss

    # ----------------------------------------------------------------------------------------------
    # Critic loss
    # ----------------------------------------------------------------------------------------------

    @staticmethod
    def critic_loss(
        real_score,
        fake_score,
    ):
        """
        WGAN-style critic objective:

            L_D = E[D(fake)] - E[D(real)]
        """

        SPPGANLossInterface._validate_score_tensor(
            real_score,
            "real_score",
        )

        SPPGANLossInterface._validate_score_tensor(
            fake_score,
            "fake_score",
        )

        loss = (
            fake_score.mean()
            - real_score.mean()
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                "Critic loss is NaN or infinite."
            )

        return loss

    # ----------------------------------------------------------------------------------------------
    # Statistical loss interface
    # ----------------------------------------------------------------------------------------------

    def statistical_loss(
        self,
        marginal_loss,
        moment_loss,
        dependency_loss,
        categorical_loss,
    ):
        """
        Assemble the four statistical-guidance components:

            L_stat =
                lambda_m  * L_marg
                + lambda_mu * L_mom
                + lambda_d  * L_dep
                + lambda_c  * L_cat

        The component losses themselves are supplied by the downstream
        statistical-guidance implementation.
        """

        component_losses = {
            "marginal_loss": marginal_loss,
            "moment_loss": moment_loss,
            "dependency_loss": dependency_loss,
            "categorical_loss": categorical_loss,
        }

        for name, value in component_losses.items():

            if not isinstance(
                value,
                torch.Tensor,
            ):
                raise TypeError(
                    f"{name} must be a torch.Tensor."
                )

            if value.numel() != 1:
                raise ValueError(
                    f"{name} must be a scalar tensor."
                )

            if not torch.isfinite(value):
                raise ValueError(
                    f"{name} is NaN or infinite."
                )

        loss = (
            self.lambda_m * marginal_loss
            + self.lambda_mu * moment_loss
            + self.lambda_d * dependency_loss
            + self.lambda_c * categorical_loss
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                "Statistical guidance loss is NaN or infinite."
            )

        return loss

    # ----------------------------------------------------------------------------------------------
    # Generator total loss
    # ----------------------------------------------------------------------------------------------

    def generator_loss(
        self,
        adversarial_loss,
        statistical_loss,
    ):
        """
        Compute the complete SPP-GAN generator objective:

            L_G = L_adv + lambda_stat * L_stat
        """

        if not isinstance(
            adversarial_loss,
            torch.Tensor,
        ):
            raise TypeError(
                "adversarial_loss must be a torch.Tensor."
            )

        if not isinstance(
            statistical_loss,
            torch.Tensor,
        ):
            raise TypeError(
                "statistical_loss must be a torch.Tensor."
            )

        if adversarial_loss.numel() != 1:
            raise ValueError(
                "adversarial_loss must be scalar."
            )

        if statistical_loss.numel() != 1:
            raise ValueError(
                "statistical_loss must be scalar."
            )

        if not torch.isfinite(
            adversarial_loss
        ):
            raise ValueError(
                "adversarial_loss is NaN or infinite."
            )

        if not torch.isfinite(
            statistical_loss
        ):
            raise ValueError(
                "statistical_loss is NaN or infinite."
            )

        loss = (
            adversarial_loss
            + self.lambda_stat * statistical_loss
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                "Generator total loss is NaN or infinite."
            )

        return loss

    # ----------------------------------------------------------------------------------------------
    # Summary
    # ----------------------------------------------------------------------------------------------

    def summary(self):

        return {
            "generator_objective": (
                "L_G = L_adv + lambda_stat * L_stat"
            ),
            "adversarial_component": (
                "L_adv = -E[D(G(z,c))]"
            ),
            "statistical_component": (
                "L_stat = "
                "lambda_m*L_marg + "
                "lambda_mu*L_mom + "
                "lambda_d*L_dep + "
                "lambda_c*L_cat"
            ),
            "lambda_stat": self.lambda_stat,
            "lambda_m": self.lambda_m,
            "lambda_mu": self.lambda_mu,
            "lambda_d": self.lambda_d,
            "lambda_c": self.lambda_c,
            "privacy_mechanism": (
                "DP training and privacy accounting; "
                "not an additive loss"
            ),
            "statistical_loss_implementation": (
                "downstream training/statistical-guidance stage"
            ),
        }


# --------------------------------------------------------------------------------------------------
# 1. Instantiate SPP-GAN loss interface
# --------------------------------------------------------------------------------------------------

LOSS_INTERFACE = SPPGANLossInterface(
    lambda_stat=1.0,
    lambda_m=1.0,
    lambda_mu=1.0,
    lambda_d=1.0,
    lambda_c=1.0,
)


# --------------------------------------------------------------------------------------------------
# 2. Architecture-level validation
# --------------------------------------------------------------------------------------------------

LOSS_SUMMARY = LOSS_INTERFACE.summary()

if LOSS_INTERFACE.lambda_stat < 0:
    raise RuntimeError(
        "lambda_stat validation failed."
    )

if any(
    value < 0
    for value in [
        LOSS_INTERFACE.lambda_m,
        LOSS_INTERFACE.lambda_mu,
        LOSS_INTERFACE.lambda_d,
        LOSS_INTERFACE.lambda_c,
    ]
):
    raise RuntimeError(
        "Statistical loss-weight validation failed."
    )


# --------------------------------------------------------------------------------------------------
# 3. Functional validation
# --------------------------------------------------------------------------------------------------

_TEST_REAL_SCORE = torch.tensor(
    [0.5, 1.0, -0.5, 0.25],
    dtype=torch.float32,
)

_TEST_FAKE_SCORE = torch.tensor(
    [0.1, 0.8, -0.2, 0.4],
    dtype=torch.float32,
)

_TEST_ADV_LOSS = (
    LOSS_INTERFACE.adversarial_generator_loss(
        _TEST_FAKE_SCORE
    )
)

_TEST_CRITIC_LOSS = (
    LOSS_INTERFACE.critic_loss(
        _TEST_REAL_SCORE,
        _TEST_FAKE_SCORE,
    )
)

_TEST_MARGINAL_LOSS = torch.tensor(
    0.10,
    dtype=torch.float32,
)

_TEST_MOMENT_LOSS = torch.tensor(
    0.20,
    dtype=torch.float32,
)

_TEST_DEPENDENCY_LOSS = torch.tensor(
    0.30,
    dtype=torch.float32,
)

_TEST_CATEGORICAL_LOSS = torch.tensor(
    0.40,
    dtype=torch.float32,
)

_TEST_STATISTICAL_LOSS = (
    LOSS_INTERFACE.statistical_loss(
        marginal_loss=_TEST_MARGINAL_LOSS,
        moment_loss=_TEST_MOMENT_LOSS,
        dependency_loss=_TEST_DEPENDENCY_LOSS,
        categorical_loss=_TEST_CATEGORICAL_LOSS,
    )
)

_TEST_GENERATOR_LOSS = (
    LOSS_INTERFACE.generator_loss(
        adversarial_loss=_TEST_ADV_LOSS,
        statistical_loss=_TEST_STATISTICAL_LOSS,
    )
)


# --------------------------------------------------------------------------------------------------
# 4. Numerical validation
# --------------------------------------------------------------------------------------------------

for name, value in {
    "generator adversarial loss": _TEST_ADV_LOSS,
    "critic loss": _TEST_CRITIC_LOSS,
    "statistical loss": _TEST_STATISTICAL_LOSS,
    "generator total loss": _TEST_GENERATOR_LOSS,
}.items():

    if not torch.isfinite(value):
        raise RuntimeError(
            f"{name} validation failed."
        )


# --------------------------------------------------------------------------------------------------
# 5. Cleanup test tensors
# --------------------------------------------------------------------------------------------------

del _TEST_REAL_SCORE
del _TEST_FAKE_SCORE
del _TEST_ADV_LOSS
del _TEST_CRITIC_LOSS
del _TEST_MARGINAL_LOSS
del _TEST_MOMENT_LOSS
del _TEST_DEPENDENCY_LOSS
del _TEST_CATEGORICAL_LOSS
del _TEST_STATISTICAL_LOSS
del _TEST_GENERATOR_LOSS


# --------------------------------------------------------------------------------------------------
# 6. Final reporting
# --------------------------------------------------------------------------------------------------

print(
    "✓ SPP-GAN loss interface defined."
)

print(
    "✓ Generator objective: "
    "L_G = L_adv + lambda_stat × L_stat."
)

print(
    "✓ WGAN-style generator adversarial loss defined."
)

print(
    "✓ WGAN-style critic loss defined."
)

print(
    "✓ Statistical loss interface contains "
    "marginal, moment, dependency, and categorical components."
)

print(
    "✓ All loss weights are validated as finite and non-negative."
)

print(
    "✓ Scalar loss and numerical-finiteness validation passed."
)

print(
    "✓ Privacy is treated as a DP training mechanism, "
    "not an additive loss."
)

print(
    "✓ Actual statistical-guidance computation remains "
    "downstream."
)

print(
    "✓ SECTION 12 STATUS: PASS"
)

12. DEFINE MODEL LOSS INTERFACE
✓ SPP-GAN loss interface defined.
✓ Generator objective: L_G = L_adv + lambda_stat × L_stat.
✓ WGAN-style generator adversarial loss defined.
✓ WGAN-style critic loss defined.
✓ Statistical loss interface contains marginal, moment, dependency, and categorical components.
✓ All loss weights are validated as finite and non-negative.
✓ Scalar loss and numerical-finiteness validation passed.
✓ Privacy is treated as a DP training mechanism, not an additive loss.
✓ Actual statistical-guidance computation remains downstream.
✓ SECTION 12 STATUS: PASS


In [38]:
# ==================================================================================================
# 13. DEFINE STATISTICAL-GUIDANCE INTERFACE
# ==================================================================================================

print("=" * 100)
print("13. DEFINE STATISTICAL-GUIDANCE INTERFACE")
print("=" * 100)


class SPPGANStatisticalGuidance:
    """
    Statistical-guidance interface for SPP-GAN.

    The statistical objective is:

        L_stat =
            lambda_m  * L_marg
            + lambda_mu * L_mom
            + lambda_d  * L_dep
            + lambda_c  * L_cat

    Components:

        L_marg
            Marginal distribution discrepancy.

        L_mom
            Numerical moment discrepancy.

        L_dep
            Numerical dependency/correlation discrepancy.

        L_cat
            Categorical distribution/dependency discrepancy.

    Notebook 03 provides the authoritative statistical reference.

    The actual differentiable implementation is supplied by the
    downstream statistical-guidance/training stage.

    IMPORTANT:
        Notebook 08 defines only the interface.
        It must not silently substitute a zero statistical loss for
        an unimplemented statistical-guidance mechanism.

    Statistical guidance is external reference information and is
    not concatenated to the record-level generator input tensor.
    """

    COMPONENT_NAMES = (
        "marginal",
        "moment",
        "dependency",
        "categorical",
    )

    def __init__(
        self,
        lambda_m=1.0,
        lambda_mu=1.0,
        lambda_d=1.0,
        lambda_c=1.0,
    ):
        self.lambda_m = float(
            lambda_m
        )

        self.lambda_mu = float(
            lambda_mu
        )

        self.lambda_d = float(
            lambda_d
        )

        self.lambda_c = float(
            lambda_c
        )

        # ------------------------------------------------------------------------------------------
        # Validate statistical-guidance weights
        # ------------------------------------------------------------------------------------------

        weights = {
            "lambda_m": self.lambda_m,
            "lambda_mu": self.lambda_mu,
            "lambda_d": self.lambda_d,
            "lambda_c": self.lambda_c,
        }

        for name, value in weights.items():

            if not np.isfinite(value):
                raise ValueError(
                    f"{name} must be finite."
                )

            if value < 0:
                raise ValueError(
                    f"{name} must be non-negative."
                )

        # ------------------------------------------------------------------------------------------
        # Architecture notebook does not enable statistical guidance.
        # ------------------------------------------------------------------------------------------

        self.enabled = False

        # Explicitly indicate that the implementation is not attached.
        self.method = (
            "interface_only_pending_downstream_implementation"
        )

    # ----------------------------------------------------------------------------------------------
    # Statistical component weights
    # ----------------------------------------------------------------------------------------------

    @property
    def weights(self):
        return {
            "lambda_m": self.lambda_m,
            "lambda_mu": self.lambda_mu,
            "lambda_d": self.lambda_d,
            "lambda_c": self.lambda_c,
        }

    # ----------------------------------------------------------------------------------------------
    # Validate statistical reference
    # ----------------------------------------------------------------------------------------------

    @staticmethod
    def validate_reference(
        statistical_reference,
    ):
        """
        Validate that a statistical reference object has been supplied
        before downstream loss computation.

        The detailed Notebook 03 schema remains authoritative and is
        validated when the reference is loaded.
        """

        if statistical_reference is None:
            raise ValueError(
                "A statistical_reference from Notebook 03 is required "
                "for statistical-guidance computation."
            )

        if not isinstance(
            statistical_reference,
            dict,
        ):
            raise TypeError(
                "statistical_reference must be a dictionary."
            )

        required_keys = {
            "dataset_profile",
            "feature_profiles",
            "pearson_correlation",
            "spearman_correlation",
            "categorical_dependency",
        }

        missing_keys = (
            required_keys
            - set(statistical_reference.keys())
        )

        if missing_keys:
            raise ValueError(
                "Statistical reference is missing required "
                f"Notebook 03 components: {sorted(missing_keys)}"
            )

        return True

    # ----------------------------------------------------------------------------------------------
    # Validate generated batch
    # ----------------------------------------------------------------------------------------------

    @staticmethod
    def validate_batch(
        synthetic_batch,
        name="synthetic_batch",
    ):
        """
        Validate a differentiable generated batch.
        """

        if not isinstance(
            synthetic_batch,
            torch.Tensor,
        ):
            raise TypeError(
                f"{name} must be a torch.Tensor."
            )

        if synthetic_batch.ndim != 2:
            raise ValueError(
                f"{name} must be a 2D tensor."
            )

        if synthetic_batch.shape[0] <= 0:
            raise ValueError(
                f"{name} must contain at least one sample."
            )

        if synthetic_batch.shape[1] <= 0:
            raise ValueError(
                f"{name} must contain at least one feature."
            )

        if not torch.isfinite(
            synthetic_batch
        ).all():
            raise ValueError(
                f"{name} contains NaN or infinite values."
            )

        return True

    # ----------------------------------------------------------------------------------------------
    # Validate real/generated batch compatibility
    # ----------------------------------------------------------------------------------------------

    @staticmethod
    def validate_batch_pair(
        real_batch,
        synthetic_batch,
    ):
        """
        Validate the basic tensor contract between real and synthetic
        transformed representations.

        Batch sizes may differ in downstream implementations, but the
        feature dimension must be identical.
        """

        SPPGANStatisticalGuidance.validate_batch(
            real_batch,
            name="real_batch",
        )

        SPPGANStatisticalGuidance.validate_batch(
            synthetic_batch,
            name="synthetic_batch",
        )

        if (
            real_batch.shape[1]
            != synthetic_batch.shape[1]
        ):
            raise ValueError(
                "Real and synthetic batches must have "
                "the same feature dimension: "
                f"real={real_batch.shape[1]}, "
                f"synthetic={synthetic_batch.shape[1]}."
            )

        return True

    # ----------------------------------------------------------------------------------------------
    # Component contract
    # ----------------------------------------------------------------------------------------------

    @staticmethod
    def validate_component_loss(
        loss,
        component_name,
    ):
        """
        Validate one scalar differentiable statistical component loss.
        """

        if not isinstance(
            loss,
            torch.Tensor,
        ):
            raise TypeError(
                f"{component_name} loss must be a torch.Tensor."
            )

        if loss.numel() != 1:
            raise ValueError(
                f"{component_name} loss must be scalar."
            )

        if not torch.isfinite(
            loss
        ):
            raise ValueError(
                f"{component_name} loss is NaN or infinite."
            )

        if not loss.requires_grad:
            raise ValueError(
                f"{component_name} loss must remain differentiable "
                "with respect to the generated representation."
            )

        return True

    # ----------------------------------------------------------------------------------------------
    # Assemble statistical loss
    # ----------------------------------------------------------------------------------------------

    def assemble_loss(
        self,
        marginal_loss,
        moment_loss,
        dependency_loss,
        categorical_loss,
    ):
        """
        Assemble the four differentiable statistical components:

            L_stat =
                lambda_m  * L_marg
                + lambda_mu * L_mom
                + lambda_d  * L_dep
                + lambda_c  * L_cat
        """

        component_losses = {
            "marginal": marginal_loss,
            "moment": moment_loss,
            "dependency": dependency_loss,
            "categorical": categorical_loss,
        }

        for component_name, loss in (
            component_losses.items()
        ):
            self.validate_component_loss(
                loss,
                component_name,
            )

        statistical_loss = (
            self.lambda_m * marginal_loss
            + self.lambda_mu * moment_loss
            + self.lambda_d * dependency_loss
            + self.lambda_c * categorical_loss
        )

        if not torch.isfinite(
            statistical_loss
        ):
            raise RuntimeError(
                "Assembled statistical loss is NaN or infinite."
            )

        if not statistical_loss.requires_grad:
            raise RuntimeError(
                "Assembled statistical loss is not differentiable."
            )

        return statistical_loss

    # ----------------------------------------------------------------------------------------------
    # Downstream implementation entry point
    # ----------------------------------------------------------------------------------------------

    def compute_loss(
        self,
        real_batch,
        synthetic_batch,
        statistical_reference=None,
    ):
        """
        Entry point reserved for the downstream differentiable
        statistical-guidance implementation.

        Notebook 08 deliberately does not provide a fake zero-loss
        implementation.

        Once statistical guidance is enabled, the downstream
        implementation must:

            1. Validate real_batch and synthetic_batch.
            2. Consume the authoritative Notebook 03 reference.
            3. Compute L_marg.
            4. Compute L_mom.
            5. Compute L_dep.
            6. Compute L_cat.
            7. Assemble L_stat.
            8. Preserve gradients with respect to synthetic_batch.

        Calling this method before the downstream implementation is
        attached therefore raises an explicit error.
        """

        self.validate_batch_pair(
            real_batch,
            synthetic_batch,
        )

        self.validate_reference(
            statistical_reference,
        )

        raise NotImplementedError(
            "Differentiable statistical-guidance computation is not "
            "implemented in Notebook 08. Attach the authoritative "
            "downstream implementation before enabling statistical "
            "guidance."
        )

    # ----------------------------------------------------------------------------------------------
    # Summary
    # ----------------------------------------------------------------------------------------------

    def summary(self):

        return {
            "enabled": self.enabled,
            "implementation": self.method,
            "training_notebook": "Notebook_09",
            "objective": (
                "L_stat = "
                "lambda_m*L_marg + "
                "lambda_mu*L_mom + "
                "lambda_d*L_dep + "
                "lambda_c*L_cat"
            ),
            "components": {
                "marginal": "L_marg",
                "moment": "L_mom",
                "dependency": "L_dep",
                "categorical": "L_cat",
            },
            "weights": self.weights,
            "reference_source": "Notebook_03",
            "differentiable_requirement": True,
            "record_level_input": False,
            "implementation_status": (
                "interface_only"
            ),
        }


# --------------------------------------------------------------------------------------------------
# 1. Instantiate statistical-guidance interface
# --------------------------------------------------------------------------------------------------

STATISTICAL_GUIDANCE = (
    SPPGANStatisticalGuidance(
        lambda_m=1.0,
        lambda_mu=1.0,
        lambda_d=1.0,
        lambda_c=1.0,
    )
)


# --------------------------------------------------------------------------------------------------
# 2. Validate interface configuration
# --------------------------------------------------------------------------------------------------

if not STATISTICAL_GUIDANCE.enabled:
    print(
        "✓ Statistical guidance remains disabled "
        "in the architecture notebook."
    )

if STATISTICAL_GUIDANCE.method != (
    "interface_only_pending_downstream_implementation"
):
    raise RuntimeError(
        "Unexpected statistical-guidance implementation state."
    )

if set(
    STATISTICAL_GUIDANCE.weights.keys()
) != {
    "lambda_m",
    "lambda_mu",
    "lambda_d",
    "lambda_c",
}:
    raise RuntimeError(
        "Statistical-guidance weight registry is incomplete."
    )


# --------------------------------------------------------------------------------------------------
# 3. Interface summary
# --------------------------------------------------------------------------------------------------

GUIDANCE_SUMMARY = (
    STATISTICAL_GUIDANCE.summary()
)

if GUIDANCE_SUMMARY[
    "reference_source"
] != "Notebook_03":
    raise RuntimeError(
        "Statistical reference source must be Notebook_03."
    )

if GUIDANCE_SUMMARY[
    "record_level_input"
]:
    raise RuntimeError(
        "Statistical guidance must not be treated as "
        "record-level generator input."
    )


# --------------------------------------------------------------------------------------------------
# 4. Final reporting
# --------------------------------------------------------------------------------------------------

print(
    "✓ Statistical-guidance interface defined."
)

print(
    "✓ Marginal, moment, dependency, and categorical "
    "components are explicitly registered."
)

print(
    "✓ All statistical-guidance weights are finite "
    "and non-negative."
)

print(
    "✓ Statistical reference is sourced from Notebook 03."
)

print(
    "✓ Statistical guidance remains external to the "
    "record-level input tensor."
)

print(
    "✓ Differentiable statistical loss is required "
    "for downstream training."
)

print(
    "✓ Zero-loss substitution has been explicitly prevented."
)

print(
    "✓ Actual statistical-guidance implementation remains "
    "deferred to the downstream implementation."
)

print(
    "✓ SECTION 13 STATUS: PASS"
)

13. DEFINE STATISTICAL-GUIDANCE INTERFACE
✓ Statistical guidance remains disabled in the architecture notebook.
✓ Statistical-guidance interface defined.
✓ Marginal, moment, dependency, and categorical components are explicitly registered.
✓ All statistical-guidance weights are finite and non-negative.
✓ Statistical reference is sourced from Notebook 03.
✓ Statistical guidance remains external to the record-level input tensor.
✓ Differentiable statistical loss is required for downstream training.
✓ Zero-loss substitution has been explicitly prevented.
✓ Actual statistical-guidance implementation remains deferred to the downstream implementation.
✓ SECTION 13 STATUS: PASS


In [41]:
# ==================================================================================================
# 14. DEFINE PRIVACY INTERFACE
# ==================================================================================================

print("=" * 100)
print("14. DEFINE PRIVACY INTERFACE")
print("=" * 100)

class SPPGANPrivacyInterface:
    """
    Privacy interface for SPP-GAN.

    Notebook 08:
        Defines the privacy interface only.
        No privacy mechanism is applied.
        No privacy budget is consumed.

    Notebook 10:
        Implements the differential privacy mechanism.

    Notebook 11:
        Performs privacy accounting and verification.

    Important:
        The presence of this interface alone does not establish
        differential privacy or an end-to-end privacy guarantee.
        All data-dependent operations within the claimed privacy
        boundary must be protected or accounted for downstream.
    """

    def __init__(
        self,
        enabled=False,
        epsilon=None,
        delta=None,
        max_grad_norm=None,
        accountant=None,
    ):
        self.enabled = bool(enabled)
        self.epsilon = epsilon
        self.delta = delta
        self.max_grad_norm = max_grad_norm
        self.accountant = accountant

        self._validate_configuration()

    def _validate_configuration(self):
        """Validate supplied privacy parameters."""

        if self.epsilon is not None:
            if not np.isfinite(float(self.epsilon)):
                raise ValueError("epsilon must be finite.")
            if float(self.epsilon) <= 0:
                raise ValueError("epsilon must be > 0.")

        if self.delta is not None:
            if not np.isfinite(float(self.delta)):
                raise ValueError("delta must be finite.")
            if not (0 < float(self.delta) < 1):
                raise ValueError("delta must satisfy 0 < delta < 1.")

        if self.max_grad_norm is not None:
            if not np.isfinite(float(self.max_grad_norm)):
                raise ValueError("max_grad_norm must be finite.")
            if float(self.max_grad_norm) <= 0:
                raise ValueError("max_grad_norm must be > 0.")

    def attach(self, module):
        """
        Interface for the future privacy mechanism.

        Actual DP attachment is implemented in Notebook 10.

        Privacy is NOT silently applied in Notebook 08.
        """
        if module is None:
            raise ValueError("module must not be None.")

        raise NotImplementedError(
            "Differential privacy attachment is implemented in Notebook 10."
        )

    def summary(self):
        return {
            "enabled": self.enabled,
            "epsilon": self.epsilon,
            "delta": self.delta,
            "max_grad_norm": self.max_grad_norm,
            "accountant": self.accountant,
            "mechanism_implementation": "Notebook_10",
            "accounting_implementation": "Notebook_11",
            "privacy_budget_consumed": False,
            "end_to_end_dp_claim": False,
        }


PRIVACY_INTERFACE = SPPGANPrivacyInterface(
    enabled=False,
)

# -----------------------------------------------------------------------------------------------
# Interface Validation
# -----------------------------------------------------------------------------------------------

assert PRIVACY_INTERFACE.enabled is False
assert PRIVACY_INTERFACE.epsilon is None
assert PRIVACY_INTERFACE.delta is None
assert PRIVACY_INTERFACE.max_grad_norm is None
assert PRIVACY_INTERFACE.summary()["privacy_budget_consumed"] is False
assert PRIVACY_INTERFACE.summary()["end_to_end_dp_claim"] is False

print("✓ Privacy interface defined.")
print("✓ Privacy mechanism deferred to Notebook 10.")
print("✓ Privacy accounting deferred to Notebook 11.")
print("✓ No privacy budget is consumed in Notebook 08.")
print("✓ End-to-end DP is not claimed in Notebook 08.")
print("✓ Privacy parameters are validated when supplied.")
print("✓ Silent privacy attachment is prevented.")
print("✓ SECTION 14 STATUS: PASS")

14. DEFINE PRIVACY INTERFACE
✓ Privacy interface defined.
✓ Privacy mechanism deferred to Notebook 10.
✓ Privacy accounting deferred to Notebook 11.
✓ No privacy budget is consumed in Notebook 08.
✓ End-to-end DP is not claimed in Notebook 08.
✓ Privacy parameters are validated when supplied.
✓ Silent privacy attachment is prevented.
✓ SECTION 14 STATUS: PASS


In [43]:
# ==================================================================================================
# 15. INITIALIZE SPP-GAN
# ==================================================================================================

print("=" * 100)
print("15. INITIALIZE SPP-GAN")
print("=" * 100)


# -----------------------------------------------------------------------------------------------
# 1. Imports required by Section 15
# -----------------------------------------------------------------------------------------------

import json
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn


# -----------------------------------------------------------------------------------------------
# 2. Architecture configuration
# -----------------------------------------------------------------------------------------------

SPPGAN_CONFIG = {
    "latent_dim": 128,

    "generator": {
        "hidden_dim_1": 256,
        "hidden_dim_2": 256,
    },

    "critic": {
        "hidden_dim_1": 256,
        "hidden_dim_2": 256,
    },

    "loss": {
        "lambda_stat": 1.0,
        "objective": "L_G = L_adv + lambda_stat * L_stat",
    },

    "representation": {
        "numerical_activation": "identity",
        "categorical_training_activation": "gumbel_softmax",
        "categorical_probability_mapping": "softmax",
        "hard_decoding": "Notebook_13",
        "transformation_source": "Notebook_02",
    },

    "privacy": {
        "enabled": False,
        "implementation_notebook": 10,
        "accounting_notebook": 11,
    },

    "training": {
        "enabled": False,
        "implementation_notebook": 12,
    },

    "seed": MASTER_SEED,
}


# -----------------------------------------------------------------------------------------------
# 3. Validate architecture configuration
# -----------------------------------------------------------------------------------------------

if not isinstance(
    SPPGAN_CONFIG["latent_dim"],
    int,
) or SPPGAN_CONFIG["latent_dim"] <= 0:
    raise ValueError(
        "SPP-GAN latent dimension must be a positive integer."
    )

for component in ("generator", "critic"):
    for key in ("hidden_dim_1", "hidden_dim_2"):
        value = SPPGAN_CONFIG[component][key]

        if not isinstance(value, int) or value <= 0:
            raise ValueError(
                f"{component}.{key} must be a positive integer."
            )

lambda_stat = SPPGAN_CONFIG["loss"]["lambda_stat"]

if not np.isfinite(float(lambda_stat)):
    raise ValueError(
        "lambda_stat must be finite."
    )

if float(lambda_stat) < 0:
    raise ValueError(
        "lambda_stat must be non-negative."
    )

if SPPGAN_CONFIG["representation"]["numerical_activation"] not in {
    "identity",
    "tanh",
}:
    raise ValueError(
        "Unsupported numerical activation."
    )

if SPPGAN_CONFIG["representation"][
    "categorical_training_activation"
] != "gumbel_softmax":
    raise ValueError(
        "SPP-GAN requires Gumbel-Softmax for categorical training."
    )

if SPPGAN_CONFIG["representation"][
    "categorical_probability_mapping"
] != "softmax":
    raise ValueError(
        "Categorical probability mapping must be softmax."
    )

if SPPGAN_CONFIG["privacy"]["enabled"]:
    raise ValueError(
        "Privacy must remain disabled in Notebook 08."
    )

if SPPGAN_CONFIG["training"]["enabled"]:
    raise ValueError(
        "Training must remain disabled in Notebook 08."
    )

print("✓ Architecture configuration validated.")
print("✓ Latent and hidden dimensions are valid.")
print("✓ Statistical-loss weight is valid.")
print("✓ Numerical activation configuration is valid.")
print("✓ Categorical representation configuration is valid.")
print("✓ Privacy remains disabled.")
print("✓ Training remains disabled.")


# -----------------------------------------------------------------------------------------------
# 4. Notebook 02 authoritative metadata locations
# -----------------------------------------------------------------------------------------------

NB02_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_02"
)

NB02_METADATA_ROOT = (
    NB02_ROOT
    / "schemas"
    / "metadata"
)

if not NB02_ROOT.exists():
    raise FileNotFoundError(
        f"Notebook 02 root not found:\n{NB02_ROOT}"
    )

if not NB02_METADATA_ROOT.exists():
    raise FileNotFoundError(
        f"Notebook 02 metadata root not found:\n"
        f"{NB02_METADATA_ROOT}"
    )

print("✓ Notebook 02 root verified.")
print("✓ Notebook 02 metadata root verified.")


# -----------------------------------------------------------------------------------------------
# 5. Load authoritative Notebook 02 preprocessing metadata
# -----------------------------------------------------------------------------------------------

def load_notebook_02_preprocessing_metadata(dataset_name):
    """
    Load authoritative Notebook 02 preprocessing metadata.

    Feature types, generative columns, target policy, transformed
    dimensions, and transformed feature names are inherited from
    Notebook 02.

    No feature types or transformed dimensions are inferred from
    sampled training data.
    """

    metadata_path = (
        NB02_METADATA_ROOT
        / f"{dataset_name}_preprocessing_metadata.json"
    )

    if not metadata_path.exists():
        raise FileNotFoundError(
            f"Notebook 02 preprocessing metadata not found:\n"
            f"{metadata_path}"
        )

    with open(
        metadata_path,
        "r",
        encoding="utf-8",
    ) as f:
        metadata = json.load(f)

    if metadata.get("dataset_id") != dataset_name:
        raise ValueError(
            f"Notebook 02 metadata identity mismatch: "
            f"expected={dataset_name!r}, "
            f"found={metadata.get('dataset_id')!r}."
        )

    required_keys = {
        "dataset_id",
        "training_rows",
        "input_columns",
        "numeric_columns",
        "categorical_columns",
        "target_column",
        "target_retained_in_training_dataset",
        "target_excluded_from_preprocessor_input",
        "identifier_columns",
        "provenance_column",
        "provenance_excluded_from_modeling",
        "identifiers_excluded_from_modeling",
        "generative_columns",
        "fit_dataset",
        "fit_scope",
        "fit_policy",
        "preprocessor_artifact",
        "schema_artifact",
        "preprocessing_feature_count",
        "generative_column_count",
        "target_retained_in_generative_schema",
        "target_excluded_from_transformed_features",
        "raw_target_manually_appended",
        "identifiers_excluded",
        "provenance_excluded_from_model_input",
        "transformed_feature_count",
        "transformed_feature_names",
    }

    missing_keys = required_keys.difference(
        metadata.keys()
    )

    if missing_keys:
        raise KeyError(
            f"Notebook 02 metadata for {dataset_name} is missing "
            f"required keys: {sorted(missing_keys)}"
        )

    return metadata


# -----------------------------------------------------------------------------------------------
# 6. Resolve exact categorical cardinalities
# -----------------------------------------------------------------------------------------------

def resolve_categorical_cardinalities(
    dataset_name,
    metadata,
):
    """
    Resolve categorical cardinalities from the fitted Notebook 02
    preprocessing artifact.

    Cardinalities are obtained from the fitted encoder rather than
    from a bounded sample of the training dataset.

    A cardinality of 1 is permitted because a categorical feature
    may contain only one observed training category.
    """

    categorical_columns = list(
        metadata["categorical_columns"]
    )

    if not categorical_columns:
        return []

    preprocessor_path = Path(
        metadata["preprocessor_artifact"]
    )

    if not preprocessor_path.exists():
        raise FileNotFoundError(
            f"Fitted preprocessor not found for {dataset_name}:\n"
            f"{preprocessor_path}"
        )

    fitted_preprocessor = joblib.load(
        preprocessor_path
    )

    # -------------------------------------------------------------------------------------------
    # Locate fitted categorical transformer
    # -------------------------------------------------------------------------------------------

    categorical_transformer = None

    if hasattr(
        fitted_preprocessor,
        "named_transformers_",
    ):
        named_transformers = (
            fitted_preprocessor.named_transformers_
        )

        for candidate_name in (
            "categorical",
            "cat",
            "categorical_transformer",
        ):
            if candidate_name in named_transformers:
                categorical_transformer = (
                    named_transformers[candidate_name]
                )
                break

    if categorical_transformer is None:
        raise RuntimeError(
            f"Unable to locate fitted categorical transformer "
            f"for {dataset_name}."
        )

    # -------------------------------------------------------------------------------------------
    # Locate fitted categorical encoder
    # -------------------------------------------------------------------------------------------

    encoder = None

    if hasattr(
        categorical_transformer,
        "named_steps",
    ):
        named_steps = (
            categorical_transformer.named_steps
        )

        for step_name in (
            "onehot",
            "encoder",
            "categorical_encoder",
        ):
            if step_name in named_steps:
                candidate_encoder = (
                    named_steps[step_name]
                )

                if hasattr(
                    candidate_encoder,
                    "categories_",
                ):
                    encoder = candidate_encoder
                    break

    elif hasattr(
        categorical_transformer,
        "categories_",
    ):
        encoder = categorical_transformer

    if encoder is None:
        raise RuntimeError(
            f"Unable to locate a fitted categorical encoder "
            f"with categories_ for {dataset_name}."
        )

    fitted_categories = list(
        encoder.categories_
    )

    if len(fitted_categories) != len(
        categorical_columns
    ):
        raise ValueError(
            f"Categorical encoder dimension mismatch for "
            f"{dataset_name}: "
            f"{len(fitted_categories)} fitted categorical blocks "
            f"vs {len(categorical_columns)} categorical columns."
        )

    # -------------------------------------------------------------------------------------------
    # Extract exact cardinalities
    # -------------------------------------------------------------------------------------------

    cardinalities = []

    for column, categories in zip(
        categorical_columns,
        fitted_categories,
    ):
        cardinality = len(categories)

        if cardinality < 1:
            raise ValueError(
                f"Invalid categorical cardinality for "
                f"'{column}' in {dataset_name}: "
                f"{cardinality}"
            )

        if cardinality == 1:
            print(
                f"  ⚠ Single-category feature: "
                f"{column} → cardinality=1"
            )

        cardinalities.append(
            int(cardinality)
        )

    return cardinalities


# -----------------------------------------------------------------------------------------------
# 7. Architecture registries
# -----------------------------------------------------------------------------------------------

SPPGAN_MODELS = {}
SPPGAN_ARCHITECTURES = {}


# -----------------------------------------------------------------------------------------------
# 8. Initialize one SPP-GAN architecture per dataset
# -----------------------------------------------------------------------------------------------

for dataset_name, schema in FEATURE_SCHEMAS.items():

    print("\n" + "-" * 100)
    print(
        f"Initializing SPP-GAN : {dataset_name}"
    )
    print("-" * 100)

    # -------------------------------------------------------------------------------------------
    # Load authoritative Notebook 02 metadata
    # -------------------------------------------------------------------------------------------

    metadata = (
        load_notebook_02_preprocessing_metadata(
            dataset_name
        )
    )

    numerical_columns = list(
        metadata["numeric_columns"]
    )

    categorical_columns = list(
        metadata["categorical_columns"]
    )

    generative_columns = list(
        metadata["generative_columns"]
    )

    target_column = metadata[
        "target_column"
    ]

    identifier_columns = list(
        metadata["identifier_columns"]
    )

    provenance_column = metadata[
        "provenance_column"
    ]

    transformed_dim = int(
        metadata["transformed_feature_count"]
    )

    transformed_feature_names = list(
        metadata["transformed_feature_names"]
    )

    # -------------------------------------------------------------------------------------------
    # Validate metadata against frozen Notebook 02 schema registry
    # -------------------------------------------------------------------------------------------

    if generative_columns != list(
        schema["generative_columns"]
    ):
        raise ValueError(
            f"Generative-column mismatch between Notebook 02 "
            f"metadata and Section 3 FEATURE_SCHEMAS for "
            f"{dataset_name}."
        )

    if target_column != schema["target"]:
        raise ValueError(
            f"Target mismatch for {dataset_name}: "
            f"Notebook 02 metadata={target_column!r}, "
            f"Section 3 schema={schema['target']!r}."
        )

    if len(generative_columns) != int(
        schema["generative_dimension"]
    ):
        raise ValueError(
            f"Generative dimension mismatch for "
            f"{dataset_name}."
        )

    if len(numerical_columns) + len(
        categorical_columns
    ) != len(generative_columns) - 1:
        raise ValueError(
            f"Feature-type count mismatch for {dataset_name}. "
            f"Numerical + categorical feature count must equal "
            f"generative dimension minus target."
        )

    if target_column in numerical_columns:
        raise ValueError(
            f"Target column '{target_column}' incorrectly appears "
            f"in numerical features for {dataset_name}."
        )

    if target_column in categorical_columns:
        raise ValueError(
            f"Target column '{target_column}' incorrectly appears "
            f"in categorical features for {dataset_name}."
        )

    if provenance_column in numerical_columns:
        raise ValueError(
            f"Provenance column '{provenance_column}' appears "
            f"in numerical features for {dataset_name}."
        )

    if provenance_column in categorical_columns:
        raise ValueError(
            f"Provenance column '{provenance_column}' appears "
            f"in categorical features for {dataset_name}."
        )

    if set(identifier_columns) & set(
        numerical_columns + categorical_columns
    ):
        raise ValueError(
            f"Identifier columns leaked into model features "
            f"for {dataset_name}."
        )

    # -------------------------------------------------------------------------------------------
    # Validate target / identifier / provenance policy
    # -------------------------------------------------------------------------------------------

    if not metadata[
        "target_retained_in_generative_schema"
    ]:
        raise ValueError(
            f"Target retention policy invalid for {dataset_name}."
        )

    if not metadata[
        "target_excluded_from_preprocessor_input"
    ]:
        raise ValueError(
            f"Target must be excluded from preprocessor input "
            f"for {dataset_name}."
        )

    if not metadata[
        "target_excluded_from_transformed_features"
    ]:
        raise ValueError(
            f"Target must be excluded from transformed features "
            f"for {dataset_name}."
        )

    if not metadata[
        "provenance_excluded_from_modeling"
    ]:
        raise ValueError(
            f"Provenance column must be excluded from modeling "
            f"for {dataset_name}."
        )

    if not metadata[
        "provenance_excluded_from_model_input"
    ]:
        raise ValueError(
            f"Provenance column must be excluded from model input "
            f"for {dataset_name}."
        )

    if not metadata[
        "identifiers_excluded_from_modeling"
    ]:
        raise ValueError(
            f"Identifier exclusion policy invalid for "
            f"{dataset_name}."
        )

    if not metadata[
        "identifiers_excluded"
    ]:
        raise ValueError(
            f"Identifier exclusion flag invalid for "
            f"{dataset_name}."
        )

    # -------------------------------------------------------------------------------------------
    # Validate train-only preprocessing policy
    # -------------------------------------------------------------------------------------------

    if metadata["fit_policy"] != "train_only":
        raise ValueError(
            f"Notebook 02 preprocessing for {dataset_name} "
            f"is not marked train_only."
        )

    if metadata["fit_scope"] != "training_features_only":
        raise ValueError(
            f"Notebook 02 preprocessing for {dataset_name} "
            f"is not marked training_features_only."
        )

    if metadata["fit_dataset"] != "train_only":
        raise ValueError(
            f"Notebook 02 fit dataset policy invalid for "
            f"{dataset_name}: {metadata['fit_dataset']}"
        )

    # -------------------------------------------------------------------------------------------
    # Resolve exact categorical cardinalities
    # -------------------------------------------------------------------------------------------

    categorical_cardinalities = (
        resolve_categorical_cardinalities(
            dataset_name,
            metadata,
        )
    )

    num_numerical = len(
        numerical_columns
    )

    num_categorical = len(
        categorical_columns
    )

    # -------------------------------------------------------------------------------------------
    # Validate categorical cardinality contract
    # -------------------------------------------------------------------------------------------

    if len(categorical_cardinalities) != num_categorical:
        raise ValueError(
            f"Categorical cardinality count mismatch for "
            f"{dataset_name}: "
            f"expected={num_categorical}, "
            f"found={len(categorical_cardinalities)}."
        )

    if any(
        not isinstance(cardinality, int)
        or cardinality < 1
        for cardinality in categorical_cardinalities
    ):
        raise ValueError(
            f"Invalid categorical cardinality configuration "
            f"for {dataset_name}."
        )

    # -------------------------------------------------------------------------------------------
    # Validate transformed representation
    # -------------------------------------------------------------------------------------------

    if transformed_dim <= 0:
        raise ValueError(
            f"Invalid transformed dimension for "
            f"{dataset_name}: {transformed_dim}"
        )

    if len(transformed_feature_names) != transformed_dim:
        raise ValueError(
            f"Transformed feature-name count mismatch for "
            f"{dataset_name}: "
            f"count={len(transformed_feature_names)}, "
            f"declared={transformed_dim}."
        )

    # -------------------------------------------------------------------------------------------
    # Validate transformed-dimension contract
    # -------------------------------------------------------------------------------------------

    expected_transformed_dim = (
        num_numerical
        + sum(categorical_cardinalities)
    )

    if expected_transformed_dim != transformed_dim:
        raise ValueError(
            f"Transformed-dimension mismatch for "
            f"{dataset_name}: "
            f"expected={expected_transformed_dim}, "
            f"Notebook 02={transformed_dim}."
        )

    # -------------------------------------------------------------------------------------------
    # Initialize input representation
    # -------------------------------------------------------------------------------------------

    input_representation = (
        SPPGANInputRepresentation(
            num_numerical=num_numerical,
            num_categorical=num_categorical,
            categorical_cardinalities=(
                categorical_cardinalities
            ),
            conditional_dim=SPPGAN_CONDITIONAL_DIM,
            transformed_dim=transformed_dim,
        )
    )

    # -------------------------------------------------------------------------------------------
    # Initialize generator
    # -------------------------------------------------------------------------------------------

    generator = SPPGANGenerator(
        latent_dim=SPPGAN_CONFIG[
            "latent_dim"
        ],
        hidden_dim_1=SPPGAN_CONFIG[
            "generator"
        ]["hidden_dim_1"],
        hidden_dim_2=SPPGAN_CONFIG[
            "generator"
        ]["hidden_dim_2"],
        num_numerical=num_numerical,
        categorical_cardinalities=(
            categorical_cardinalities
        ),
        conditional_dim=SPPGAN_CONDITIONAL_DIM,
    )

    # -------------------------------------------------------------------------------------------
    # Initialize scalar WGAN-style critic
    # -------------------------------------------------------------------------------------------

    critic = SPPGANCritic(
        input_dim=transformed_dim,
        hidden_dim_1=SPPGAN_CONFIG[
            "critic"
        ]["hidden_dim_1"],
        hidden_dim_2=SPPGAN_CONFIG[
            "critic"
        ]["hidden_dim_2"],
    )

    # -------------------------------------------------------------------------------------------
    # Validate instantiated architecture contracts
    # -------------------------------------------------------------------------------------------

    if generator.num_numerical != num_numerical:
        raise ValueError(
            f"Generator numerical dimension mismatch for "
            f"{dataset_name}: "
            f"expected={num_numerical}, "
            f"found={generator.num_numerical}."
        )

    if list(
        generator.categorical_cardinalities
    ) != list(categorical_cardinalities):
        raise ValueError(
            f"Generator categorical cardinalities mismatch "
            f"for {dataset_name}."
        )

    if generator.latent_dim != SPPGAN_CONFIG[
        "latent_dim"
    ]:
        raise ValueError(
            f"Generator latent dimension mismatch for "
            f"{dataset_name}: "
            f"expected={SPPGAN_CONFIG['latent_dim']}, "
            f"found={generator.latent_dim}."
        )

    if generator.conditional_dim != (
        SPPGAN_CONDITIONAL_DIM
    ):
        raise ValueError(
            f"Generator conditional dimension mismatch "
            f"for {dataset_name}."
        )

    if critic.input_dim != transformed_dim:
        raise ValueError(
            f"Critic input dimension mismatch for "
            f"{dataset_name}: "
            f"expected={transformed_dim}, "
            f"found={critic.input_dim}."
        )

    # -------------------------------------------------------------------------------------------
    # Validate input representation contract
    # -------------------------------------------------------------------------------------------

    if input_representation.transformed_dim != transformed_dim:
        raise ValueError(
            f"Input representation transformed dimension "
            f"mismatch for {dataset_name}."
        )

    if input_representation.num_numerical != num_numerical:
        raise ValueError(
            f"Input representation numerical dimension "
            f"mismatch for {dataset_name}."
        )

    if input_representation.num_categorical != num_categorical:
        raise ValueError(
            f"Input representation categorical dimension "
            f"mismatch for {dataset_name}."
        )

    # -------------------------------------------------------------------------------------------
    # Store initialized models
    # -------------------------------------------------------------------------------------------

    SPPGAN_MODELS[dataset_name] = {
        "generator": generator,
        "critic": critic,
        "input_representation": (
            input_representation
        ),
    }

    # -------------------------------------------------------------------------------------------
    # Store architecture metadata
    # -------------------------------------------------------------------------------------------

    SPPGAN_ARCHITECTURES[dataset_name] = {
        "dataset": dataset_name,

        "target": target_column,

        "generative_columns": (
            generative_columns
        ),

        "identifier_columns": (
            identifier_columns
        ),

        "provenance_column": (
            provenance_column
        ),

        "numerical_columns": (
            numerical_columns
        ),

        "categorical_columns": (
            categorical_columns
        ),

        "categorical_cardinalities": (
            categorical_cardinalities
        ),

        "num_numerical": (
            num_numerical
        ),

        "num_categorical": (
            num_categorical
        ),

        "generative_dimension": (
            len(generative_columns)
        ),

        "transformed_dim": (
            transformed_dim
        ),

        "expected_transformed_dim": (
            expected_transformed_dim
        ),

        "transformed_feature_names": (
            transformed_feature_names
        ),

        "latent_dim": (
            SPPGAN_CONFIG["latent_dim"]
        ),

        "conditional_dim": (
            SPPGAN_CONDITIONAL_DIM
        ),

        "generator_hidden_dims": [
            SPPGAN_CONFIG["generator"][
                "hidden_dim_1"
            ],
            SPPGAN_CONFIG["generator"][
                "hidden_dim_2"
            ],
        ],

        "critic_hidden_dims": [
            SPPGAN_CONFIG["critic"][
                "hidden_dim_1"
            ],
            SPPGAN_CONFIG["critic"][
                "hidden_dim_2"
            ],
        ],

        "representation": {
            "numerical_activation": (
                SPPGAN_CONFIG[
                    "representation"
                ]["numerical_activation"]
            ),
            "categorical_training_activation": (
                SPPGAN_CONFIG[
                    "representation"
                ]["categorical_training_activation"]
            ),
            "categorical_probability_mapping": (
                SPPGAN_CONFIG[
                    "representation"
                ]["categorical_probability_mapping"]
            ),
            "transformation_source": (
                "Notebook_02"
            ),
        },

        "loss": {
            "objective": (
                "L_G = L_adv + "
                "lambda_stat * L_stat"
            ),
            "lambda_stat": (
                SPPGAN_CONFIG[
                    "loss"
                ]["lambda_stat"]
            ),
        },

        "privacy": {
            "enabled": False,
            "implementation_notebook": 10,
            "accounting_notebook": 11,
        },

        "training": {
            "enabled": False,
            "implementation_notebook": 12,
        },
    }

    # -------------------------------------------------------------------------------------------
    # Dataset-level architecture summary
    # -------------------------------------------------------------------------------------------

    print(
        f"Numerical features   : "
        f"{num_numerical}"
    )

    print(
        f"Categorical features : "
        f"{num_categorical}"
    )

    print(
        f"Generative dimension : "
        f"{len(generative_columns)}"
    )

    print(
        f"Transformed dimension: "
        f"{transformed_dim}"
    )

    print(
        f"Latent dimension     : "
        f"{SPPGAN_CONFIG['latent_dim']}"
    )

    print(
        f"Target               : "
        f"{target_column}"
    )

    print(
        f"Identifiers excluded : "
        f"{len(identifier_columns)}"
    )

    print(
        "✓ Transformed-dimension contract validated."
    )

    print(
        "✓ Generator output contract validated."
    )

    print(
        "✓ Generator latent dimension validated."
    )

    print(
        "✓ Generator conditional dimension validated."
    )

    print(
        "✓ Critic input dimension validated."
    )

    print(
        "✓ Input-representation dimension validated."
    )

    print(
        "✓ Categorical cardinalities validated."
    )


# -----------------------------------------------------------------------------------------------
# 9. Final architecture initialization validation
# -----------------------------------------------------------------------------------------------

expected_datasets = set(
    FEATURE_SCHEMAS.keys()
)

initialized_datasets = set(
    SPPGAN_MODELS.keys()
)

if initialized_datasets != expected_datasets:
    raise RuntimeError(
        "Initialized dataset registry does not match "
        "FEATURE_SCHEMAS."
    )

if set(SPPGAN_ARCHITECTURES.keys()) != expected_datasets:
    raise RuntimeError(
        "Architecture registry does not match "
        "FEATURE_SCHEMAS."
    )

for dataset_name in expected_datasets:

    model_bundle = SPPGAN_MODELS[
        dataset_name
    ]

    architecture = SPPGAN_ARCHITECTURES[
        dataset_name
    ]

    if not isinstance(
        model_bundle["generator"],
        nn.Module,
    ):
        raise TypeError(
            f"Generator for {dataset_name} is not a "
            f"PyTorch module."
        )

    if not isinstance(
        model_bundle["critic"],
        nn.Module,
    ):
        raise TypeError(
            f"Critic for {dataset_name} is not a "
            f"PyTorch module."
        )

    if architecture["transformed_dim"] != (
        architecture["expected_transformed_dim"]
    ):
        raise ValueError(
            f"Final transformed-dimension validation failed "
            f"for {dataset_name}."
        )

    if architecture["privacy"]["enabled"]:
        raise ValueError(
            f"Privacy must remain disabled for {dataset_name} "
            f"in Notebook 08."
        )

    if architecture["training"]["enabled"]:
        raise ValueError(
            f"Training must remain disabled for {dataset_name} "
            f"in Notebook 08."
        )


# -----------------------------------------------------------------------------------------------
# 10. Final architecture initialization status
# -----------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SPP-GAN ARCHITECTURE INITIALIZATION COMPLETE")
print("=" * 100)

print(
    f"✓ Datasets initialized : "
    f"{len(SPPGAN_MODELS)}"
)

print(
    f"✓ Dataset registry     : "
    f"{list(SPPGAN_MODELS.keys())}"
)

print(
    "✓ Notebook 02 remains the authoritative "
    "preprocessing source."
)

print(
    "✓ Feature types are inherited from "
    "Notebook 02 metadata."
)

print(
    "✓ Transformed dimensions are inherited "
    "from Notebook 02 metadata."
)

print(
    "✓ Categorical cardinalities are obtained "
    "from fitted preprocessing artifacts."
)

print(
    "✓ Single-category fitted features are "
    "retained without artificial category expansion."
)

print(
    "✓ Raw identifiers and provenance are "
    "excluded from model input."
)

print(
    "✓ Transformed-dimension contracts are validated."
)

print(
    "✓ Generator output contracts are validated."
)

print(
    "✓ Scalar WGAN-style critic initialized."
)

print(
    "✓ Statistical guidance remains external "
    "to the record-level feature tensor."
)

print(
    "✓ Privacy mechanism remains disabled "
    "in Notebook 08."
)

print(
    "✓ Training remains disabled in Notebook 08."
)

print(
    "✓ Synthetic generation remains disabled "
    "in Notebook 08."
)

print(
    "✓ Architecture registries validated."
)

print(
    "✓ SECTION 15 STATUS: PASS"
)

print("=" * 100)

15. INITIALIZE SPP-GAN
✓ Architecture configuration validated.
✓ Latent and hidden dimensions are valid.
✓ Statistical-loss weight is valid.
✓ Numerical activation configuration is valid.
✓ Categorical representation configuration is valid.
✓ Privacy remains disabled.
✓ Training remains disabled.
✓ Notebook 02 root verified.
✓ Notebook 02 metadata root verified.

----------------------------------------------------------------------------------------------------
Initializing SPP-GAN : adult_income
----------------------------------------------------------------------------------------------------
Numerical features   : 6
Categorical features : 8
Generative dimension : 15
Transformed dimension: 105
Latent dimension     : 128
Target               : income
Identifiers excluded : 0
✓ Transformed-dimension contract validated.
✓ Generator output contract validated.
✓ Generator latent dimension validated.
✓ Generator conditional dimension validated.
✓ Critic input dimension validated.
✓ Input

In [45]:
# ==================================================================================================
# 16. PARAMETER COUNT
# ==================================================================================================

print("=" * 100)
print("16. PARAMETER COUNT")
print("=" * 100)


# -----------------------------------------------------------------------------------------------
# 1. Parameter-counting utility
# -----------------------------------------------------------------------------------------------

def count_parameters(model, trainable_only=True):
    """
    Count model parameters.

    Parameters
    ----------
    model : torch.nn.Module
        PyTorch model whose parameters are counted.

    trainable_only : bool
        If True, count only parameters with requires_grad=True.
        If False, count all model parameters.

    Returns
    -------
    int
        Number of parameters.
    """

    if not isinstance(model, nn.Module):
        raise TypeError(
            f"Expected a PyTorch nn.Module, "
            f"received {type(model).__name__}."
        )

    return int(
        sum(
            parameter.numel()
            for parameter in model.parameters()
            if (parameter.requires_grad or not trainable_only)
        )
    )


# -----------------------------------------------------------------------------------------------
# 2. Initialize parameter records
# -----------------------------------------------------------------------------------------------

PARAMETER_RECORDS = []


# -----------------------------------------------------------------------------------------------
# 3. Validate initialized model registry
# -----------------------------------------------------------------------------------------------

if not SPPGAN_MODELS:
    raise RuntimeError(
        "SPPGAN_MODELS is empty. "
        "Section 15 must initialize the SPP-GAN architectures first."
    )

expected_datasets = set(
    FEATURE_SCHEMAS.keys()
)

initialized_datasets = set(
    SPPGAN_MODELS.keys()
)

if initialized_datasets != expected_datasets:
    raise RuntimeError(
        "Initialized model registry does not match FEATURE_SCHEMAS.\n"
        f"Expected : {sorted(expected_datasets)}\n"
        f"Found    : {sorted(initialized_datasets)}"
    )

print(
    f"✓ Initialized model registry validated "
    f"for {len(initialized_datasets)} datasets."
)


# -----------------------------------------------------------------------------------------------
# 4. Count generator and critic parameters
# -----------------------------------------------------------------------------------------------

for dataset_name in SPPGAN_MODELS:

    models = SPPGAN_MODELS[
        dataset_name
    ]

    if "generator" not in models:
        raise KeyError(
            f"Generator is missing for dataset: "
            f"{dataset_name}"
        )

    if "critic" not in models:
        raise KeyError(
            f"Critic is missing for dataset: "
            f"{dataset_name}"
        )

    generator = models["generator"]
    critic = models["critic"]

    # -------------------------------------------------------------------------------------------
    # Validate model types
    # -------------------------------------------------------------------------------------------

    if not isinstance(
        generator,
        nn.Module,
    ):
        raise TypeError(
            f"Generator for {dataset_name} is not "
            f"a PyTorch nn.Module."
        )

    if not isinstance(
        critic,
        nn.Module,
    ):
        raise TypeError(
            f"Critic for {dataset_name} is not "
            f"a PyTorch nn.Module."
        )

    # -------------------------------------------------------------------------------------------
    # Count trainable parameters
    # -------------------------------------------------------------------------------------------

    generator_trainable = count_parameters(
        generator,
        trainable_only=True,
    )

    critic_trainable = count_parameters(
        critic,
        trainable_only=True,
    )

    total_trainable = (
        generator_trainable
        + critic_trainable
    )

    # -------------------------------------------------------------------------------------------
    # Count all parameters
    # -------------------------------------------------------------------------------------------

    generator_total = count_parameters(
        generator,
        trainable_only=False,
    )

    critic_total = count_parameters(
        critic,
        trainable_only=False,
    )

    total_parameters = (
        generator_total
        + critic_total
    )

    # -------------------------------------------------------------------------------------------
    # Validate parameter counts
    # -------------------------------------------------------------------------------------------

    if generator_trainable <= 0:
        raise ValueError(
            f"Generator for {dataset_name} has "
            f"no trainable parameters."
        )

    if critic_trainable <= 0:
        raise ValueError(
            f"Critic for {dataset_name} has "
            f"no trainable parameters."
        )

    if generator_total < generator_trainable:
        raise ValueError(
            f"Generator total parameter count is smaller "
            f"than its trainable parameter count for "
            f"{dataset_name}."
        )

    if critic_total < critic_trainable:
        raise ValueError(
            f"Critic total parameter count is smaller "
            f"than its trainable parameter count for "
            f"{dataset_name}."
        )

    if total_trainable != (
        generator_trainable + critic_trainable
    ):
        raise ValueError(
            f"Trainable parameter arithmetic mismatch "
            f"for {dataset_name}."
        )

    if total_parameters != (
        generator_total + critic_total
    ):
        raise ValueError(
            f"Total parameter arithmetic mismatch "
            f"for {dataset_name}."
        )

    # -------------------------------------------------------------------------------------------
    # Create publication-ready record
    # -------------------------------------------------------------------------------------------

    record = {
        "dataset": dataset_name,
        "generator_trainable_parameters": (
            generator_trainable
        ),
        "critic_trainable_parameters": (
            critic_trainable
        ),
        "total_trainable_parameters": (
            total_trainable
        ),
        "generator_total_parameters": (
            generator_total
        ),
        "critic_total_parameters": (
            critic_total
        ),
        "total_parameters": (
            total_parameters
        ),
    }

    PARAMETER_RECORDS.append(
        record
    )

    # -------------------------------------------------------------------------------------------
    # Dataset-level output
    # -------------------------------------------------------------------------------------------

    print("\n" + "-" * 100)
    print(
        f"Dataset : {dataset_name}"
    )
    print("-" * 100)

    print(
        f"Generator trainable parameters : "
        f"{generator_trainable:,}"
    )

    print(
        f"Critic trainable parameters     : "
        f"{critic_trainable:,}"
    )

    print(
        f"Total trainable parameters     : "
        f"{total_trainable:,}"
    )

    print(
        f"Generator total parameters     : "
        f"{generator_total:,}"
    )

    print(
        f"Critic total parameters         : "
        f"{critic_total:,}"
    )

    print(
        f"Total parameters                : "
        f"{total_parameters:,}"
    )

    print(
        "✓ Generator parameter count validated."
    )

    print(
        "✓ Critic parameter count validated."
    )

    print(
        "✓ Trainable parameter arithmetic validated."
    )

    print(
        "✓ Total parameter arithmetic validated."
    )


# -----------------------------------------------------------------------------------------------
# 5. Create parameter-count DataFrame
# -----------------------------------------------------------------------------------------------

PARAMETER_COUNT_DF = pd.DataFrame(
    PARAMETER_RECORDS
)

if PARAMETER_COUNT_DF.empty:
    raise RuntimeError(
        "Parameter-count DataFrame is empty."
    )

if len(PARAMETER_COUNT_DF) != len(
    expected_datasets
):
    raise RuntimeError(
        "Parameter-count record count does not "
        "match the number of initialized datasets."
    )

if set(
    PARAMETER_COUNT_DF["dataset"]
) != expected_datasets:
    raise RuntimeError(
        "Parameter-count dataset coverage does not "
        "match the initialized dataset registry."
    )


# -----------------------------------------------------------------------------------------------
# 6. Validate parameter-count DataFrame
# -----------------------------------------------------------------------------------------------

required_parameter_columns = [
    "dataset",
    "generator_trainable_parameters",
    "critic_trainable_parameters",
    "total_trainable_parameters",
    "generator_total_parameters",
    "critic_total_parameters",
    "total_parameters",
]

missing_parameter_columns = [
    column
    for column in required_parameter_columns
    if column not in PARAMETER_COUNT_DF.columns
]

if missing_parameter_columns:
    raise RuntimeError(
        "Parameter-count DataFrame is missing required "
        f"columns: {missing_parameter_columns}"
    )

for column in required_parameter_columns[1:]:

    if not pd.api.types.is_integer_dtype(
        PARAMETER_COUNT_DF[column]
    ):
        raise TypeError(
            f"Parameter-count column '{column}' "
            f"must contain integer values."
        )

    if (
        PARAMETER_COUNT_DF[column] <= 0
    ).any():
        raise ValueError(
            f"Parameter-count column '{column}' "
            f"contains non-positive values."
        )


# -----------------------------------------------------------------------------------------------
# 7. Validate persisted parameter-count records
# -----------------------------------------------------------------------------------------------

for _, row in PARAMETER_COUNT_DF.iterrows():

    if row["total_trainable_parameters"] != (
        row["generator_trainable_parameters"]
        + row["critic_trainable_parameters"]
    ):
        raise ValueError(
            f"Trainable parameter total mismatch "
            f"for {row['dataset']}."
        )

    if row["total_parameters"] != (
        row["generator_total_parameters"]
        + row["critic_total_parameters"]
    ):
        raise ValueError(
            f"Total parameter mismatch "
            f"for {row['dataset']}."
        )

    if row["generator_total_parameters"] < (
        row["generator_trainable_parameters"]
    ):
        raise ValueError(
            f"Generator parameter hierarchy invalid "
            f"for {row['dataset']}."
        )

    if row["critic_total_parameters"] < (
        row["critic_trainable_parameters"]
    ):
        raise ValueError(
            f"Critic parameter hierarchy invalid "
            f"for {row['dataset']}."
        )


print("\n" + "-" * 100)
print("PARAMETER COUNT VALIDATION")
print("-" * 100)

print(
    f"✓ Parameter records generated : "
    f"{len(PARAMETER_COUNT_DF)}"
)

print(
    "✓ Dataset coverage validated."
)

print(
    "✓ Trainable parameter counts validated."
)

print(
    "✓ Total parameter counts validated."
)

print(
    "✓ Generator + critic arithmetic validated."
)


# -----------------------------------------------------------------------------------------------
# 8. Persist parameter-count artifact
# -----------------------------------------------------------------------------------------------

parameter_path = (
    DIRS["metadata"]
    / "sppgan_parameter_count.csv"
)

parameter_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

PARAMETER_COUNT_DF.to_csv(
    parameter_path,
    index=False,
)


if not parameter_path.exists():
    raise IOError(
        f"Parameter-count artifact was not created:\n"
        f"{parameter_path}"
    )

if parameter_path.stat().st_size <= 0:
    raise IOError(
        f"Parameter-count artifact is empty:\n"
        f"{parameter_path}"
    )


print(
    f"✓ Saved : {parameter_path}"
)


# -----------------------------------------------------------------------------------------------
# 9. Final Section 16 status
# -----------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("16. PARAMETER COUNT — COMPLETE")
print("=" * 100)

print(
    f"✓ Datasets evaluated : "
    f"{len(PARAMETER_COUNT_DF)}"
)

print(
    "✓ Generator parameters counted."
)

print(
    "✓ Critic parameters counted."
)

print(
    "✓ Trainable parameters explicitly reported."
)

print(
    "✓ Total parameters explicitly reported."
)

print(
    "✓ Parameter-count arithmetic validated."
)

print(
    "✓ Parameter-count artifact persisted."
)

print(
    "✓ SECTION 16 STATUS: PASS"
)

print("=" * 100)

16. PARAMETER COUNT
✓ Initialized model registry validated for 3 datasets.

----------------------------------------------------------------------------------------------------
Dataset : adult_income
----------------------------------------------------------------------------------------------------
Generator trainable parameters : 125,801
Critic trainable parameters     : 93,185
Total trainable parameters     : 218,986
Generator total parameters     : 125,801
Critic total parameters         : 93,185
Total parameters                : 218,986
✓ Generator parameter count validated.
✓ Critic parameter count validated.
✓ Trainable parameter arithmetic validated.
✓ Total parameter arithmetic validated.

----------------------------------------------------------------------------------------------------
Dataset : bank_marketing
----------------------------------------------------------------------------------------------------
Generator trainable parameters : 111,923
Critic trainable paramet

In [47]:
# ==================================================================================================
# 17. ARCHITECTURE SUMMARY
# ==================================================================================================

print("=" * 100)
print("17. ARCHITECTURE SUMMARY")
print("=" * 100)


# -----------------------------------------------------------------------------------------------
# 1. Validate required upstream registries
# -----------------------------------------------------------------------------------------------

if not SPPGAN_ARCHITECTURES:
    raise RuntimeError(
        "SPPGAN_ARCHITECTURES is empty. "
        "Section 15 must be completed before Section 17."
    )

if not SPPGAN_MODELS:
    raise RuntimeError(
        "SPPGAN_MODELS is empty. "
        "Section 15 must be completed before Section 17."
    )

if "PARAMETER_COUNT_DF" not in globals():
    raise RuntimeError(
        "PARAMETER_COUNT_DF is not available. "
        "Section 16 must be completed before Section 17."
    )


# -----------------------------------------------------------------------------------------------
# 2. Define expected dataset registry
# -----------------------------------------------------------------------------------------------

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

expected_dataset_set = set(
    EXPECTED_DATASETS
)

architecture_dataset_set = set(
    SPPGAN_ARCHITECTURES.keys()
)

model_dataset_set = set(
    SPPGAN_MODELS.keys()
)

parameter_dataset_set = set(
    PARAMETER_COUNT_DF["dataset"].tolist()
)


if architecture_dataset_set != expected_dataset_set:
    raise ValueError(
        "SPP-GAN architecture registry does not match "
        "the expected dataset registry.\n"
        f"Expected: {EXPECTED_DATASETS}\n"
        f"Found   : {sorted(architecture_dataset_set)}"
    )

if model_dataset_set != expected_dataset_set:
    raise ValueError(
        "SPP-GAN model registry does not match "
        "the expected dataset registry."
    )

if parameter_dataset_set != expected_dataset_set:
    raise ValueError(
        "Section 16 parameter-count dataset registry does not "
        "match the expected dataset registry."
    )

print(
    f"✓ Dataset registry validated : "
    f"{len(EXPECTED_DATASETS)} datasets."
)


# -----------------------------------------------------------------------------------------------
# 3. Validate Section 16 parameter-count schema
# -----------------------------------------------------------------------------------------------

required_parameter_columns = [
    "dataset",
    "generator_trainable_parameters",
    "critic_trainable_parameters",
    "total_trainable_parameters",
    "generator_total_parameters",
    "critic_total_parameters",
    "total_parameters",
]

missing_parameter_columns = [
    column
    for column in required_parameter_columns
    if column not in PARAMETER_COUNT_DF.columns
]

if missing_parameter_columns:
    raise KeyError(
        "Section 16 parameter-count artifact is missing "
        f"required columns: {missing_parameter_columns}"
    )

if PARAMETER_COUNT_DF["dataset"].duplicated().any():
    raise ValueError(
        "Section 16 parameter-count artifact contains "
        "duplicate dataset records."
    )

print(
    "✓ Section 16 parameter-count schema validated."
)


# -----------------------------------------------------------------------------------------------
# 4. Initialize final architecture summary registry
# -----------------------------------------------------------------------------------------------

ARCHITECTURE_SUMMARY = []


# -----------------------------------------------------------------------------------------------
# 5. Build and validate dataset-level architecture manifest
# -----------------------------------------------------------------------------------------------

for dataset_name in EXPECTED_DATASETS:

    if dataset_name not in SPPGAN_ARCHITECTURES:
        raise KeyError(
            f"Architecture missing for dataset: "
            f"{dataset_name}"
        )

    if dataset_name not in SPPGAN_MODELS:
        raise KeyError(
            f"Model bundle missing for dataset: "
            f"{dataset_name}"
        )

    architecture = SPPGAN_ARCHITECTURES[
        dataset_name
    ]

    models = SPPGAN_MODELS[
        dataset_name
    ]

    # -------------------------------------------------------------------------------------------
    # Retrieve Section 16 parameter record
    # -------------------------------------------------------------------------------------------

    parameter_rows = PARAMETER_COUNT_DF[
        PARAMETER_COUNT_DF["dataset"] == dataset_name
    ]

    if len(parameter_rows) != 1:
        raise ValueError(
            f"Expected exactly one Section 16 parameter record "
            f"for {dataset_name}, found {len(parameter_rows)}."
        )

    parameter_record = parameter_rows.iloc[0]

    # -------------------------------------------------------------------------------------------
    # Authoritative dimensions from Section 15
    # -------------------------------------------------------------------------------------------

    generative_dimension = int(
        architecture["generative_dimension"]
    )

    transformed_dimension = int(
        architecture["transformed_dim"]
    )

    expected_transformed_dimension = int(
        architecture.get(
            "expected_transformed_dim",
            transformed_dimension,
        )
    )

    num_numerical = int(
        architecture["num_numerical"]
    )

    num_categorical = int(
        architecture["num_categorical"]
    )

    latent_dim = int(
        architecture["latent_dim"]
    )

    conditional_dim = int(
        architecture["conditional_dim"]
    )

    generator_hidden_dims = list(
        architecture["generator_hidden_dims"]
    )

    critic_hidden_dims = list(
        architecture["critic_hidden_dims"]
    )

    categorical_cardinalities = list(
        architecture[
            "categorical_cardinalities"
        ]
    )

    # -------------------------------------------------------------------------------------------
    # Validate dimensions
    # -------------------------------------------------------------------------------------------

    if generative_dimension <= 0:
        raise ValueError(
            f"Invalid generative dimension for "
            f"{dataset_name}: {generative_dimension}"
        )

    if transformed_dimension <= 0:
        raise ValueError(
            f"Invalid transformed dimension for "
            f"{dataset_name}: {transformed_dimension}"
        )

    if expected_transformed_dimension != transformed_dimension:
        raise ValueError(
            f"Section 15 transformed-dimension contract failed "
            f"for {dataset_name}."
        )

    if num_numerical < 0:
        raise ValueError(
            f"Invalid numerical feature count for "
            f"{dataset_name}."
        )

    if num_categorical < 0:
        raise ValueError(
            f"Invalid categorical feature count for "
            f"{dataset_name}."
        )

    if (
        num_numerical
        + num_categorical
        != generative_dimension - 1
    ):
        raise ValueError(
            f"Feature-count inconsistency for "
            f"{dataset_name}: "
            f"numerical + categorical must equal "
            f"generative dimension - 1."
        )

    if latent_dim <= 0:
        raise ValueError(
            f"Invalid latent dimension for "
            f"{dataset_name}."
        )

    if conditional_dim < 0:
        raise ValueError(
            f"Invalid conditional dimension for "
            f"{dataset_name}."
        )

    # -------------------------------------------------------------------------------------------
    # Validate hidden-layer configuration
    # -------------------------------------------------------------------------------------------

    if len(generator_hidden_dims) != 2:
        raise ValueError(
            f"Generator hidden-layer configuration must contain "
            f"two dimensions for {dataset_name}."
        )

    if len(critic_hidden_dims) != 2:
        raise ValueError(
            f"Critic hidden-layer configuration must contain "
            f"two dimensions for {dataset_name}."
        )

    if any(
        int(value) <= 0
        for value in generator_hidden_dims
    ):
        raise ValueError(
            f"Invalid generator hidden dimensions for "
            f"{dataset_name}."
        )

    if any(
        int(value) <= 0
        for value in critic_hidden_dims
    ):
        raise ValueError(
            f"Invalid critic hidden dimensions for "
            f"{dataset_name}."
        )

    # -------------------------------------------------------------------------------------------
    # Validate categorical cardinalities
    # -------------------------------------------------------------------------------------------

    if len(categorical_cardinalities) != num_categorical:
        raise ValueError(
            f"Categorical cardinality count mismatch for "
            f"{dataset_name}: "
            f"expected={num_categorical}, "
            f"found={len(categorical_cardinalities)}."
        )

    if any(
        not isinstance(cardinality, int)
        or cardinality < 1
        for cardinality in categorical_cardinalities
    ):
        raise ValueError(
            f"Invalid categorical cardinality for "
            f"{dataset_name}."
        )

    cardinality_transformed_dimension = (
        num_numerical
        + sum(categorical_cardinalities)
    )

    if (
        cardinality_transformed_dimension
        != transformed_dimension
    ):
        raise ValueError(
            f"Categorical-cardinality transformed dimension "
            f"does not match Notebook 02 transformed dimension "
            f"for {dataset_name}: "
            f"expected={cardinality_transformed_dimension}, "
            f"declared={transformed_dimension}."
        )

    # -------------------------------------------------------------------------------------------
    # Validate Section 15 model architecture
    # -------------------------------------------------------------------------------------------

    generator = models["generator"]
    critic = models["critic"]

    if generator.latent_dim != latent_dim:
        raise ValueError(
            f"Generator latent dimension mismatch for "
            f"{dataset_name}: "
            f"Section 15={latent_dim}, "
            f"model={generator.latent_dim}."
        )

    if generator.conditional_dim != conditional_dim:
        raise ValueError(
            f"Generator conditional dimension mismatch for "
            f"{dataset_name}."
        )

    if generator.num_numerical != num_numerical:
        raise ValueError(
            f"Generator numerical dimension mismatch for "
            f"{dataset_name}."
        )

    if list(
        generator.categorical_cardinalities
    ) != categorical_cardinalities:
        raise ValueError(
            f"Generator categorical cardinality mismatch "
            f"for {dataset_name}."
        )

    if critic.input_dim != transformed_dimension:
        raise ValueError(
            f"Critic input dimension mismatch for "
            f"{dataset_name}: "
            f"Section 15={transformed_dimension}, "
            f"model={critic.input_dim}."
        )

    # -------------------------------------------------------------------------------------------
    # Retrieve Section 16 parameter counts
    # -------------------------------------------------------------------------------------------

    generator_trainable_parameters = int(
        parameter_record[
            "generator_trainable_parameters"
        ]
    )

    critic_trainable_parameters = int(
        parameter_record[
            "critic_trainable_parameters"
        ]
    )

    total_trainable_parameters = int(
        parameter_record[
            "total_trainable_parameters"
        ]
    )

    generator_total_parameters = int(
        parameter_record[
            "generator_total_parameters"
        ]
    )

    critic_total_parameters = int(
        parameter_record[
            "critic_total_parameters"
        ]
    )

    total_parameters = int(
        parameter_record[
            "total_parameters"
        ]
    )

    # -------------------------------------------------------------------------------------------
    # Independently recompute parameter counts for cross-validation
    # -------------------------------------------------------------------------------------------

    computed_generator_trainable = count_parameters(
        generator,
        trainable_only=True,
    )

    computed_critic_trainable = count_parameters(
        critic,
        trainable_only=True,
    )

    computed_generator_total = count_parameters(
        generator,
        trainable_only=False,
    )

    computed_critic_total = count_parameters(
        critic,
        trainable_only=False,
    )

    if generator_trainable_parameters != (
        computed_generator_trainable
    ):
        raise ValueError(
            f"Generator trainable parameter mismatch between "
            f"Sections 16 and 17 for {dataset_name}."
        )

    if critic_trainable_parameters != (
        computed_critic_trainable
    ):
        raise ValueError(
            f"Critic trainable parameter mismatch between "
            f"Sections 16 and 17 for {dataset_name}."
        )

    if generator_total_parameters != (
        computed_generator_total
    ):
        raise ValueError(
            f"Generator total parameter mismatch between "
            f"Sections 16 and 17 for {dataset_name}."
        )

    if critic_total_parameters != (
        computed_critic_total
    ):
        raise ValueError(
            f"Critic total parameter mismatch between "
            f"Sections 16 and 17 for {dataset_name}."
        )

    if total_trainable_parameters != (
        generator_trainable_parameters
        + critic_trainable_parameters
    ):
        raise ValueError(
            f"Total trainable parameter arithmetic mismatch "
            f"for {dataset_name}."
        )

    if total_parameters != (
        generator_total_parameters
        + critic_total_parameters
    ):
        raise ValueError(
            f"Total parameter arithmetic mismatch "
            f"for {dataset_name}."
        )

    # -------------------------------------------------------------------------------------------
    # Validate representation settings
    # -------------------------------------------------------------------------------------------

    representation = architecture[
        "representation"
    ]

    numerical_activation = representation[
        "numerical_activation"
    ]

    categorical_training_activation = (
        representation[
            "categorical_training_activation"
        ]
    )

    categorical_probability_mapping = (
        representation[
            "categorical_probability_mapping"
        ]
    )

    transformation_source = representation[
        "transformation_source"
    ]

    hard_decoding = representation.get(
        "hard_decoding",
        "Notebook_13",
    )

    if numerical_activation != "identity":
        raise ValueError(
            f"Numerical activation mismatch for "
            f"{dataset_name}: "
            f"expected='identity', "
            f"found={numerical_activation!r}."
        )

    if categorical_training_activation != (
        "gumbel_softmax"
    ):
        raise ValueError(
            f"Categorical training activation mismatch "
            f"for {dataset_name}."
        )

    if categorical_probability_mapping != "softmax":
        raise ValueError(
            f"Categorical probability mapping mismatch "
            f"for {dataset_name}."
        )

    if transformation_source != "Notebook_02":
        raise ValueError(
            f"Transformation source mismatch for "
            f"{dataset_name}: "
            f"{transformation_source!r}."
        )

    if hard_decoding != "Notebook_13":
        raise ValueError(
            f"Hard decoding source mismatch for "
            f"{dataset_name}: "
            f"{hard_decoding!r}."
        )

    # -------------------------------------------------------------------------------------------
    # Validate methodological delegation
    # -------------------------------------------------------------------------------------------

    if architecture["privacy"]["enabled"]:
        raise ValueError(
            f"Privacy must remain disabled in Notebook 08 "
            f"for {dataset_name}."
        )

    if architecture["training"]["enabled"]:
        raise ValueError(
            f"Training must remain disabled in Notebook 08 "
            f"for {dataset_name}."
        )

    if architecture["privacy"][
        "implementation_notebook"
    ] != 10:
        raise ValueError(
            f"DP mechanism must be delegated to Notebook 10 "
            f"for {dataset_name}."
        )

    if architecture["privacy"][
        "accounting_notebook"
    ] != 11:
        raise ValueError(
            f"Privacy accounting must be delegated to "
            f"Notebook 11 for {dataset_name}."
        )

    if architecture["training"][
        "implementation_notebook"
    ] != 12:
        raise ValueError(
            f"Training must be delegated to Notebook 12 "
            f"for {dataset_name}."
        )

    # -------------------------------------------------------------------------------------------
    # Build final architecture manifest record
    # -------------------------------------------------------------------------------------------

    summary = {
        "dataset": dataset_name,

        "target": architecture[
            "target"
        ],

        "generative_dimension": (
            generative_dimension
        ),

        "transformed_dimension": (
            transformed_dimension
        ),

        "numerical_features": (
            num_numerical
        ),

        "categorical_features": (
            num_categorical
        ),

        "categorical_cardinality_sum": (
            sum(categorical_cardinalities)
        ),

        "latent_dim": (
            latent_dim
        ),

        "conditional_dim": (
            conditional_dim
        ),

        "generator_hidden_1": (
            int(generator_hidden_dims[0])
        ),

        "generator_hidden_2": (
            int(generator_hidden_dims[1])
        ),

        "critic_hidden_1": (
            int(critic_hidden_dims[0])
        ),

        "critic_hidden_2": (
            int(critic_hidden_dims[1])
        ),

        "generator_trainable_parameters": (
            generator_trainable_parameters
        ),

        "critic_trainable_parameters": (
            critic_trainable_parameters
        ),

        "total_trainable_parameters": (
            total_trainable_parameters
        ),

        "generator_total_parameters": (
            generator_total_parameters
        ),

        "critic_total_parameters": (
            critic_total_parameters
        ),

        "total_parameters": (
            total_parameters
        ),

        # ---------------------------------------------------------------------------------------
        # Representation
        # ---------------------------------------------------------------------------------------

        "numerical_activation": (
            numerical_activation
        ),

        "categorical_training_activation": (
            categorical_training_activation
        ),

        "categorical_probability_mapping": (
            categorical_probability_mapping
        ),

        "hard_decoding": (
            hard_decoding
        ),

        "transformation_source": (
            transformation_source
        ),

        "critic_output": "scalar",

        # ---------------------------------------------------------------------------------------
        # SPP-GAN methodological components
        # ---------------------------------------------------------------------------------------

        "statistical_guidance": (
            "Notebook_09"
        ),

        "differential_privacy": (
            "Notebook_10"
        ),

        "privacy_accounting": (
            "Notebook_11"
        ),

        "training": (
            "Notebook_12"
        ),

        "synthetic_generation": (
            "Notebook_13"
        ),
    }

    ARCHITECTURE_SUMMARY.append(
        summary
    )

    print(
        f"✓ Architecture manifest validated : "
        f"{dataset_name}"
    )

    print(
        f"  Generative dimension   : "
        f"{generative_dimension}"
    )

    print(
        f"  Transformed dimension  : "
        f"{transformed_dimension}"
    )

    print(
        f"  Trainable parameters   : "
        f"{total_trainable_parameters:,}"
    )


# -----------------------------------------------------------------------------------------------
# 6. Convert final manifest to DataFrame
# -----------------------------------------------------------------------------------------------

ARCHITECTURE_SUMMARY_DF = pd.DataFrame(
    ARCHITECTURE_SUMMARY
)


# -----------------------------------------------------------------------------------------------
# 7. Validate final manifest structure
# -----------------------------------------------------------------------------------------------

if ARCHITECTURE_SUMMARY_DF.empty:
    raise RuntimeError(
        "Architecture summary is empty."
    )

if len(
    ARCHITECTURE_SUMMARY_DF
) != len(EXPECTED_DATASETS):
    raise RuntimeError(
        "Architecture summary record count does not "
        "match the expected dataset count."
    )

actual_datasets = (
    ARCHITECTURE_SUMMARY_DF[
        "dataset"
    ].tolist()
)

if actual_datasets != EXPECTED_DATASETS:
    raise ValueError(
        "Architecture summary dataset order/content mismatch.\n"
        f"Expected: {EXPECTED_DATASETS}\n"
        f"Found   : {actual_datasets}"
    )

if ARCHITECTURE_SUMMARY_DF[
    "dataset"
].duplicated().any():
    raise ValueError(
        "Architecture summary contains duplicate datasets."
    )


# -----------------------------------------------------------------------------------------------
# 8. Cross-validate Section 17 against Section 15
# -----------------------------------------------------------------------------------------------

for _, row in ARCHITECTURE_SUMMARY_DF.iterrows():

    dataset_name = row[
        "dataset"
    ]

    architecture = SPPGAN_ARCHITECTURES[
        dataset_name
    ]

    if int(row["generative_dimension"]) != int(
        architecture["generative_dimension"]
    ):
        raise ValueError(
            f"Generative dimension mismatch for "
            f"{dataset_name}."
        )

    if int(row["transformed_dimension"]) != int(
        architecture["transformed_dim"]
    ):
        raise ValueError(
            f"Transformed dimension mismatch for "
            f"{dataset_name}."
        )

    if int(row["numerical_features"]) != int(
        architecture["num_numerical"]
    ):
        raise ValueError(
            f"Numerical feature count mismatch for "
            f"{dataset_name}."
        )

    if int(row["categorical_features"]) != int(
        architecture["num_categorical"]
    ):
        raise ValueError(
            f"Categorical feature count mismatch for "
            f"{dataset_name}."
        )

    if int(row["latent_dim"]) != int(
        architecture["latent_dim"]
    ):
        raise ValueError(
            f"Latent dimension mismatch for "
            f"{dataset_name}."
        )

    if int(row["conditional_dim"]) != int(
        architecture["conditional_dim"]
    ):
        raise ValueError(
            f"Conditional dimension mismatch for "
            f"{dataset_name}."
        )

print(
    "✓ Section 17 dimensions cross-validated against Section 15."
)


# -----------------------------------------------------------------------------------------------
# 9. Cross-validate Section 17 against Section 16
# -----------------------------------------------------------------------------------------------

parameter_comparison = (
    ARCHITECTURE_SUMMARY_DF[
        [
            "dataset",
            "generator_trainable_parameters",
            "critic_trainable_parameters",
            "total_trainable_parameters",
            "generator_total_parameters",
            "critic_total_parameters",
            "total_parameters",
        ]
    ]
    .merge(
        PARAMETER_COUNT_DF[
            [
                "dataset",
                "generator_trainable_parameters",
                "critic_trainable_parameters",
                "total_trainable_parameters",
                "generator_total_parameters",
                "critic_total_parameters",
                "total_parameters",
            ]
        ],
        on="dataset",
        how="inner",
        suffixes=(
            "_section17",
            "_section16",
        ),
        validate="one_to_one",
    )
)

if len(parameter_comparison) != len(
    EXPECTED_DATASETS
):
    raise RuntimeError(
        "Section 16 and Section 17 parameter records "
        "could not be matched one-to-one."
    )

parameter_columns = [
    "generator_trainable_parameters",
    "critic_trainable_parameters",
    "total_trainable_parameters",
    "generator_total_parameters",
    "critic_total_parameters",
    "total_parameters",
]

for column in parameter_columns:

    if not (
        parameter_comparison[
            f"{column}_section17"
        ]
        ==
        parameter_comparison[
            f"{column}_section16"
        ]
    ).all():
        raise ValueError(
            f"Parameter-count mismatch between Sections "
            f"16 and 17 for column '{column}'."
        )

print(
    "✓ Section 17 parameter counts cross-validated "
    "against Section 16."
)


# -----------------------------------------------------------------------------------------------
# 10. Validate authoritative representation settings
# -----------------------------------------------------------------------------------------------

if not all(
    ARCHITECTURE_SUMMARY_DF[
        "numerical_activation"
    ] == "identity"
):
    raise ValueError(
        "Numerical activation mismatch."
    )

if not all(
    ARCHITECTURE_SUMMARY_DF[
        "categorical_training_activation"
    ] == "gumbel_softmax"
):
    raise ValueError(
        "Categorical training activation mismatch."
    )

if not all(
    ARCHITECTURE_SUMMARY_DF[
        "categorical_probability_mapping"
    ] == "softmax"
):
    raise ValueError(
        "Categorical probability mapping mismatch."
    )

if not all(
    ARCHITECTURE_SUMMARY_DF[
        "critic_output"
    ] == "scalar"
):
    raise ValueError(
        "Critic output mismatch."
    )

if not all(
    ARCHITECTURE_SUMMARY_DF[
        "transformation_source"
    ] == "Notebook_02"
):
    raise ValueError(
        "Transformation source mismatch."
    )

if not all(
    ARCHITECTURE_SUMMARY_DF[
        "hard_decoding"
    ] == "Notebook_13"
):
    raise ValueError(
        "Hard-decoding source mismatch."
    )

print(
    "✓ Representation configuration validated."
)

print(
    "✓ Numerical activation recorded as identity."
)

print(
    "✓ Gumbel-Softmax recorded as categorical "
    "training mechanism."
)

print(
    "✓ Softmax recorded as categorical "
    "probability mapping."
)

print(
    "✓ Scalar WGAN-style critic recorded."
)


# -----------------------------------------------------------------------------------------------
# 11. Validate methodological delegation
# -----------------------------------------------------------------------------------------------

if not all(
    ARCHITECTURE_SUMMARY_DF[
        "statistical_guidance"
    ] == "Notebook_09"
):
    raise ValueError(
        "Statistical-guidance delegation mismatch."
    )

if not all(
    ARCHITECTURE_SUMMARY_DF[
        "differential_privacy"
    ] == "Notebook_10"
):
    raise ValueError(
        "Differential-privacy delegation mismatch."
    )

if not all(
    ARCHITECTURE_SUMMARY_DF[
        "privacy_accounting"
    ] == "Notebook_11"
):
    raise ValueError(
        "Privacy-accounting delegation mismatch."
    )

if not all(
    ARCHITECTURE_SUMMARY_DF[
        "training"
    ] == "Notebook_12"
):
    raise ValueError(
        "Training delegation mismatch."
    )

if not all(
    ARCHITECTURE_SUMMARY_DF[
        "synthetic_generation"
    ] == "Notebook_13"
):
    raise ValueError(
        "Synthetic-generation delegation mismatch."
    )

print(
    "✓ Statistical guidance delegated to Notebook 09."
)

print(
    "✓ Differential privacy delegated to Notebook 10."
)

print(
    "✓ Privacy accounting delegated to Notebook 11."
)

print(
    "✓ Training delegated to Notebook 12."
)

print(
    "✓ Synthetic generation delegated to Notebook 13."
)


# -----------------------------------------------------------------------------------------------
# 12. Save final architecture manifest
# -----------------------------------------------------------------------------------------------

summary_path = (
    DIRS["architecture"]
    / "sppgan_architecture_summary.csv"
)

summary_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

ARCHITECTURE_SUMMARY_DF.to_csv(
    summary_path,
    index=False,
)


if not summary_path.exists():
    raise IOError(
        f"Architecture summary was not created:\n"
        f"{summary_path}"
    )

if summary_path.stat().st_size <= 0:
    raise IOError(
        f"Architecture summary artifact is empty:\n"
        f"{summary_path}"
    )


# -----------------------------------------------------------------------------------------------
# 13. Display final architecture manifest
# -----------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("FINAL SPP-GAN ARCHITECTURE MANIFEST")
print("-" * 100)

print(
    ARCHITECTURE_SUMMARY_DF.to_string(
        index=False
    )
)

print(
    f"\n✓ Saved : {summary_path}"
)


# -----------------------------------------------------------------------------------------------
# 14. Final Section 17 status
# -----------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("17. ARCHITECTURE SUMMARY — COMPLETE")
print("=" * 100)

print(
    f"✓ Datasets summarized : "
    f"{len(ARCHITECTURE_SUMMARY_DF)}"
)

print(
    "✓ Section 15 architecture registry cross-validated."
)

print(
    "✓ Section 16 parameter counts cross-validated."
)

print(
    "✓ Generative and transformed dimensions validated."
)

print(
    "✓ Feature-type counts validated."
)

print(
    "✓ Categorical cardinalities validated."
)

print(
    "✓ Latent and conditional dimensions validated."
)

print(
    "✓ Generator and critic hidden dimensions validated."
)

print(
    "✓ Generator and critic parameter counts validated."
)

print(
    "✓ Trainable and total parameter counts validated."
)

print(
    "✓ Numerical activation recorded as identity."
)

print(
    "✓ Gumbel-Softmax recorded as categorical "
    "training mechanism."
)

print(
    "✓ Softmax recorded as categorical "
    "probability mapping."
)

print(
    "✓ Scalar WGAN-style critic recorded."
)

print(
    "✓ Statistical guidance delegated to Notebook 09."
)

print(
    "✓ Differential privacy delegated to Notebook 10."
)

print(
    "✓ Privacy accounting delegated to Notebook 11."
)

print(
    "✓ Training delegated to Notebook 12."
)

print(
    "✓ Synthetic generation delegated to Notebook 13."
)

print(
    "✓ Final architecture manifest persisted."
)

print(
    "✓ SECTION 17 STATUS: PASS"
)

print("=" * 100)

17. ARCHITECTURE SUMMARY
✓ Dataset registry validated : 3 datasets.
✓ Section 16 parameter-count schema validated.
✓ Architecture manifest validated : adult_income
  Generative dimension   : 15
  Transformed dimension  : 105
  Trainable parameters   : 218,986
✓ Architecture manifest validated : bank_marketing
  Generative dimension   : 17
  Transformed dimension  : 51
  Trainable parameters   : 191,284
✓ Architecture manifest validated : diabetes_130us
  Generative dimension   : 48
  Transformed dimension  : 2329
  Trainable parameters   : 1,359,898
✓ Section 17 dimensions cross-validated against Section 15.
✓ Section 17 parameter counts cross-validated against Section 16.
✓ Representation configuration validated.
✓ Numerical activation recorded as identity.
✓ Gumbel-Softmax recorded as categorical training mechanism.
✓ Softmax recorded as categorical probability mapping.
✓ Scalar WGAN-style critic recorded.
✓ Statistical guidance delegated to Notebook 09.
✓ Differential privacy delega

In [50]:
# ==================================================================================================
# 18. TENSOR SHAPE & OUTPUT CONTRACT TESTS
# ==================================================================================================

print("=" * 100)
print("18. TENSOR SHAPE & OUTPUT CONTRACT TESTS")
print("=" * 100)

# -----------------------------------------------------------------------------------------------
# 1. Validation containers and test configuration
# -----------------------------------------------------------------------------------------------

SHAPE_TEST_RESULTS = []

TEST_BATCH_SIZE = 4

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

assert TEST_BATCH_SIZE > 0, (
    "TEST_BATCH_SIZE must be positive."
)

assert set(SPPGAN_ARCHITECTURES.keys()) == set(EXPECTED_DATASETS), (
    "SPPGAN_ARCHITECTURES dataset coverage mismatch."
)

assert set(SPPGAN_MODELS.keys()) == set(EXPECTED_DATASETS), (
    "SPPGAN_MODELS dataset coverage mismatch."
)

# -----------------------------------------------------------------------------------------------
# 2. Determine validation device
# -----------------------------------------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"\nValidation device : {DEVICE}")
print(f"Test batch size   : {TEST_BATCH_SIZE}")

# -----------------------------------------------------------------------------------------------
# 3. Tensor validation helper
# -----------------------------------------------------------------------------------------------

def validate_tensor(
    tensor,
    expected_shape,
    tensor_name,
):
    """
    Validate tensor type, shape, and finite values.
    """

    assert isinstance(
        tensor,
        torch.Tensor,
    ), (
        f"{tensor_name} must be a torch.Tensor."
    )

    actual_shape = tuple(
        tensor.shape
    )

    assert actual_shape == tuple(
        expected_shape
    ), (
        f"{tensor_name} shape mismatch: "
        f"expected {tuple(expected_shape)}, "
        f"got {actual_shape}."
    )

    assert torch.isfinite(
        tensor
    ).all().item(), (
        f"{tensor_name} contains non-finite values."
    )

    return True


# -----------------------------------------------------------------------------------------------
# 4. Categorical probability validation helper
# -----------------------------------------------------------------------------------------------

def validate_probability_tensor(
    tensor,
    expected_shape,
    tensor_name,
):
    """
    Validate categorical probability output.
    """

    validate_tensor(
        tensor=tensor,
        expected_shape=expected_shape,
        tensor_name=tensor_name,
    )

    assert torch.all(
        tensor >= 0
    ).item(), (
        f"{tensor_name} contains negative probabilities."
    )

    row_sums = tensor.sum(
        dim=1
    )

    assert torch.allclose(
        row_sums,
        torch.ones_like(row_sums),
        atol=1e-5,
        rtol=1e-5,
    ), (
        f"{tensor_name} rows do not sum to 1."
    )

    return True


# -----------------------------------------------------------------------------------------------
# 5. Run tests for every dataset
# -----------------------------------------------------------------------------------------------

for dataset_name in EXPECTED_DATASETS:

    print("\n" + "-" * 100)
    print(f"Dataset : {dataset_name}")
    print("-" * 100)

    architecture = SPPGAN_ARCHITECTURES[
        dataset_name
    ]

    model_entry = SPPGAN_MODELS[
        dataset_name
    ]

    generator = model_entry[
        "generator"
    ]

    # -------------------------------------------------------------------------------------------
    # 5.1 Generator validation
    # -------------------------------------------------------------------------------------------

    assert isinstance(
        generator,
        torch.nn.Module,
    ), (
        f"Generator for {dataset_name} "
        "is not a torch.nn.Module."
    )

    # -------------------------------------------------------------------------------------------
    # 5.2 Read architecture contract
    # -------------------------------------------------------------------------------------------

    latent_dim = int(
        architecture["latent_dim"]
    )

    conditional_dim = int(
        architecture.get(
            "conditional_dim",
            0,
        )
    )

    num_numerical = int(
        architecture["num_numerical"]
    )

    categorical_cardinalities = [
        int(cardinality)
        for cardinality in architecture[
            "categorical_cardinalities"
        ]
    ]

    transformed_dim = int(
        architecture["transformed_dim"]
    )

    # -------------------------------------------------------------------------------------------
    # 5.3 Validate dimensional contract
    # -------------------------------------------------------------------------------------------

    expected_categorical_dim = sum(
        categorical_cardinalities
    )

    expected_transformed_dim = (
        num_numerical
        + expected_categorical_dim
    )

    assert transformed_dim == expected_transformed_dim, (
        f"{dataset_name}: transformed dimension mismatch. "
        f"Declared={transformed_dim}, "
        f"Calculated={expected_transformed_dim}."
    )

    assert latent_dim > 0, (
        f"{dataset_name}: latent_dim must be positive."
    )

    assert conditional_dim >= 0, (
        f"{dataset_name}: conditional_dim cannot be negative."
    )

    # -------------------------------------------------------------------------------------------
    # 5.4 Preserve original generator state
    # -------------------------------------------------------------------------------------------

    original_training_mode = generator.training

    original_devices = {
        parameter.device
        for parameter in generator.parameters()
    }

    assert len(original_devices) == 1, (
        f"{dataset_name}: generator parameters "
        "are on multiple devices."
    )

    original_device = next(
        iter(original_devices)
    )

    # -------------------------------------------------------------------------------------------
    # 5.5 Move to validation device
    # -------------------------------------------------------------------------------------------

    generator = generator.to(
        DEVICE
    )

    generator.eval()

    # -------------------------------------------------------------------------------------------
    # 5.6 Create latent input
    # -------------------------------------------------------------------------------------------

    z = torch.randn(
        TEST_BATCH_SIZE,
        latent_dim,
        device=DEVICE,
    )

    validate_tensor(
        tensor=z,
        expected_shape=(
            TEST_BATCH_SIZE,
            latent_dim,
        ),
        tensor_name=f"{dataset_name}.latent_input",
    )

    # -------------------------------------------------------------------------------------------
    # 5.7 Conditional-input contract
    # -------------------------------------------------------------------------------------------

    if conditional_dim > 0:

        condition = torch.zeros(
            TEST_BATCH_SIZE,
            conditional_dim,
            device=DEVICE,
        )

        generator_inputs = (
            z,
            condition,
        )

    else:

        condition = None

        generator_inputs = (
            z,
        )

    # -------------------------------------------------------------------------------------------
    # 5.8 Generator forward pass
    # -------------------------------------------------------------------------------------------

    with torch.no_grad():

        outputs = generator(
            *generator_inputs
        )

    assert isinstance(
        outputs,
        dict,
    ), (
        f"{dataset_name}: generator output "
        "must be a dictionary."
    )

    assert "numerical" in outputs, (
        f"{dataset_name}: generator output "
        "missing 'numerical'."
    )

    assert "categorical_logits" in outputs, (
        f"{dataset_name}: generator output "
        "missing 'categorical_logits'."
    )

    numerical_output = outputs[
        "numerical"
    ]

    categorical_logits = outputs[
        "categorical_logits"
    ]

    # -------------------------------------------------------------------------------------------
    # 5.9 Numerical output validation
    # -------------------------------------------------------------------------------------------

    if num_numerical > 0:

        validate_tensor(
            tensor=numerical_output,
            expected_shape=(
                TEST_BATCH_SIZE,
                num_numerical,
            ),
            tensor_name=(
                f"{dataset_name}.numerical_output"
            ),
        )

    else:

        assert isinstance(
            numerical_output,
            torch.Tensor,
        ), (
            f"{dataset_name}: numerical output "
            "must be a tensor."
        )

        assert tuple(
            numerical_output.shape
        ) == (
            TEST_BATCH_SIZE,
            0,
        ), (
            f"{dataset_name}: expected numerical "
            f"output shape {(TEST_BATCH_SIZE, 0)}, "
            f"got {tuple(numerical_output.shape)}."
        )

    # -------------------------------------------------------------------------------------------
    # 5.10 Categorical head count validation
    # -------------------------------------------------------------------------------------------

    assert isinstance(
        categorical_logits,
        (list, tuple),
    ), (
        f"{dataset_name}: categorical_logits "
        "must be a list or tuple."
    )

    assert len(
        categorical_logits
    ) == len(
        categorical_cardinalities
    ), (
        f"{dataset_name}: categorical head count mismatch. "
        f"Expected {len(categorical_cardinalities)}, "
        f"got {len(categorical_logits)}."
    )

    # -------------------------------------------------------------------------------------------
    # 5.11 Validate every categorical head
    # -------------------------------------------------------------------------------------------

    categorical_probability_outputs = []

    for feature_index, (
        logits,
        cardinality,
    ) in enumerate(
        zip(
            categorical_logits,
            categorical_cardinalities,
        )
    ):

        expected_shape = (
            TEST_BATCH_SIZE,
            cardinality,
        )

        validate_tensor(
            tensor=logits,
            expected_shape=expected_shape,
            tensor_name=(
                f"{dataset_name}."
                f"categorical_logits[{feature_index}]"
            ),
        )

        # ---------------------------------------------------------------------------------------
        # Gumbel-Softmax output is converted to probabilities for explicit simplex validation.
        # ---------------------------------------------------------------------------------------

        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        validate_probability_tensor(
            tensor=probabilities,
            expected_shape=expected_shape,
            tensor_name=(
                f"{dataset_name}."
                f"categorical_probabilities[{feature_index}]"
            ),
        )

        categorical_probability_outputs.append(
            probabilities
        )

    # -------------------------------------------------------------------------------------------
    # 5.12 Validate transformed-dimension assembly
    # -------------------------------------------------------------------------------------------

    numerical_dim_actual = int(
        numerical_output.shape[1]
    )

    categorical_dim_actual = sum(
        int(logits.shape[1])
        for logits in categorical_logits
    )

    assembled_transformed_dim = (
        numerical_dim_actual
        + categorical_dim_actual
    )

    assert (
        assembled_transformed_dim
        == transformed_dim
    ), (
        f"{dataset_name}: assembled transformed "
        f"dimension mismatch. "
        f"Expected={transformed_dim}, "
        f"Got={assembled_transformed_dim}."
    )

    # -------------------------------------------------------------------------------------------
    # 5.13 Explicit transformed tensor assembly
    # -------------------------------------------------------------------------------------------

    transformed_parts = []

    if num_numerical > 0:

        transformed_parts.append(
            numerical_output
        )

    transformed_parts.extend(
        categorical_logits
    )

    assert len(
        transformed_parts
    ) > 0, (
        f"{dataset_name}: no generator outputs "
        "available for transformed assembly."
    )

    transformed_output = torch.cat(
        transformed_parts,
        dim=1,
    )

    validate_tensor(
        tensor=transformed_output,
        expected_shape=(
            TEST_BATCH_SIZE,
            transformed_dim,
        ),
        tensor_name=(
            f"{dataset_name}.assembled_transformed_output"
        ),
    )

    # -------------------------------------------------------------------------------------------
    # 5.14 Batch preservation
    # -------------------------------------------------------------------------------------------

    assert (
        numerical_output.shape[0]
        == TEST_BATCH_SIZE
    ), (
        f"{dataset_name}: numerical batch dimension changed."
    )

    for feature_index, logits in enumerate(
        categorical_logits
    ):

        assert (
            logits.shape[0]
            == TEST_BATCH_SIZE
        ), (
            f"{dataset_name}: categorical head "
            f"{feature_index} batch dimension changed."
        )

    assert (
        transformed_output.shape[0]
        == TEST_BATCH_SIZE
    ), (
        f"{dataset_name}: transformed output "
        "batch dimension changed."
    )

    # -------------------------------------------------------------------------------------------
    # 5.15 Explicit finite-output validation
    # -------------------------------------------------------------------------------------------

    assert torch.isfinite(
        numerical_output
    ).all().item(), (
        f"{dataset_name}: numerical output "
        "contains non-finite values."
    )

    for feature_index, logits in enumerate(
        categorical_logits
    ):

        assert torch.isfinite(
            logits
        ).all().item(), (
            f"{dataset_name}: categorical head "
            f"{feature_index} contains non-finite values."
        )

    assert torch.isfinite(
        transformed_output
    ).all().item(), (
        f"{dataset_name}: assembled transformed "
        "output contains non-finite values."
    )

    # -------------------------------------------------------------------------------------------
    # 5.16 Record result
    # -------------------------------------------------------------------------------------------

    record = {
        "dataset": dataset_name,
        "batch_size": TEST_BATCH_SIZE,
        "latent_dim": latent_dim,
        "conditional_dim": conditional_dim,
        "num_numerical": num_numerical,
        "num_categorical": len(
            categorical_cardinalities
        ),
        "categorical_dimension": categorical_dim_actual,
        "transformed_dim_declared": transformed_dim,
        "transformed_dim_assembled": assembled_transformed_dim,
        "shape_test": "PASS",
        "finite_output_test": "PASS",
        "categorical_probability_test": "PASS",
        "batch_preservation_test": "PASS",
        "output_contract_test": "PASS",
    }

    SHAPE_TEST_RESULTS.append(
        record
    )

    print(
        f"  Numerical output       : "
        f"{tuple(numerical_output.shape)}"
    )

    print(
        f"  Categorical heads      : "
        f"{len(categorical_logits)}"
    )

    print(
        f"  Categorical dimension  : "
        f"{categorical_dim_actual}"
    )

    print(
        f"  Transformed dimension  : "
        f"{assembled_transformed_dim} / "
        f"{transformed_dim}"
    )

    print(
        "  Finite outputs         : PASS"
    )

    print(
        "  Probability validation : PASS"
    )

    print(
        "  Batch preservation     : PASS"
    )

    print(
        "  Output assembly        : PASS"
    )

    print(
        "  Output contract        : PASS"
    )

    # -------------------------------------------------------------------------------------------
    # 5.17 Restore original model state
    # -------------------------------------------------------------------------------------------

    generator = generator.to(
        original_device
    )

    generator.train(
        original_training_mode
    )

    SPPGAN_MODELS[
        dataset_name
    ]["generator"] = generator


# -----------------------------------------------------------------------------------------------
# 6. Build validation DataFrame
# -----------------------------------------------------------------------------------------------

SHAPE_TEST_DF = pd.DataFrame(
    SHAPE_TEST_RESULTS
)

EXPECTED_RESULT_COLUMNS = [
    "dataset",
    "batch_size",
    "latent_dim",
    "conditional_dim",
    "num_numerical",
    "num_categorical",
    "categorical_dimension",
    "transformed_dim_declared",
    "transformed_dim_assembled",
    "shape_test",
    "finite_output_test",
    "categorical_probability_test",
    "batch_preservation_test",
    "output_contract_test",
]

assert list(
    SHAPE_TEST_DF.columns
) == EXPECTED_RESULT_COLUMNS, (
    "Tensor shape test schema mismatch."
)

assert len(
    SHAPE_TEST_DF
) == len(
    EXPECTED_DATASETS
), (
    "Tensor shape test result count mismatch."
)

assert set(
    SHAPE_TEST_DF["dataset"]
) == set(
    EXPECTED_DATASETS
), (
    "Tensor shape test dataset coverage mismatch."
)

# -----------------------------------------------------------------------------------------------
# 7. Validate all final results
# -----------------------------------------------------------------------------------------------

VALIDATION_COLUMNS = [
    "shape_test",
    "finite_output_test",
    "categorical_probability_test",
    "batch_preservation_test",
    "output_contract_test",
]

for column in VALIDATION_COLUMNS:

    assert (
        SHAPE_TEST_DF[column] == "PASS"
    ).all(), (
        f"Failure detected in {column}."
    )

dimension_match_count = int(
    (
        SHAPE_TEST_DF[
            "transformed_dim_declared"
        ]
        ==
        SHAPE_TEST_DF[
            "transformed_dim_assembled"
        ]
    ).sum()
)

assert dimension_match_count == len(
    EXPECTED_DATASETS
), (
    "Declared and assembled transformed dimensions "
    "do not match for all datasets."
)

# -----------------------------------------------------------------------------------------------
# 8. Persist validation artifact
# -----------------------------------------------------------------------------------------------

shape_path = (
    DIRS["validation"]
    / "sppgan_tensor_shape_tests.csv"
)

SHAPE_TEST_DF.to_csv(
    shape_path,
    index=False,
)

assert shape_path.exists(), (
    f"Tensor shape test artifact was not created: "
    f"{shape_path}"
)

assert shape_path.stat().st_size > 0, (
    f"Tensor shape test artifact is empty: "
    f"{shape_path}"
)

# -----------------------------------------------------------------------------------------------
# 9. Final Section 18 summary
# -----------------------------------------------------------------------------------------------

dataset_count = len(
    SHAPE_TEST_DF
)

shape_pass_count = int(
    SHAPE_TEST_DF[
        "shape_test"
    ].eq("PASS").sum()
)

finite_pass_count = int(
    SHAPE_TEST_DF[
        "finite_output_test"
    ].eq("PASS").sum()
)

probability_pass_count = int(
    SHAPE_TEST_DF[
        "categorical_probability_test"
    ].eq("PASS").sum()
)

batch_pass_count = int(
    SHAPE_TEST_DF[
        "batch_preservation_test"
    ].eq("PASS").sum()
)

contract_pass_count = int(
    SHAPE_TEST_DF[
        "output_contract_test"
    ].eq("PASS").sum()
)

print("\n" + "=" * 100)
print("SECTION 18 VALIDATION SUMMARY")
print("=" * 100)

print(
    f"Datasets tested              : "
    f"{dataset_count}"
)

print(
    f"Shape tests                  : "
    f"{shape_pass_count}/{dataset_count} PASS"
)

print(
    f"Finite-output tests          : "
    f"{finite_pass_count}/{dataset_count} PASS"
)

print(
    f"Categorical probability      : "
    f"{probability_pass_count}/{dataset_count} PASS"
)

print(
    f"Batch-preservation tests     : "
    f"{batch_pass_count}/{dataset_count} PASS"
)

print(
    f"Output-contract tests        : "
    f"{contract_pass_count}/{dataset_count} PASS"
)

print(
    f"Declared/assembled dimensions: "
    f"{dimension_match_count}/{dataset_count} PASS"
)

print(
    f"\n✓ Saved : {shape_path}"
)

print(
    "\nSECTION 18 STATUS: PASS"
)

print("=" * 100)

18. TENSOR SHAPE & OUTPUT CONTRACT TESTS

Validation device : cpu
Test batch size   : 4

----------------------------------------------------------------------------------------------------
Dataset : adult_income
----------------------------------------------------------------------------------------------------
  Numerical output       : (4, 6)
  Categorical heads      : 8
  Categorical dimension  : 99
  Transformed dimension  : 105 / 105
  Finite outputs         : PASS
  Probability validation : PASS
  Batch preservation     : PASS
  Output assembly        : PASS
  Output contract        : PASS

----------------------------------------------------------------------------------------------------
Dataset : bank_marketing
----------------------------------------------------------------------------------------------------
  Numerical output       : (4, 7)
  Categorical heads      : 9
  Categorical dimension  : 44
  Transformed dimension  : 51 / 51
  Finite outputs         : PASS
  Probab

In [52]:
# ==================================================================================================
# 19. FORWARD-PASS TEST
# ==================================================================================================

print("=" * 100)
print("19. FORWARD-PASS TEST")
print("=" * 100)


# -----------------------------------------------------------------------------------------------
# 1. Initialize validation configuration
# -----------------------------------------------------------------------------------------------

FORWARD_TEST_RESULTS = []

TEST_BATCH_SIZE = 4

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

assert TEST_BATCH_SIZE > 0, (
    "TEST_BATCH_SIZE must be positive."
)

assert set(
    SPPGAN_ARCHITECTURES.keys()
) == set(
    EXPECTED_DATASETS
), (
    "SPPGAN_ARCHITECTURES dataset coverage mismatch."
)

assert set(
    SPPGAN_MODELS.keys()
) == set(
    EXPECTED_DATASETS
), (
    "SPPGAN_MODELS dataset coverage mismatch."
)


# -----------------------------------------------------------------------------------------------
# 2. Forward-pass validation
# -----------------------------------------------------------------------------------------------

for dataset_name in EXPECTED_DATASETS:

    print("\n" + "-" * 100)
    print(f"Dataset : {dataset_name}")
    print("-" * 100)

    architecture = SPPGAN_ARCHITECTURES[
        dataset_name
    ]

    models = SPPGAN_MODELS[
        dataset_name
    ]

    generator = models[
        "generator"
    ]

    critic = models[
        "critic"
    ]

    # -------------------------------------------------------------------------------------------
    # 2.1 Validate model objects
    # -------------------------------------------------------------------------------------------

    assert isinstance(
        generator,
        torch.nn.Module,
    ), (
        f"{dataset_name}: generator is not a torch.nn.Module."
    )

    assert isinstance(
        critic,
        torch.nn.Module,
    ), (
        f"{dataset_name}: critic is not a torch.nn.Module."
    )

    # -------------------------------------------------------------------------------------------
    # 2.2 Read architecture contract
    # -------------------------------------------------------------------------------------------

    latent_dim = int(
        architecture["latent_dim"]
    )

    conditional_dim = int(
        architecture.get(
            "conditional_dim",
            0,
        )
    )

    num_numerical = int(
        architecture["num_numerical"]
    )

    categorical_cardinalities = [
        int(cardinality)
        for cardinality in architecture[
            "categorical_cardinalities"
        ]
    ]

    num_categorical = len(
        categorical_cardinalities
    )

    declared_transformed_dim = int(
        architecture["transformed_dim"]
    )

    # -------------------------------------------------------------------------------------------
    # 2.3 Validate transformed-dimension arithmetic
    # -------------------------------------------------------------------------------------------

    expected_categorical_dimension = sum(
        categorical_cardinalities
    )

    expected_transformed_dim = (
        num_numerical
        + expected_categorical_dimension
    )

    assert (
        declared_transformed_dim
        == expected_transformed_dim
    ), (
        f"{dataset_name}: transformed dimension contract mismatch. "
        f"Declared={declared_transformed_dim}, "
        f"Calculated={expected_transformed_dim}."
    )

    # -------------------------------------------------------------------------------------------
    # 2.4 Preserve original model states and devices
    # -------------------------------------------------------------------------------------------

    generator_training_state = (
        generator.training
    )

    critic_training_state = (
        critic.training
    )

    generator_devices = {
        parameter.device
        for parameter in generator.parameters()
    }

    critic_devices = {
        parameter.device
        for parameter in critic.parameters()
    }

    assert len(generator_devices) == 1, (
        f"{dataset_name}: generator parameters "
        "are located on multiple devices."
    )

    assert len(critic_devices) == 1, (
        f"{dataset_name}: critic parameters "
        "are located on multiple devices."
    )

    generator_original_device = next(
        iter(generator_devices)
    )

    critic_original_device = next(
        iter(critic_devices)
    )

    # -------------------------------------------------------------------------------------------
    # 2.5 Use existing model device
    # -------------------------------------------------------------------------------------------

    assert (
        generator_original_device
        == critic_original_device
    ), (
        f"{dataset_name}: generator and critic "
        "are located on different devices."
    )

    device = generator_original_device

    generator.eval()
    critic.eval()

    # -------------------------------------------------------------------------------------------
    # 2.6 Generate latent input
    # -------------------------------------------------------------------------------------------

    z = torch.randn(
        TEST_BATCH_SIZE,
        latent_dim,
        device=device,
    )

    assert tuple(
        z.shape
    ) == (
        TEST_BATCH_SIZE,
        latent_dim,
    ), (
        f"{dataset_name}: latent input shape mismatch."
    )

    assert torch.isfinite(
        z
    ).all().item(), (
        f"{dataset_name}: latent input contains "
        "non-finite values."
    )

    # -------------------------------------------------------------------------------------------
    # 2.7 Validate conditional-input contract
    # -------------------------------------------------------------------------------------------

    if conditional_dim > 0:

        condition = torch.zeros(
            TEST_BATCH_SIZE,
            conditional_dim,
            device=device,
        )

        assert tuple(
            condition.shape
        ) == (
            TEST_BATCH_SIZE,
            conditional_dim,
        ), (
            f"{dataset_name}: conditional input shape mismatch."
        )

        generator_inputs = (
            z,
            condition,
        )

        conditional_test = "APPLIED"

    else:

        condition = None

        generator_inputs = (
            z,
        )

        conditional_test = "NOT_REQUIRED_DIM_0"

    # -------------------------------------------------------------------------------------------
    # 2.8 Generator forward pass
    # -------------------------------------------------------------------------------------------

    with torch.no_grad():

        generated = generator(
            *generator_inputs
        )

    assert isinstance(
        generated,
        dict,
    ), (
        f"{dataset_name}: generator output "
        "must be a dictionary."
    )

    assert "numerical" in generated, (
        f"{dataset_name}: generator output "
        "does not contain 'numerical'."
    )

    assert "categorical_logits" in generated, (
        f"{dataset_name}: generator output "
        "does not contain 'categorical_logits'."
    )

    numerical_output = generated[
        "numerical"
    ]

    categorical_outputs = generated[
        "categorical_logits"
    ]

    # -------------------------------------------------------------------------------------------
    # 2.9 Validate numerical generator output
    # -------------------------------------------------------------------------------------------

    assert isinstance(
        numerical_output,
        torch.Tensor,
    ), (
        f"{dataset_name}: numerical generator "
        "output must be a tensor."
    )

    expected_numerical_shape = (
        TEST_BATCH_SIZE,
        num_numerical,
    )

    assert tuple(
        numerical_output.shape
    ) == expected_numerical_shape, (
        f"{dataset_name}: numerical output shape mismatch. "
        f"Expected={expected_numerical_shape}, "
        f"Observed={tuple(numerical_output.shape)}."
    )

    assert torch.isfinite(
        numerical_output
    ).all().item(), (
        f"{dataset_name}: numerical output "
        "contains non-finite values."
    )

    # -------------------------------------------------------------------------------------------
    # 2.10 Validate categorical output collection
    # -------------------------------------------------------------------------------------------

    assert isinstance(
        categorical_outputs,
        (list, tuple),
    ), (
        f"{dataset_name}: categorical generator "
        "outputs must be a list or tuple."
    )

    assert len(
        categorical_outputs
    ) == num_categorical, (
        f"{dataset_name}: categorical head count mismatch. "
        f"Expected={num_categorical}, "
        f"Observed={len(categorical_outputs)}."
    )

    # -------------------------------------------------------------------------------------------
    # 2.11 Validate categorical outputs
    # -------------------------------------------------------------------------------------------

    categorical_probability_outputs = []

    categorical_shape_results = []

    for feature_index, (
        categorical_output,
        cardinality,
    ) in enumerate(
        zip(
            categorical_outputs,
            categorical_cardinalities,
        )
    ):

        expected_shape = (
            TEST_BATCH_SIZE,
            cardinality,
        )

        assert isinstance(
            categorical_output,
            torch.Tensor,
        ), (
            f"{dataset_name}: categorical output "
            f"{feature_index} must be a tensor."
        )

        assert tuple(
            categorical_output.shape
        ) == expected_shape, (
            f"{dataset_name}: categorical output "
            f"{feature_index} shape mismatch. "
            f"Expected={expected_shape}, "
            f"Observed={tuple(categorical_output.shape)}."
        )

        assert torch.isfinite(
            categorical_output
        ).all().item(), (
            f"{dataset_name}: categorical output "
            f"{feature_index} contains non-finite values."
        )

        # ---------------------------------------------------------------------------------------
        # The Section 7 generator produces differentiable categorical outputs.
        # Explicit probability normalization is used here only to validate the
        # representation passed through the forward-pass interface.
        # ---------------------------------------------------------------------------------------

        probabilities = F.softmax(
            categorical_output,
            dim=-1,
        )

        assert torch.isfinite(
            probabilities
        ).all().item(), (
            f"{dataset_name}: categorical probability "
            f"{feature_index} contains non-finite values."
        )

        assert torch.all(
            probabilities >= 0
        ).item(), (
            f"{dataset_name}: categorical probability "
            f"{feature_index} contains negative values."
        )

        probability_sums = probabilities.sum(
            dim=-1
        )

        assert torch.allclose(
            probability_sums,
            torch.ones_like(
                probability_sums
            ),
            atol=1e-5,
            rtol=1e-5,
        ), (
            f"{dataset_name}: categorical probability "
            f"{feature_index} does not sum to 1."
        )

        categorical_probability_outputs.append(
            probabilities
        )

        categorical_shape_results.append(
            expected_shape
        )

    # -------------------------------------------------------------------------------------------
    # 2.12 Construct critic input representation
    # -------------------------------------------------------------------------------------------

    representation_parts = []

    if num_numerical > 0:

        representation_parts.append(
            numerical_output
        )

    representation_parts.extend(
        categorical_probability_outputs
    )

    assert len(
        representation_parts
    ) > 0, (
        f"{dataset_name}: no generator representation "
        "components available."
    )

    synthetic_representation = torch.cat(
        representation_parts,
        dim=1,
    )

    # -------------------------------------------------------------------------------------------
    # 2.13 Validate complete transformed representation
    # -------------------------------------------------------------------------------------------

    assert synthetic_representation.ndim == 2, (
        f"{dataset_name}: transformed representation "
        "must be two-dimensional."
    )

    expected_representation_shape = (
        TEST_BATCH_SIZE,
        declared_transformed_dim,
    )

    assert tuple(
        synthetic_representation.shape
    ) == expected_representation_shape, (
        f"{dataset_name}: transformed representation "
        f"shape mismatch. "
        f"Expected={expected_representation_shape}, "
        f"Observed={tuple(synthetic_representation.shape)}."
    )

    assert (
        synthetic_representation.shape[0]
        == TEST_BATCH_SIZE
    ), (
        f"{dataset_name}: transformed representation "
        "batch size changed."
    )

    assert torch.isfinite(
        synthetic_representation
    ).all().item(), (
        f"{dataset_name}: transformed representation "
        "contains non-finite values."
    )

    # -------------------------------------------------------------------------------------------
    # 2.14 Critic forward pass
    # -------------------------------------------------------------------------------------------

    with torch.no_grad():

        critic_score = critic(
            synthetic_representation
        )

    # -------------------------------------------------------------------------------------------
    # 2.15 Validate critic output
    # -------------------------------------------------------------------------------------------

    assert isinstance(
        critic_score,
        torch.Tensor,
    ), (
        f"{dataset_name}: critic output "
        "must be a tensor."
    )

    assert critic_score.ndim == 1, (
        f"{dataset_name}: critic must produce "
        "one scalar score per sample. "
        f"Observed shape={tuple(critic_score.shape)}."
    )

    expected_critic_shape = (
        TEST_BATCH_SIZE,
    )

    assert tuple(
        critic_score.shape
    ) == expected_critic_shape, (
        f"{dataset_name}: critic output shape mismatch. "
        f"Expected={expected_critic_shape}, "
        f"Observed={tuple(critic_score.shape)}."
    )

    assert (
        critic_score.shape[0]
        == TEST_BATCH_SIZE
    ), (
        f"{dataset_name}: critic batch dimension changed."
    )

    assert torch.isfinite(
        critic_score
    ).all().item(), (
        f"{dataset_name}: critic output "
        "contains non-finite values."
    )

    # -------------------------------------------------------------------------------------------
    # 2.16 Record successful forward-pass test
    # -------------------------------------------------------------------------------------------

    FORWARD_TEST_RESULTS.append({

        "dataset": dataset_name,

        "batch_size": TEST_BATCH_SIZE,

        "latent_dim": latent_dim,

        "conditional_dim": conditional_dim,

        "conditional_test": conditional_test,

        "num_numerical": num_numerical,

        "num_categorical": num_categorical,

        "categorical_dimension": (
            expected_categorical_dimension
        ),

        "generated_numerical_shape": str(
            tuple(
                numerical_output.shape
            )
        ),

        "generated_categorical_heads": (
            num_categorical
        ),

        "transformed_representation_shape": str(
            tuple(
                synthetic_representation.shape
            )
        ),

        "expected_transformed_dim": (
            declared_transformed_dim
        ),

        "critic_shape": str(
            tuple(
                critic_score.shape
            )
        ),

        "critic_output_type": (
            "scalar_per_sample"
        ),

        "finite_generator_output": True,

        "finite_critic_output": True,

        "batch_preserved": True,

        "categorical_probability_valid": True,

        "forward_test": "PASS",
    })

    # -------------------------------------------------------------------------------------------
    # 2.17 Display dataset result
    # -------------------------------------------------------------------------------------------

    print(
        f"  Latent input            : "
        f"{tuple(z.shape)}"
    )

    print(
        f"  Numerical output        : "
        f"{tuple(numerical_output.shape)}"
    )

    print(
        f"  Categorical heads       : "
        f"{num_categorical}"
    )

    print(
        f"  Categorical dimension   : "
        f"{expected_categorical_dimension}"
    )

    print(
        f"  Transformed representation : "
        f"{tuple(synthetic_representation.shape)}"
    )

    print(
        f"  Expected transformed dim   : "
        f"{declared_transformed_dim}"
    )

    print(
        f"  Critic output            : "
        f"{tuple(critic_score.shape)}"
    )

    print(
        "  Categorical probabilities : PASS"
    )

    print(
        "  Finite generator output   : PASS"
    )

    print(
        "  Finite critic output      : PASS"
    )

    print(
        "  Batch preservation        : PASS"
    )

    print(
        "  Forward-pass contract     : PASS"
    )

    # -------------------------------------------------------------------------------------------
    # 2.18 Restore original model states
    # -------------------------------------------------------------------------------------------

    generator.train(
        generator_training_state
    )

    critic.train(
        critic_training_state
    )


# -----------------------------------------------------------------------------------------------
# 3. Create validation DataFrame
# -----------------------------------------------------------------------------------------------

FORWARD_TEST_DF = pd.DataFrame(
    FORWARD_TEST_RESULTS
)

EXPECTED_COLUMNS = [
    "dataset",
    "batch_size",
    "latent_dim",
    "conditional_dim",
    "conditional_test",
    "num_numerical",
    "num_categorical",
    "categorical_dimension",
    "generated_numerical_shape",
    "generated_categorical_heads",
    "transformed_representation_shape",
    "expected_transformed_dim",
    "critic_shape",
    "critic_output_type",
    "finite_generator_output",
    "finite_critic_output",
    "batch_preserved",
    "categorical_probability_valid",
    "forward_test",
]

assert list(
    FORWARD_TEST_DF.columns
) == EXPECTED_COLUMNS, (
    "Forward-pass validation schema mismatch."
)


# -----------------------------------------------------------------------------------------------
# 4. Validate dataset coverage
# -----------------------------------------------------------------------------------------------

assert len(
    FORWARD_TEST_DF
) == len(
    EXPECTED_DATASETS
), (
    "Forward-pass test did not produce exactly "
    "one result per dataset."
)

assert set(
    FORWARD_TEST_DF["dataset"]
) == set(
    EXPECTED_DATASETS
), (
    "Forward-pass test dataset coverage mismatch."
)


# -----------------------------------------------------------------------------------------------
# 5. Validate all forward-pass results
# -----------------------------------------------------------------------------------------------

assert (
    FORWARD_TEST_DF[
        "forward_test"
    ] == "PASS"
).all(), (
    "One or more forward-pass tests failed."
)

assert (
    FORWARD_TEST_DF[
        "finite_generator_output"
    ]
).all(), (
    "One or more generator outputs are non-finite."
)

assert (
    FORWARD_TEST_DF[
        "finite_critic_output"
    ]
).all(), (
    "One or more critic outputs are non-finite."
)

assert (
    FORWARD_TEST_DF[
        "batch_preserved"
    ]
).all(), (
    "Batch preservation failed for one or more datasets."
)

assert (
    FORWARD_TEST_DF[
        "categorical_probability_valid"
    ]
).all(), (
    "Categorical probability validation failed."
)


# -----------------------------------------------------------------------------------------------
# 6. Validate transformed-dimension contract
# -----------------------------------------------------------------------------------------------

for dataset_name in EXPECTED_DATASETS:

    architecture = SPPGAN_ARCHITECTURES[
        dataset_name
    ]

    expected_dim = int(
        architecture["transformed_dim"]
    )

    num_numerical = int(
        architecture["num_numerical"]
    )

    cardinalities = [
        int(cardinality)
        for cardinality in architecture[
            "categorical_cardinalities"
        ]
    ]

    calculated_dim = (
        num_numerical
        + sum(cardinalities)
    )

    assert calculated_dim == expected_dim, (
        f"{dataset_name}: final transformed-dimension "
        "contract validation failed."
    )


# -----------------------------------------------------------------------------------------------
# 7. Save validation artifact
# -----------------------------------------------------------------------------------------------

forward_path = (
    DIRS["validation"]
    / "sppgan_forward_pass_tests.csv"
)

FORWARD_TEST_DF.to_csv(
    forward_path,
    index=False,
)

assert forward_path.exists(), (
    f"Forward-pass validation artifact was not created: "
    f"{forward_path}"
)

assert forward_path.stat().st_size > 0, (
    f"Forward-pass validation artifact is empty: "
    f"{forward_path}"
)


# -----------------------------------------------------------------------------------------------
# 8. Final validation counts
# -----------------------------------------------------------------------------------------------

dataset_count = len(
    FORWARD_TEST_DF
)

forward_pass_count = int(
    FORWARD_TEST_DF[
        "forward_test"
    ].eq("PASS").sum()
)

generator_finite_count = int(
    FORWARD_TEST_DF[
        "finite_generator_output"
    ].sum()
)

critic_finite_count = int(
    FORWARD_TEST_DF[
        "finite_critic_output"
    ].sum()
)

batch_pass_count = int(
    FORWARD_TEST_DF[
        "batch_preserved"
    ].sum()
)

probability_pass_count = int(
    FORWARD_TEST_DF[
        "categorical_probability_valid"
    ].sum()
)


# -----------------------------------------------------------------------------------------------
# 9. Final Section 19 status
# -----------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 19 VALIDATION SUMMARY")
print("=" * 100)

print(
    f"Datasets tested              : "
    f"{dataset_count}"
)

print(
    f"Forward-pass tests           : "
    f"{forward_pass_count}/{dataset_count} PASS"
)

print(
    f"Finite generator outputs     : "
    f"{generator_finite_count}/{dataset_count} PASS"
)

print(
    f"Finite critic outputs        : "
    f"{critic_finite_count}/{dataset_count} PASS"
)

print(
    f"Categorical probabilities    : "
    f"{probability_pass_count}/{dataset_count} PASS"
)

print(
    f"Batch preservation           : "
    f"{batch_pass_count}/{dataset_count} PASS"
)

print(
    f"Transformed-dimension        : "
    f"{dataset_count}/{dataset_count} PASS"
)

print(
    f"\n✓ Saved : {forward_path}"
)

print(
    "✓ Generator forward pass validated."
)

print(
    "✓ Conditional-input contract validated."
)

print(
    "✓ Numerical and categorical outputs validated."
)

print(
    "✓ Categorical probability representation validated."
)

print(
    "✓ Transformed representation dimensions validated."
)

print(
    "✓ Critic scalar-per-sample output validated."
)

print(
    "✓ Generator and critic outputs are finite."
)

print(
    "✓ Batch dimensions preserved."
)

print(
    "\nSECTION 19 STATUS: PASS"
)

print("=" * 100)

19. FORWARD-PASS TEST

----------------------------------------------------------------------------------------------------
Dataset : adult_income
----------------------------------------------------------------------------------------------------
  Latent input            : (4, 128)
  Numerical output        : (4, 6)
  Categorical heads       : 8
  Categorical dimension   : 99
  Transformed representation : (4, 105)
  Expected transformed dim   : 105
  Critic output            : (4,)
  Categorical probabilities : PASS
  Finite generator output   : PASS
  Finite critic output      : PASS
  Batch preservation        : PASS
  Forward-pass contract     : PASS

----------------------------------------------------------------------------------------------------
Dataset : bank_marketing
----------------------------------------------------------------------------------------------------
  Latent input            : (4, 128)
  Numerical output        : (4, 7)
  Categorical heads       : 9
  Cat

In [55]:
# ==================================================================================================
# 20. BACKWARD-PASS TEST
# ==================================================================================================

print("=" * 100)
print("20. BACKWARD-PASS TEST")
print("=" * 100)


# -----------------------------------------------------------------------------------------------
# 1. Initialize validation configuration
# -----------------------------------------------------------------------------------------------

BACKWARD_TEST_RESULTS = []

TEST_BATCH_SIZE = 4

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

assert TEST_BATCH_SIZE > 0, (
    "TEST_BATCH_SIZE must be positive."
)

assert set(
    SPPGAN_ARCHITECTURES.keys()
) == set(
    EXPECTED_DATASETS
), (
    "SPPGAN_ARCHITECTURES dataset coverage mismatch."
)

assert set(
    SPPGAN_MODELS.keys()
) == set(
    EXPECTED_DATASETS
), (
    "SPPGAN_MODELS dataset coverage mismatch."
)


# -----------------------------------------------------------------------------------------------
# 2. Gradient validation helper
# -----------------------------------------------------------------------------------------------

def validate_model_gradients(
    model,
    model_name,
    dataset_name,
):
    """
    Validate gradient existence, finiteness, and non-zero propagation
    for trainable parameters.
    """

    trainable_parameters = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]

    assert len(
        trainable_parameters
    ) > 0, (
        f"{dataset_name}: {model_name} has no "
        "trainable parameters."
    )

    parameters_with_grad = 0
    parameters_with_finite_grad = 0
    parameters_with_nonzero_grad = 0

    total_gradient_elements = 0
    finite_gradient_elements = 0
    nonzero_gradient_elements = 0

    gradient_norm_squared = 0.0

    for parameter in trainable_parameters:

        if parameter.grad is None:
            continue

        parameters_with_grad += 1

        gradient = parameter.grad.detach()

        assert torch.isfinite(
            gradient
        ).all().item(), (
            f"{dataset_name}: {model_name} contains "
            "non-finite gradients."
        )

        parameters_with_finite_grad += 1

        total_gradient_elements += gradient.numel()

        finite_gradient_elements += int(
            torch.isfinite(
                gradient
            ).sum().item()
        )

        nonzero_elements = int(
            torch.count_nonzero(
                gradient
            ).item()
        )

        nonzero_gradient_elements += (
            nonzero_elements
        )

        if nonzero_elements > 0:
            parameters_with_nonzero_grad += 1

        gradient_norm_squared += float(
            torch.sum(
                gradient.detach() ** 2
            ).cpu()
        )

    assert parameters_with_grad > 0, (
        f"{dataset_name}: {model_name} received "
        "no gradients."
    )

    assert (
        parameters_with_finite_grad
        == parameters_with_grad
    ), (
        f"{dataset_name}: {model_name} contains "
        "parameters with non-finite gradients."
    )

    assert parameters_with_nonzero_grad > 0, (
        f"{dataset_name}: {model_name} has no "
        "non-zero parameter gradients."
    )

    assert total_gradient_elements > 0, (
        f"{dataset_name}: {model_name} produced "
        "no gradient elements."
    )

    gradient_norm = (
        gradient_norm_squared ** 0.5
    )

    assert torch.isfinite(
        torch.tensor(
            gradient_norm
        )
    ).item(), (
        f"{dataset_name}: {model_name} gradient "
        "norm is non-finite."
    )

    assert gradient_norm > 0.0, (
        f"{dataset_name}: {model_name} gradient "
        "norm is zero."
    )

    return {
        "trainable_parameters": len(
            trainable_parameters
        ),
        "parameters_with_grad": (
            parameters_with_grad
        ),
        "parameters_with_finite_grad": (
            parameters_with_finite_grad
        ),
        "parameters_with_nonzero_grad": (
            parameters_with_nonzero_grad
        ),
        "gradient_elements": (
            total_gradient_elements
        ),
        "finite_gradient_elements": (
            finite_gradient_elements
        ),
        "nonzero_gradient_elements": (
            nonzero_gradient_elements
        ),
        "gradient_norm": gradient_norm,
    }


# -----------------------------------------------------------------------------------------------
# 3. Run backward-pass tests
# -----------------------------------------------------------------------------------------------

for dataset_name in EXPECTED_DATASETS:

    print("\n" + "-" * 100)
    print(f"Dataset : {dataset_name}")
    print("-" * 100)

    architecture = SPPGAN_ARCHITECTURES[
        dataset_name
    ]

    models = SPPGAN_MODELS[
        dataset_name
    ]

    generator = models[
        "generator"
    ]

    critic = models[
        "critic"
    ]

    # -------------------------------------------------------------------------------------------
    # 3.1 Validate model objects
    # -------------------------------------------------------------------------------------------

    assert isinstance(
        generator,
        torch.nn.Module,
    ), (
        f"{dataset_name}: generator is not "
        "a torch.nn.Module."
    )

    assert isinstance(
        critic,
        torch.nn.Module,
    ), (
        f"{dataset_name}: critic is not "
        "a torch.nn.Module."
    )

    # -------------------------------------------------------------------------------------------
    # 3.2 Read architecture contract
    # -------------------------------------------------------------------------------------------

    latent_dim = int(
        architecture["latent_dim"]
    )

    conditional_dim = int(
        architecture.get(
            "conditional_dim",
            0,
        )
    )

    num_numerical = int(
        architecture["num_numerical"]
    )

    categorical_cardinalities = [
        int(cardinality)
        for cardinality in architecture[
            "categorical_cardinalities"
        ]
    ]

    num_categorical = len(
        categorical_cardinalities
    )

    declared_transformed_dim = int(
        architecture["transformed_dim"]
    )

    expected_categorical_dimension = sum(
        categorical_cardinalities
    )

    expected_transformed_dim = (
        num_numerical
        + expected_categorical_dimension
    )

    assert (
        declared_transformed_dim
        == expected_transformed_dim
    ), (
        f"{dataset_name}: transformed dimension "
        "contract mismatch."
    )

    # -------------------------------------------------------------------------------------------
    # 3.3 Preserve original model states and devices
    # -------------------------------------------------------------------------------------------

    generator_training_state = (
        generator.training
    )

    critic_training_state = (
        critic.training
    )

    generator_devices = {
        parameter.device
        for parameter in generator.parameters()
    }

    critic_devices = {
        parameter.device
        for parameter in critic.parameters()
    }

    assert len(
        generator_devices
    ) == 1, (
        f"{dataset_name}: generator parameters "
        "are on multiple devices."
    )

    assert len(
        critic_devices
    ) == 1, (
        f"{dataset_name}: critic parameters "
        "are on multiple devices."
    )

    generator_device = next(
        iter(generator_devices)
    )

    critic_device = next(
        iter(critic_devices)
    )

    assert (
        generator_device
        == critic_device
    ), (
        f"{dataset_name}: generator and critic "
        "are on different devices."
    )

    device = generator_device

    # -------------------------------------------------------------------------------------------
    # 3.4 Enable training mode for gradient propagation
    # -------------------------------------------------------------------------------------------

    generator.train()
    critic.train()

    # -------------------------------------------------------------------------------------------
    # 3.5 Clear existing gradients
    # -------------------------------------------------------------------------------------------

    generator.zero_grad(
        set_to_none=True
    )

    critic.zero_grad(
        set_to_none=True
    )

    # -------------------------------------------------------------------------------------------
    # 3.6 Generate latent input
    # -------------------------------------------------------------------------------------------

    z = torch.randn(
        TEST_BATCH_SIZE,
        latent_dim,
        device=device,
        requires_grad=True,
    )

    assert tuple(
        z.shape
    ) == (
        TEST_BATCH_SIZE,
        latent_dim,
    ), (
        f"{dataset_name}: latent input shape mismatch."
    )

    assert z.requires_grad, (
        f"{dataset_name}: latent input "
        "does not require gradients."
    )

    assert torch.isfinite(
        z
    ).all().item(), (
        f"{dataset_name}: latent input "
        "contains non-finite values."
    )

    # -------------------------------------------------------------------------------------------
    # 3.7 Construct conditional input when required
    # -------------------------------------------------------------------------------------------

    if conditional_dim > 0:

        condition = torch.zeros(
            TEST_BATCH_SIZE,
            conditional_dim,
            device=device,
            requires_grad=False,
        )

        generator_inputs = (
            z,
            condition,
        )

        conditional_test = "APPLIED"

    else:

        condition = None

        generator_inputs = (
            z,
        )

        conditional_test = "NOT_REQUIRED_DIM_0"

    # -------------------------------------------------------------------------------------------
    # 3.8 Generator forward pass
    # -------------------------------------------------------------------------------------------

    generated = generator(
        *generator_inputs
    )

    assert isinstance(
        generated,
        dict,
    ), (
        f"{dataset_name}: generator output "
        "must be a dictionary."
    )

    assert "numerical" in generated, (
        f"{dataset_name}: generator output "
        "missing 'numerical'."
    )

    assert "categorical_logits" in generated, (
        f"{dataset_name}: generator output "
        "missing 'categorical_logits'."
    )

    numerical_output = generated[
        "numerical"
    ]

    categorical_outputs = generated[
        "categorical_logits"
    ]

    # -------------------------------------------------------------------------------------------
    # 3.9 Validate numerical output
    # -------------------------------------------------------------------------------------------

    assert isinstance(
        numerical_output,
        torch.Tensor,
    ), (
        f"{dataset_name}: numerical output "
        "must be a tensor."
    )

    assert tuple(
        numerical_output.shape
    ) == (
        TEST_BATCH_SIZE,
        num_numerical,
    ), (
        f"{dataset_name}: numerical output "
        "shape mismatch."
    )

    assert torch.isfinite(
        numerical_output
    ).all().item(), (
        f"{dataset_name}: numerical output "
        "contains non-finite values."
    )

    # -------------------------------------------------------------------------------------------
    # 3.10 Validate categorical outputs
    # -------------------------------------------------------------------------------------------

    assert isinstance(
        categorical_outputs,
        (list, tuple),
    ), (
        f"{dataset_name}: categorical outputs "
        "must be a list or tuple."
    )

    assert len(
        categorical_outputs
    ) == num_categorical, (
        f"{dataset_name}: categorical head "
        "count mismatch."
    )

    categorical_probability_outputs = []

    for feature_index, (
        categorical_output,
        cardinality,
    ) in enumerate(
        zip(
            categorical_outputs,
            categorical_cardinalities,
        )
    ):

        expected_shape = (
            TEST_BATCH_SIZE,
            cardinality,
        )

        assert isinstance(
            categorical_output,
            torch.Tensor,
        ), (
            f"{dataset_name}: categorical output "
            f"{feature_index} must be a tensor."
        )

        assert tuple(
            categorical_output.shape
        ) == expected_shape, (
            f"{dataset_name}: categorical output "
            f"{feature_index} shape mismatch."
        )

        assert torch.isfinite(
            categorical_output
        ).all().item(), (
            f"{dataset_name}: categorical output "
            f"{feature_index} contains non-finite values."
        )

        # ---------------------------------------------------------------------------------------
        # Validate categorical probability representation.
        # ---------------------------------------------------------------------------------------

        probabilities = F.softmax(
            categorical_output,
            dim=-1,
        )

        assert torch.isfinite(
            probabilities
        ).all().item(), (
            f"{dataset_name}: categorical probability "
            f"{feature_index} contains non-finite values."
        )

        assert torch.all(
            probabilities >= 0
        ).item(), (
            f"{dataset_name}: categorical probability "
            f"{feature_index} contains negative values."
        )

        probability_sums = probabilities.sum(
            dim=-1
        )

        assert torch.allclose(
            probability_sums,
            torch.ones_like(
                probability_sums
            ),
            atol=1e-5,
            rtol=1e-5,
        ), (
            f"{dataset_name}: categorical probability "
            f"{feature_index} does not sum to 1."
        )

        categorical_probability_outputs.append(
            probabilities
        )

    # -------------------------------------------------------------------------------------------
    # 3.11 Assemble transformed representation
    # -------------------------------------------------------------------------------------------

    representation_parts = []

    if num_numerical > 0:

        representation_parts.append(
            numerical_output
        )

    representation_parts.extend(
        categorical_probability_outputs
    )

    assert len(
        representation_parts
    ) > 0, (
        f"{dataset_name}: no representation "
        "components available."
    )

    synthetic_representation = torch.cat(
        representation_parts,
        dim=1,
    )

    # -------------------------------------------------------------------------------------------
    # 3.12 Validate transformed representation
    # -------------------------------------------------------------------------------------------

    assert synthetic_representation.ndim == 2, (
        f"{dataset_name}: transformed representation "
        "must be two-dimensional."
    )

    assert tuple(
        synthetic_representation.shape
    ) == (
        TEST_BATCH_SIZE,
        declared_transformed_dim,
    ), (
        f"{dataset_name}: transformed representation "
        "shape mismatch."
    )

    assert torch.isfinite(
        synthetic_representation
    ).all().item(), (
        f"{dataset_name}: transformed representation "
        "contains non-finite values."
    )

    # -------------------------------------------------------------------------------------------
    # 3.13 Critic forward pass
    # -------------------------------------------------------------------------------------------

    critic_score = critic(
        synthetic_representation
    )

    # -------------------------------------------------------------------------------------------
    # 3.14 Validate critic output
    # -------------------------------------------------------------------------------------------

    assert isinstance(
        critic_score,
        torch.Tensor,
    ), (
        f"{dataset_name}: critic output "
        "must be a tensor."
    )

    assert critic_score.ndim == 1, (
        f"{dataset_name}: critic must return "
        "one scalar per sample."
    )

    assert tuple(
        critic_score.shape
    ) == (
        TEST_BATCH_SIZE,
    ), (
        f"{dataset_name}: critic output shape mismatch."
    )

    assert torch.isfinite(
        critic_score
    ).all().item(), (
        f"{dataset_name}: critic output "
        "contains non-finite values."
    )

    # -------------------------------------------------------------------------------------------
    # 3.15 Construct differentiable test loss
    # -------------------------------------------------------------------------------------------

    loss = -critic_score.mean()

    assert loss.ndim == 0, (
        f"{dataset_name}: backward test loss "
        "must be scalar."
    )

    assert torch.isfinite(
        loss
    ).item(), (
        f"{dataset_name}: backward test loss "
        "is non-finite."
    )

    assert loss.requires_grad, (
        f"{dataset_name}: backward test loss "
        "does not require gradients."
    )

    # -------------------------------------------------------------------------------------------
    # 3.16 Backward pass
    # -------------------------------------------------------------------------------------------

    loss.backward()

    # -------------------------------------------------------------------------------------------
    # 3.17 Validate generator gradients
    # -------------------------------------------------------------------------------------------

    generator_gradient_results = (
        validate_model_gradients(
            model=generator,
            model_name="generator",
            dataset_name=dataset_name,
        )
    )

    # -------------------------------------------------------------------------------------------
    # 3.18 Validate critic gradients
    # -------------------------------------------------------------------------------------------

    critic_gradient_results = (
        validate_model_gradients(
            model=critic,
            model_name="critic",
            dataset_name=dataset_name,
        )
    )

    # -------------------------------------------------------------------------------------------
    # 3.19 Validate latent gradient propagation
    # -------------------------------------------------------------------------------------------

    assert z.grad is not None, (
        f"{dataset_name}: latent input "
        "received no gradient."
    )

    assert torch.isfinite(
        z.grad
    ).all().item(), (
        f"{dataset_name}: latent input gradient "
        "contains non-finite values."
    )

    latent_gradient_norm = float(
        torch.linalg.vector_norm(
            z.grad.detach()
        ).cpu()
    )

    assert latent_gradient_norm > 0.0, (
        f"{dataset_name}: latent input "
        "gradient norm is zero."
    )

    # -------------------------------------------------------------------------------------------
    # 3.20 Record successful validation
    # -------------------------------------------------------------------------------------------

    BACKWARD_TEST_RESULTS.append({

        "dataset": dataset_name,

        "batch_size": TEST_BATCH_SIZE,

        "latent_dim": latent_dim,

        "conditional_dim": conditional_dim,

        "conditional_test": conditional_test,

        "num_numerical": num_numerical,

        "num_categorical": num_categorical,

        "transformed_dim": declared_transformed_dim,

        "loss": float(
            loss.detach().cpu()
        ),

        "generator_trainable_parameters": (
            generator_gradient_results[
                "trainable_parameters"
            ]
        ),

        "generator_parameters_with_grad": (
            generator_gradient_results[
                "parameters_with_grad"
            ]
        ),

        "generator_parameters_with_nonzero_grad": (
            generator_gradient_results[
                "parameters_with_nonzero_grad"
            ]
        ),

        "generator_gradient_norm": (
            generator_gradient_results[
                "gradient_norm"
            ]
        ),

        "critic_trainable_parameters": (
            critic_gradient_results[
                "trainable_parameters"
            ]
        ),

        "critic_parameters_with_grad": (
            critic_gradient_results[
                "parameters_with_grad"
            ]
        ),

        "critic_parameters_with_nonzero_grad": (
            critic_gradient_results[
                "parameters_with_nonzero_grad"
            ]
        ),

        "critic_gradient_norm": (
            critic_gradient_results[
                "gradient_norm"
            ]
        ),

        "latent_gradient_norm": (
            latent_gradient_norm
        ),

        "finite_loss": True,

        "finite_generator_gradients": True,

        "finite_critic_gradients": True,

        "latent_gradient_valid": True,

        "backward_test": "PASS",
    })

    # -------------------------------------------------------------------------------------------
    # 3.21 Display validation result
    # -------------------------------------------------------------------------------------------

    print(
        f"  Loss                       : "
        f"{float(loss.detach().cpu()):.6f}"
    )

    print(
        f"  Generator gradients        : "
        f"{generator_gradient_results['parameters_with_grad']}/"
        f"{generator_gradient_results['trainable_parameters']} "
        f"parameters"
    )

    print(
        f"  Generator non-zero grads   : "
        f"{generator_gradient_results['parameters_with_nonzero_grad']}/"
        f"{generator_gradient_results['trainable_parameters']} "
        f"parameters"
    )

    print(
        f"  Generator gradient norm    : "
        f"{generator_gradient_results['gradient_norm']:.6e}"
    )

    print(
        f"  Critic gradients           : "
        f"{critic_gradient_results['parameters_with_grad']}/"
        f"{critic_gradient_results['trainable_parameters']} "
        f"parameters"
    )

    print(
        f"  Critic non-zero grads      : "
        f"{critic_gradient_results['parameters_with_nonzero_grad']}/"
        f"{critic_gradient_results['trainable_parameters']} "
        f"parameters"
    )

    print(
        f"  Critic gradient norm       : "
        f"{critic_gradient_results['gradient_norm']:.6e}"
    )

    print(
        f"  Latent gradient norm       : "
        f"{latent_gradient_norm:.6e}"
    )

    print(
        "  Finite loss               : PASS"
    )

    print(
        "  Finite generator gradients: PASS"
    )

    print(
        "  Finite critic gradients   : PASS"
    )

    print(
        "  Latent gradient           : PASS"
    )

    print(
        "  Backward-pass contract    : PASS"
    )

    # -------------------------------------------------------------------------------------------
    # 3.22 Clear gradients before restoring state
    # -------------------------------------------------------------------------------------------

    generator.zero_grad(
        set_to_none=True
    )

    critic.zero_grad(
        set_to_none=True
    )

    # -------------------------------------------------------------------------------------------
    # 3.23 Restore original model states
    # -------------------------------------------------------------------------------------------

    generator.train(
        generator_training_state
    )

    critic.train(
        critic_training_state
    )


# -----------------------------------------------------------------------------------------------
# 4. Create validation DataFrame
# -----------------------------------------------------------------------------------------------

BACKWARD_TEST_DF = pd.DataFrame(
    BACKWARD_TEST_RESULTS
)

EXPECTED_COLUMNS = [
    "dataset",
    "batch_size",
    "latent_dim",
    "conditional_dim",
    "conditional_test",
    "num_numerical",
    "num_categorical",
    "transformed_dim",
    "loss",
    "generator_trainable_parameters",
    "generator_parameters_with_grad",
    "generator_parameters_with_nonzero_grad",
    "generator_gradient_norm",
    "critic_trainable_parameters",
    "critic_parameters_with_grad",
    "critic_parameters_with_nonzero_grad",
    "critic_gradient_norm",
    "latent_gradient_norm",
    "finite_loss",
    "finite_generator_gradients",
    "finite_critic_gradients",
    "latent_gradient_valid",
    "backward_test",
]

assert list(
    BACKWARD_TEST_DF.columns
) == EXPECTED_COLUMNS, (
    "Backward-pass validation schema mismatch."
)


# -----------------------------------------------------------------------------------------------
# 5. Validate dataset coverage
# -----------------------------------------------------------------------------------------------

assert len(
    BACKWARD_TEST_DF
) == len(
    EXPECTED_DATASETS
), (
    "Backward-pass test did not produce exactly "
    "one result per dataset."
)

assert set(
    BACKWARD_TEST_DF["dataset"]
) == set(
    EXPECTED_DATASETS
), (
    "Backward-pass test dataset coverage mismatch."
)


# -----------------------------------------------------------------------------------------------
# 6. Validate final results
# -----------------------------------------------------------------------------------------------

assert (
    BACKWARD_TEST_DF[
        "backward_test"
    ] == "PASS"
).all(), (
    "One or more backward-pass tests failed."
)

assert (
    BACKWARD_TEST_DF[
        "finite_loss"
    ]
).all(), (
    "One or more backward-pass losses are non-finite."
)

assert (
    BACKWARD_TEST_DF[
        "finite_generator_gradients"
    ]
).all(), (
    "One or more generator gradient sets are non-finite."
)

assert (
    BACKWARD_TEST_DF[
        "finite_critic_gradients"
    ]
).all(), (
    "One or more critic gradient sets are non-finite."
)

assert (
    BACKWARD_TEST_DF[
        "latent_gradient_valid"
    ]
).all(), (
    "Latent gradient propagation failed."
)

assert (
    BACKWARD_TEST_DF[
        "generator_parameters_with_grad"
    ]
    ==
    BACKWARD_TEST_DF[
        "generator_trainable_parameters"
    ]
).all(), (
    "Not all trainable generator parameters "
    "received gradients."
)

assert (
    BACKWARD_TEST_DF[
        "critic_parameters_with_grad"
    ]
    ==
    BACKWARD_TEST_DF[
        "critic_trainable_parameters"
    ]
).all(), (
    "Not all trainable critic parameters "
    "received gradients."
)

assert (
    BACKWARD_TEST_DF[
        "generator_parameters_with_nonzero_grad"
    ] > 0
).all(), (
    "Generator has no non-zero parameter gradients."
)

assert (
    BACKWARD_TEST_DF[
        "critic_parameters_with_nonzero_grad"
    ] > 0
).all(), (
    "Critic has no non-zero parameter gradients."
)

assert (
    BACKWARD_TEST_DF[
        "generator_gradient_norm"
    ] > 0
).all(), (
    "One or more generator gradient norms are zero."
)

assert (
    BACKWARD_TEST_DF[
        "critic_gradient_norm"
    ] > 0
).all(), (
    "One or more critic gradient norms are zero."
)

assert (
    BACKWARD_TEST_DF[
        "latent_gradient_norm"
    ] > 0
).all(), (
    "One or more latent gradient norms are zero."
)


# -----------------------------------------------------------------------------------------------
# 7. Validate transformed dimensions
# -----------------------------------------------------------------------------------------------

for dataset_name in EXPECTED_DATASETS:

    architecture = SPPGAN_ARCHITECTURES[
        dataset_name
    ]

    expected_dim = int(
        architecture["transformed_dim"]
    )

    num_numerical = int(
        architecture["num_numerical"]
    )

    cardinalities = [
        int(cardinality)
        for cardinality in architecture[
            "categorical_cardinalities"
        ]
    ]

    calculated_dim = (
        num_numerical
        + sum(cardinalities)
    )

    assert calculated_dim == expected_dim, (
        f"{dataset_name}: transformed-dimension "
        "validation failed."
    )


# -----------------------------------------------------------------------------------------------
# 8. Save validation artifact
# -----------------------------------------------------------------------------------------------

backward_path = (
    DIRS["validation"]
    / "sppgan_backward_pass_tests.csv"
)

BACKWARD_TEST_DF.to_csv(
    backward_path,
    index=False,
)

assert backward_path.exists(), (
    f"Backward-pass validation artifact was not created: "
    f"{backward_path}"
)

assert backward_path.stat().st_size > 0, (
    f"Backward-pass validation artifact is empty: "
    f"{backward_path}"
)


# -----------------------------------------------------------------------------------------------
# 9. Final validation counts
# -----------------------------------------------------------------------------------------------

dataset_count = len(
    BACKWARD_TEST_DF
)

backward_pass_count = int(
    BACKWARD_TEST_DF[
        "backward_test"
    ].eq("PASS").sum()
)

generator_gradient_pass_count = int(
    (
        BACKWARD_TEST_DF[
            "generator_parameters_with_grad"
        ]
        ==
        BACKWARD_TEST_DF[
            "generator_trainable_parameters"
        ]
    ).sum()
)

critic_gradient_pass_count = int(
    (
        BACKWARD_TEST_DF[
            "critic_parameters_with_grad"
        ]
        ==
        BACKWARD_TEST_DF[
            "critic_trainable_parameters"
        ]
    ).sum()
)

finite_loss_count = int(
    BACKWARD_TEST_DF[
        "finite_loss"
    ].sum()
)

finite_generator_gradient_count = int(
    BACKWARD_TEST_DF[
        "finite_generator_gradients"
    ].sum()
)

finite_critic_gradient_count = int(
    BACKWARD_TEST_DF[
        "finite_critic_gradients"
    ].sum()
)

latent_gradient_pass_count = int(
    BACKWARD_TEST_DF[
        "latent_gradient_valid"
    ].sum()
)


# -----------------------------------------------------------------------------------------------
# 10. Final Section 20 status
# -----------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 20 VALIDATION SUMMARY")
print("=" * 100)

print(
    f"Datasets tested               : "
    f"{dataset_count}"
)

print(
    f"Backward-pass tests           : "
    f"{backward_pass_count}/{dataset_count} PASS"
)

print(
    f"Finite loss                   : "
    f"{finite_loss_count}/{dataset_count} PASS"
)

print(
    f"Generator gradient coverage   : "
    f"{generator_gradient_pass_count}/{dataset_count} PASS"
)

print(
    f"Critic gradient coverage      : "
    f"{critic_gradient_pass_count}/{dataset_count} PASS"
)

print(
    f"Finite generator gradients    : "
    f"{finite_generator_gradient_count}/"
    f"{dataset_count} PASS"
)

print(
    f"Finite critic gradients       : "
    f"{finite_critic_gradient_count}/"
    f"{dataset_count} PASS"
)

print(
    f"Latent gradient propagation   : "
    f"{latent_gradient_pass_count}/"
    f"{dataset_count} PASS"
)

print(
    f"\n✓ Saved : {backward_path}"
)

print(
    "✓ Generator backward propagation validated."
)

print(
    "✓ Critic backward propagation validated."
)

print(
    "✓ Trainable generator parameters received gradients."
)

print(
    "✓ Trainable critic parameters received gradients."
)

print(
    "✓ Generator gradients are finite."
)

print(
    "✓ Critic gradients are finite."
)

print(
    "✓ Latent-to-generator gradient propagation validated."
)

print(
    "\nSECTION 20 STATUS: PASS"
)

print("=" * 100)

20. BACKWARD-PASS TEST

----------------------------------------------------------------------------------------------------
Dataset : adult_income
----------------------------------------------------------------------------------------------------
  Loss                       : 0.027056
  Generator gradients        : 22/22 parameters
  Generator non-zero grads   : 22/22 parameters
  Generator gradient norm    : 6.173906e-02
  Critic gradients           : 6/6 parameters
  Critic non-zero grads      : 6/6 parameters
  Critic gradient norm       : 1.301831e+00
  Latent gradient norm       : 9.686108e-04
  Finite loss               : PASS
  Finite generator gradients: PASS
  Finite critic gradients   : PASS
  Latent gradient           : PASS
  Backward-pass contract    : PASS

----------------------------------------------------------------------------------------------------
Dataset : bank_marketing
-----------------------------------------------------------------------------------------

In [58]:
# ==================================================================================================
# 21. GRADIENT VALIDATION
# ==================================================================================================

print("=" * 100)
print("21. GRADIENT VALIDATION")
print("=" * 100)


# -----------------------------------------------------------------------------------------------
# 1. Initialize validation configuration
# -----------------------------------------------------------------------------------------------

GRADIENT_RESULTS = []

TEST_BATCH_SIZE = 4

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

assert TEST_BATCH_SIZE > 0, (
    "TEST_BATCH_SIZE must be positive."
)

assert set(
    SPPGAN_ARCHITECTURES.keys()
) == set(
    EXPECTED_DATASETS
), (
    "SPPGAN_ARCHITECTURES dataset coverage mismatch."
)

assert set(
    SPPGAN_MODELS.keys()
) == set(
    EXPECTED_DATASETS
), (
    "SPPGAN_MODELS dataset coverage mismatch."
)


# -----------------------------------------------------------------------------------------------
# 2. Gradient validation helper
# -----------------------------------------------------------------------------------------------

def analyze_gradients(
    model,
    model_name,
    dataset_name,
):
    """
    Analyze gradients for all trainable parameters.

    Validates:
        - trainable parameter coverage
        - gradient existence
        - finite gradients
        - non-zero gradient coverage
        - gradient element coverage
        - aggregate gradient norm
    """

    trainable_parameters = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]

    assert len(
        trainable_parameters
    ) > 0, (
        f"{dataset_name}: {model_name} has no "
        "trainable parameters."
    )

    parameters_with_grad = 0
    parameters_without_grad = 0
    parameters_with_nonzero_grad = 0

    total_gradient_elements = 0
    finite_gradient_elements = 0
    nonzero_gradient_elements = 0

    gradient_norm_squared = 0.0

    for parameter in trainable_parameters:

        if parameter.grad is None:

            parameters_without_grad += 1

            continue

        parameters_with_grad += 1

        gradient = parameter.grad.detach()

        total_gradient_elements += gradient.numel()

        finite_mask = torch.isfinite(
            gradient
        )

        finite_count = int(
            finite_mask.sum().item()
        )

        finite_gradient_elements += (
            finite_count
        )

        assert finite_count == gradient.numel(), (
            f"{dataset_name}: {model_name} "
            "contains non-finite gradient values."
        )

        nonzero_count = int(
            torch.count_nonzero(
                gradient
            ).item()
        )

        nonzero_gradient_elements += (
            nonzero_count
        )

        if nonzero_count > 0:

            parameters_with_nonzero_grad += 1

        gradient_norm_squared += float(
            gradient.pow(2).sum().cpu()
        )

    gradient_norm = (
        gradient_norm_squared ** 0.5
    )

    assert parameters_with_grad > 0, (
        f"{dataset_name}: {model_name} "
        "received no gradients."
    )

    assert gradient_norm > 0.0, (
        f"{dataset_name}: {model_name} "
        "aggregate gradient norm is zero."
    )

    assert math.isfinite(
        gradient_norm
    ), (
        f"{dataset_name}: {model_name} "
        "gradient norm is non-finite."
    )

    gradient_coverage = (
        parameters_with_grad
        / len(trainable_parameters)
    )

    nonzero_parameter_coverage = (
        parameters_with_nonzero_grad
        / len(trainable_parameters)
    )

    if total_gradient_elements > 0:

        finite_element_coverage = (
            finite_gradient_elements
            / total_gradient_elements
        )

        nonzero_element_coverage = (
            nonzero_gradient_elements
            / total_gradient_elements
        )

    else:

        finite_element_coverage = 0.0
        nonzero_element_coverage = 0.0

    return {
        "trainable_parameters": len(
            trainable_parameters
        ),
        "parameters_with_grad": (
            parameters_with_grad
        ),
        "parameters_without_grad": (
            parameters_without_grad
        ),
        "parameters_with_nonzero_grad": (
            parameters_with_nonzero_grad
        ),
        "gradient_coverage": (
            gradient_coverage
        ),
        "nonzero_parameter_coverage": (
            nonzero_parameter_coverage
        ),
        "gradient_elements": (
            total_gradient_elements
        ),
        "finite_gradient_elements": (
            finite_gradient_elements
        ),
        "nonzero_gradient_elements": (
            nonzero_gradient_elements
        ),
        "finite_element_coverage": (
            finite_element_coverage
        ),
        "nonzero_element_coverage": (
            nonzero_element_coverage
        ),
        "gradient_norm": (
            gradient_norm
        ),
    }


# -----------------------------------------------------------------------------------------------
# 3. Validate gradients for every dataset
# -----------------------------------------------------------------------------------------------

for dataset_name in EXPECTED_DATASETS:

    print("\n" + "-" * 100)
    print(f"Dataset : {dataset_name}")
    print("-" * 100)

    architecture = SPPGAN_ARCHITECTURES[
        dataset_name
    ]

    models = SPPGAN_MODELS[
        dataset_name
    ]

    generator = models[
        "generator"
    ]

    critic = models[
        "critic"
    ]

    # -------------------------------------------------------------------------------------------
    # 3.1 Validate model objects
    # -------------------------------------------------------------------------------------------

    assert isinstance(
        generator,
        torch.nn.Module,
    ), (
        f"{dataset_name}: generator is not "
        "a torch.nn.Module."
    )

    assert isinstance(
        critic,
        torch.nn.Module,
    ), (
        f"{dataset_name}: critic is not "
        "a torch.nn.Module."
    )

    # -------------------------------------------------------------------------------------------
    # 3.2 Read architecture contract
    # -------------------------------------------------------------------------------------------

    latent_dim = int(
        architecture["latent_dim"]
    )

    conditional_dim = int(
        architecture.get(
            "conditional_dim",
            0,
        )
    )

    num_numerical = int(
        architecture["num_numerical"]
    )

    categorical_cardinalities = [
        int(cardinality)
        for cardinality in architecture[
            "categorical_cardinalities"
        ]
    ]

    num_categorical = len(
        categorical_cardinalities
    )

    transformed_dim = int(
        architecture["transformed_dim"]
    )

    calculated_transformed_dim = (
        num_numerical
        + sum(
            categorical_cardinalities
        )
    )

    assert (
        transformed_dim
        == calculated_transformed_dim
    ), (
        f"{dataset_name}: transformed dimension "
        "contract mismatch."
    )

    # -------------------------------------------------------------------------------------------
    # 3.3 Preserve original model states
    # -------------------------------------------------------------------------------------------

    generator_training_state = (
        generator.training
    )

    critic_training_state = (
        critic.training
    )

    generator_devices = {
        parameter.device
        for parameter in generator.parameters()
    }

    critic_devices = {
        parameter.device
        for parameter in critic.parameters()
    }

    assert len(
        generator_devices
    ) == 1, (
        f"{dataset_name}: generator parameters "
        "are on multiple devices."
    )

    assert len(
        critic_devices
    ) == 1, (
        f"{dataset_name}: critic parameters "
        "are on multiple devices."
    )

    generator_device = next(
        iter(generator_devices)
    )

    critic_device = next(
        iter(critic_devices)
    )

    assert (
        generator_device
        == critic_device
    ), (
        f"{dataset_name}: generator and critic "
        "are on different devices."
    )

    device = generator_device

    # -------------------------------------------------------------------------------------------
    # 3.4 Enable training mode
    # -------------------------------------------------------------------------------------------

    generator.train()
    critic.train()

    # -------------------------------------------------------------------------------------------
    # 3.5 Clear existing gradients
    # -------------------------------------------------------------------------------------------

    generator.zero_grad(
        set_to_none=True
    )

    critic.zero_grad(
        set_to_none=True
    )

    # -------------------------------------------------------------------------------------------
    # 3.6 Generate latent input
    # -------------------------------------------------------------------------------------------

    z = torch.randn(
        TEST_BATCH_SIZE,
        latent_dim,
        device=device,
        requires_grad=True,
    )

    assert tuple(
        z.shape
    ) == (
        TEST_BATCH_SIZE,
        latent_dim,
    ), (
        f"{dataset_name}: latent input shape mismatch."
    )

    assert z.requires_grad, (
        f"{dataset_name}: latent input "
        "does not require gradients."
    )

    assert torch.isfinite(
        z
    ).all().item(), (
        f"{dataset_name}: latent input "
        "contains non-finite values."
    )

    # -------------------------------------------------------------------------------------------
    # 3.7 Conditional-input contract
    # -------------------------------------------------------------------------------------------

    if conditional_dim > 0:

        condition = torch.zeros(
            TEST_BATCH_SIZE,
            conditional_dim,
            device=device,
        )

        generator_inputs = (
            z,
            condition,
        )

        conditional_test = "APPLIED"

    else:

        condition = None

        generator_inputs = (
            z,
        )

        conditional_test = "NOT_REQUIRED_DIM_0"

    # -------------------------------------------------------------------------------------------
    # 3.8 Generator forward pass
    # -------------------------------------------------------------------------------------------

    generated = generator(
        *generator_inputs
    )

    assert isinstance(
        generated,
        dict,
    ), (
        f"{dataset_name}: generator output "
        "must be a dictionary."
    )

    assert "numerical" in generated, (
        f"{dataset_name}: generator output "
        "missing 'numerical'."
    )

    assert "categorical_logits" in generated, (
        f"{dataset_name}: generator output "
        "missing 'categorical_logits'."
    )

    numerical_output = generated[
        "numerical"
    ]

    categorical_outputs = generated[
        "categorical_logits"
    ]

    # -------------------------------------------------------------------------------------------
    # 3.9 Validate numerical output
    # -------------------------------------------------------------------------------------------

    assert isinstance(
        numerical_output,
        torch.Tensor,
    ), (
        f"{dataset_name}: numerical output "
        "must be a tensor."
    )

    assert tuple(
        numerical_output.shape
    ) == (
        TEST_BATCH_SIZE,
        num_numerical,
    ), (
        f"{dataset_name}: numerical output "
        "shape mismatch."
    )

    assert torch.isfinite(
        numerical_output
    ).all().item(), (
        f"{dataset_name}: numerical output "
        "contains non-finite values."
    )

    # -------------------------------------------------------------------------------------------
    # 3.10 Validate categorical outputs
    # -------------------------------------------------------------------------------------------

    assert isinstance(
        categorical_outputs,
        (list, tuple),
    ), (
        f"{dataset_name}: categorical outputs "
        "must be a list or tuple."
    )

    assert len(
        categorical_outputs
    ) == num_categorical, (
        f"{dataset_name}: categorical head "
        "count mismatch."
    )

    categorical_probability_outputs = []

    for feature_index, (
        categorical_output,
        cardinality,
    ) in enumerate(
        zip(
            categorical_outputs,
            categorical_cardinalities,
        )
    ):

        expected_shape = (
            TEST_BATCH_SIZE,
            cardinality,
        )

        assert isinstance(
            categorical_output,
            torch.Tensor,
        ), (
            f"{dataset_name}: categorical output "
            f"{feature_index} must be a tensor."
        )

        assert tuple(
            categorical_output.shape
        ) == expected_shape, (
            f"{dataset_name}: categorical output "
            f"{feature_index} shape mismatch."
        )

        assert torch.isfinite(
            categorical_output
        ).all().item(), (
            f"{dataset_name}: categorical output "
            f"{feature_index} contains non-finite values."
        )

        # ---------------------------------------------------------------------------------------
        # Probability representation
        # ---------------------------------------------------------------------------------------

        probabilities = F.softmax(
            categorical_output,
            dim=-1,
        )

        assert torch.isfinite(
            probabilities
        ).all().item(), (
            f"{dataset_name}: categorical probability "
            f"{feature_index} contains non-finite values."
        )

        assert torch.all(
            probabilities >= 0
        ).item(), (
            f"{dataset_name}: categorical probability "
            f"{feature_index} contains negative values."
        )

        probability_sums = probabilities.sum(
            dim=-1
        )

        assert torch.allclose(
            probability_sums,
            torch.ones_like(
                probability_sums
            ),
            atol=1e-5,
            rtol=1e-5,
        ), (
            f"{dataset_name}: categorical probability "
            f"{feature_index} does not sum to 1."
        )

        categorical_probability_outputs.append(
            probabilities
        )

    # -------------------------------------------------------------------------------------------
    # 3.11 Assemble differentiable transformed representation
    # -------------------------------------------------------------------------------------------

    representation_parts = []

    if num_numerical > 0:

        representation_parts.append(
            numerical_output
        )

    representation_parts.extend(
        categorical_probability_outputs
    )

    assert len(
        representation_parts
    ) > 0, (
        f"{dataset_name}: no representation "
        "components available."
    )

    synthetic_representation = torch.cat(
        representation_parts,
        dim=1,
    )

    # -------------------------------------------------------------------------------------------
    # 3.12 Validate transformed representation
    # -------------------------------------------------------------------------------------------

    assert synthetic_representation.ndim == 2, (
        f"{dataset_name}: transformed representation "
        "must be two-dimensional."
    )

    assert tuple(
        synthetic_representation.shape
    ) == (
        TEST_BATCH_SIZE,
        transformed_dim,
    ), (
        f"{dataset_name}: transformed representation "
        "shape mismatch."
    )

    assert torch.isfinite(
        synthetic_representation
    ).all().item(), (
        f"{dataset_name}: transformed representation "
        "contains non-finite values."
    )

    assert (
        synthetic_representation.requires_grad
    ), (
        f"{dataset_name}: transformed representation "
        "is not differentiable."
    )

    # -------------------------------------------------------------------------------------------
    # 3.13 Critic forward pass
    # -------------------------------------------------------------------------------------------

    critic_score = critic(
        synthetic_representation
    )

    assert isinstance(
        critic_score,
        torch.Tensor,
    ), (
        f"{dataset_name}: critic output "
        "must be a tensor."
    )

    assert tuple(
        critic_score.shape
    ) == (
        TEST_BATCH_SIZE,
    ), (
        f"{dataset_name}: critic output "
        "shape mismatch."
    )

    assert torch.isfinite(
        critic_score
    ).all().item(), (
        f"{dataset_name}: critic output "
        "contains non-finite values."
    )

    # -------------------------------------------------------------------------------------------
    # 3.14 Construct scalar differentiable loss
    # -------------------------------------------------------------------------------------------

    loss = -critic_score.mean()

    assert loss.ndim == 0, (
        f"{dataset_name}: gradient validation "
        "loss must be scalar."
    )

    assert loss.requires_grad, (
        f"{dataset_name}: gradient validation "
        "loss does not require gradients."
    )

    assert torch.isfinite(
        loss
    ).item(), (
        f"{dataset_name}: gradient validation "
        "loss is non-finite."
    )

    # -------------------------------------------------------------------------------------------
    # 3.15 Backward propagation
    # -------------------------------------------------------------------------------------------

    loss.backward()

    # -------------------------------------------------------------------------------------------
    # 3.16 Analyze generator gradients
    # -------------------------------------------------------------------------------------------

    generator_gradient_analysis = (
        analyze_gradients(
            model=generator,
            model_name="generator",
            dataset_name=dataset_name,
        )
    )

    # -------------------------------------------------------------------------------------------
    # 3.17 Analyze critic gradients
    # -------------------------------------------------------------------------------------------

    critic_gradient_analysis = (
        analyze_gradients(
            model=critic,
            model_name="critic",
            dataset_name=dataset_name,
        )
    )

    # -------------------------------------------------------------------------------------------
    # 3.18 Validate latent gradient
    # -------------------------------------------------------------------------------------------

    assert z.grad is not None, (
        f"{dataset_name}: latent variable "
        "received no gradient."
    )

    assert torch.isfinite(
        z.grad
    ).all().item(), (
        f"{dataset_name}: latent gradient "
        "contains non-finite values."
    )

    latent_gradient_norm = float(
        torch.linalg.vector_norm(
            z.grad.detach()
        ).cpu()
    )

    assert math.isfinite(
        latent_gradient_norm
    ), (
        f"{dataset_name}: latent gradient "
        "norm is non-finite."
    )

    assert latent_gradient_norm > 0.0, (
        f"{dataset_name}: latent gradient "
        "norm is zero."
    )

    # -------------------------------------------------------------------------------------------
    # 3.19 Record results
    # -------------------------------------------------------------------------------------------

    GRADIENT_RESULTS.append({

        "dataset": dataset_name,

        "batch_size": TEST_BATCH_SIZE,

        "latent_dim": latent_dim,

        "conditional_dim": conditional_dim,

        "conditional_test": conditional_test,

        "num_numerical": num_numerical,

        "num_categorical": num_categorical,

        "transformed_dim": transformed_dim,

        "loss": float(
            loss.detach().cpu()
        ),

        "generator_trainable_parameters": (
            generator_gradient_analysis[
                "trainable_parameters"
            ]
        ),

        "generator_parameters_with_grad": (
            generator_gradient_analysis[
                "parameters_with_grad"
            ]
        ),

        "generator_parameters_without_grad": (
            generator_gradient_analysis[
                "parameters_without_grad"
            ]
        ),

        "generator_parameters_with_nonzero_grad": (
            generator_gradient_analysis[
                "parameters_with_nonzero_grad"
            ]
        ),

        "generator_gradient_coverage": (
            generator_gradient_analysis[
                "gradient_coverage"
            ]
        ),

        "generator_nonzero_parameter_coverage": (
            generator_gradient_analysis[
                "nonzero_parameter_coverage"
            ]
        ),

        "generator_gradient_elements": (
            generator_gradient_analysis[
                "gradient_elements"
            ]
        ),

        "generator_finite_gradient_elements": (
            generator_gradient_analysis[
                "finite_gradient_elements"
            ]
        ),

        "generator_nonzero_gradient_elements": (
            generator_gradient_analysis[
                "nonzero_gradient_elements"
            ]
        ),

        "generator_finite_element_coverage": (
            generator_gradient_analysis[
                "finite_element_coverage"
            ]
        ),

        "generator_nonzero_element_coverage": (
            generator_gradient_analysis[
                "nonzero_element_coverage"
            ]
        ),

        "generator_gradient_norm": (
            generator_gradient_analysis[
                "gradient_norm"
            ]
        ),

        "critic_trainable_parameters": (
            critic_gradient_analysis[
                "trainable_parameters"
            ]
        ),

        "critic_parameters_with_grad": (
            critic_gradient_analysis[
                "parameters_with_grad"
            ]
        ),

        "critic_parameters_without_grad": (
            critic_gradient_analysis[
                "parameters_without_grad"
            ]
        ),

        "critic_parameters_with_nonzero_grad": (
            critic_gradient_analysis[
                "parameters_with_nonzero_grad"
            ]
        ),

        "critic_gradient_coverage": (
            critic_gradient_analysis[
                "gradient_coverage"
            ]
        ),

        "critic_nonzero_parameter_coverage": (
            critic_gradient_analysis[
                "nonzero_parameter_coverage"
            ]
        ),

        "critic_gradient_elements": (
            critic_gradient_analysis[
                "gradient_elements"
            ]
        ),

        "critic_finite_gradient_elements": (
            critic_gradient_analysis[
                "finite_gradient_elements"
            ]
        ),

        "critic_nonzero_gradient_elements": (
            critic_gradient_analysis[
                "nonzero_gradient_elements"
            ]
        ),

        "critic_finite_element_coverage": (
            critic_gradient_analysis[
                "finite_element_coverage"
            ]
        ),

        "critic_nonzero_element_coverage": (
            critic_gradient_analysis[
                "nonzero_element_coverage"
            ]
        ),

        "critic_gradient_norm": (
            critic_gradient_analysis[
                "gradient_norm"
            ]
        ),

        "latent_gradient_norm": (
            latent_gradient_norm
        ),

        "generator_gradients_finite": True,

        "critic_gradients_finite": True,

        "latent_gradient_valid": True,

        "gradient_test": "PASS",
    })

    # -------------------------------------------------------------------------------------------
    # 3.20 Display results
    # -------------------------------------------------------------------------------------------

    print(
        f"  Loss                       : "
        f"{float(loss.detach().cpu()):.6f}"
    )

    print(
        f"  Generator gradient coverage: "
        f"{generator_gradient_analysis['parameters_with_grad']}/"
        f"{generator_gradient_analysis['trainable_parameters']}"
    )

    print(
        f"  Generator non-zero params : "
        f"{generator_gradient_analysis['parameters_with_nonzero_grad']}/"
        f"{generator_gradient_analysis['trainable_parameters']}"
    )

    print(
        f"  Generator gradient norm   : "
        f"{generator_gradient_analysis['gradient_norm']:.6e}"
    )

    print(
        f"  Critic gradient coverage  : "
        f"{critic_gradient_analysis['parameters_with_grad']}/"
        f"{critic_gradient_analysis['trainable_parameters']}"
    )

    print(
        f"  Critic non-zero params    : "
        f"{critic_gradient_analysis['parameters_with_nonzero_grad']}/"
        f"{critic_gradient_analysis['trainable_parameters']}"
    )

    print(
        f"  Critic gradient norm      : "
        f"{critic_gradient_analysis['gradient_norm']:.6e}"
    )

    print(
        f"  Latent gradient norm      : "
        f"{latent_gradient_norm:.6e}"
    )

    print(
        "  Generator gradients      : PASS"
    )

    print(
        "  Critic gradients         : PASS"
    )

    print(
        "  Finite gradients         : PASS"
    )

    print(
        "  Latent gradient          : PASS"
    )

    print(
        "  Gradient validation      : PASS"
    )

    # -------------------------------------------------------------------------------------------
    # 3.21 Clear gradients after validation
    # -------------------------------------------------------------------------------------------

    generator.zero_grad(
        set_to_none=True
    )

    critic.zero_grad(
        set_to_none=True
    )

    # -------------------------------------------------------------------------------------------
    # 3.22 Restore original model states
    # -------------------------------------------------------------------------------------------

    generator.train(
        generator_training_state
    )

    critic.train(
        critic_training_state
    )


# -----------------------------------------------------------------------------------------------
# 4. Create validation DataFrame
# -----------------------------------------------------------------------------------------------

GRADIENT_VALIDATION_DF = pd.DataFrame(
    GRADIENT_RESULTS
)

EXPECTED_COLUMNS = [
    "dataset",
    "batch_size",
    "latent_dim",
    "conditional_dim",
    "conditional_test",
    "num_numerical",
    "num_categorical",
    "transformed_dim",
    "loss",
    "generator_trainable_parameters",
    "generator_parameters_with_grad",
    "generator_parameters_without_grad",
    "generator_parameters_with_nonzero_grad",
    "generator_gradient_coverage",
    "generator_nonzero_parameter_coverage",
    "generator_gradient_elements",
    "generator_finite_gradient_elements",
    "generator_nonzero_gradient_elements",
    "generator_finite_element_coverage",
    "generator_nonzero_element_coverage",
    "generator_gradient_norm",
    "critic_trainable_parameters",
    "critic_parameters_with_grad",
    "critic_parameters_without_grad",
    "critic_parameters_with_nonzero_grad",
    "critic_gradient_coverage",
    "critic_nonzero_parameter_coverage",
    "critic_gradient_elements",
    "critic_finite_gradient_elements",
    "critic_nonzero_gradient_elements",
    "critic_finite_element_coverage",
    "critic_nonzero_element_coverage",
    "critic_gradient_norm",
    "latent_gradient_norm",
    "generator_gradients_finite",
    "critic_gradients_finite",
    "latent_gradient_valid",
    "gradient_test",
]

assert list(
    GRADIENT_VALIDATION_DF.columns
) == EXPECTED_COLUMNS, (
    "Gradient validation schema mismatch."
)


# -----------------------------------------------------------------------------------------------
# 5. Validate dataset coverage
# -----------------------------------------------------------------------------------------------

assert len(
    GRADIENT_VALIDATION_DF
) == len(
    EXPECTED_DATASETS
), (
    "Gradient validation did not produce exactly "
    "one result per dataset."
)

assert set(
    GRADIENT_VALIDATION_DF["dataset"]
) == set(
    EXPECTED_DATASETS
), (
    "Gradient validation dataset coverage mismatch."
)


# -----------------------------------------------------------------------------------------------
# 6. Validate gradient coverage
# -----------------------------------------------------------------------------------------------

assert (
    GRADIENT_VALIDATION_DF[
        "generator_parameters_with_grad"
    ]
    ==
    GRADIENT_VALIDATION_DF[
        "generator_trainable_parameters"
    ]
).all(), (
    "Not all trainable generator parameters "
    "received gradients."
)

assert (
    GRADIENT_VALIDATION_DF[
        "critic_parameters_with_grad"
    ]
    ==
    GRADIENT_VALIDATION_DF[
        "critic_trainable_parameters"
    ]
).all(), (
    "Not all trainable critic parameters "
    "received gradients."
)


# -----------------------------------------------------------------------------------------------
# 7. Validate finite gradients
# -----------------------------------------------------------------------------------------------

assert (
    GRADIENT_VALIDATION_DF[
        "generator_gradients_finite"
    ]
).all(), (
    "Non-finite generator gradients detected."
)

assert (
    GRADIENT_VALIDATION_DF[
        "critic_gradients_finite"
    ]
).all(), (
    "Non-finite critic gradients detected."
)

assert (
    GRADIENT_VALIDATION_DF[
        "latent_gradient_valid"
    ]
).all(), (
    "Invalid latent gradients detected."
)


# -----------------------------------------------------------------------------------------------
# 8. Validate positive gradient norms
# -----------------------------------------------------------------------------------------------

assert (
    GRADIENT_VALIDATION_DF[
        "generator_gradient_norm"
    ] > 0
).all(), (
    "One or more generator gradient norms are zero."
)

assert (
    GRADIENT_VALIDATION_DF[
        "critic_gradient_norm"
    ] > 0
).all(), (
    "One or more critic gradient norms are zero."
)

assert (
    GRADIENT_VALIDATION_DF[
        "latent_gradient_norm"
    ] > 0
).all(), (
    "One or more latent gradient norms are zero."
)


# -----------------------------------------------------------------------------------------------
# 9. Validate gradient element coverage
# -----------------------------------------------------------------------------------------------

assert (
    GRADIENT_VALIDATION_DF[
        "generator_finite_element_coverage"
    ] == 1.0
).all(), (
    "Generator finite gradient element coverage "
    "is incomplete."
)

assert (
    GRADIENT_VALIDATION_DF[
        "critic_finite_element_coverage"
    ] == 1.0
).all(), (
    "Critic finite gradient element coverage "
    "is incomplete."
)


# -----------------------------------------------------------------------------------------------
# 10. Validate final test status
# -----------------------------------------------------------------------------------------------

assert (
    GRADIENT_VALIDATION_DF[
        "gradient_test"
    ] == "PASS"
).all(), (
    "One or more gradient validation tests failed."
)


# -----------------------------------------------------------------------------------------------
# 11. Save validation artifact
# -----------------------------------------------------------------------------------------------

gradient_path = (
    DIRS["validation"]
    / "sppgan_gradient_validation.csv"
)

GRADIENT_VALIDATION_DF.to_csv(
    gradient_path,
    index=False,
)

assert gradient_path.exists(), (
    f"Gradient validation artifact was not created: "
    f"{gradient_path}"
)

assert gradient_path.stat().st_size > 0, (
    f"Gradient validation artifact is empty: "
    f"{gradient_path}"
)


# -----------------------------------------------------------------------------------------------
# 12. Final validation counts
# -----------------------------------------------------------------------------------------------

dataset_count = len(
    GRADIENT_VALIDATION_DF
)

gradient_pass_count = int(
    GRADIENT_VALIDATION_DF[
        "gradient_test"
    ].eq("PASS").sum()
)

generator_coverage_count = int(
    (
        GRADIENT_VALIDATION_DF[
            "generator_parameters_with_grad"
        ]
        ==
        GRADIENT_VALIDATION_DF[
            "generator_trainable_parameters"
        ]
    ).sum()
)

critic_coverage_count = int(
    (
        GRADIENT_VALIDATION_DF[
            "critic_parameters_with_grad"
        ]
        ==
        GRADIENT_VALIDATION_DF[
            "critic_trainable_parameters"
        ]
    ).sum()
)

generator_finite_count = int(
    GRADIENT_VALIDATION_DF[
        "generator_gradients_finite"
    ].sum()
)

critic_finite_count = int(
    GRADIENT_VALIDATION_DF[
        "critic_gradients_finite"
    ].sum()
)

latent_gradient_count = int(
    GRADIENT_VALIDATION_DF[
        "latent_gradient_valid"
    ].sum()
)


# -----------------------------------------------------------------------------------------------
# 13. Final Section 21 status
# -----------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 21 VALIDATION SUMMARY")
print("=" * 100)

print(
    f"Datasets tested               : "
    f"{dataset_count}"
)

print(
    f"Gradient validation           : "
    f"{gradient_pass_count}/{dataset_count} PASS"
)

print(
    f"Generator gradient coverage   : "
    f"{generator_coverage_count}/{dataset_count} PASS"
)

print(
    f"Critic gradient coverage      : "
    f"{critic_coverage_count}/{dataset_count} PASS"
)

print(
    f"Finite generator gradients    : "
    f"{generator_finite_count}/{dataset_count} PASS"
)

print(
    f"Finite critic gradients       : "
    f"{critic_finite_count}/{dataset_count} PASS"
)

print(
    f"Latent gradient propagation   : "
    f"{latent_gradient_count}/{dataset_count} PASS"
)

print(
    f"\n✓ Saved : {gradient_path}"
)

print(
    "✓ Generator gradient coverage validated."
)

print(
    "✓ Critic gradient coverage validated."
)

print(
    "✓ Generator gradient finiteness validated."
)

print(
    "✓ Critic gradient finiteness validated."
)

print(
    "✓ Latent gradient propagation validated."
)

print(
    "✓ Positive aggregate gradient norms validated."
)

print(
    "\nSECTION 21 STATUS: PASS"
)

print("=" * 100)

21. GRADIENT VALIDATION

----------------------------------------------------------------------------------------------------
Dataset : adult_income
----------------------------------------------------------------------------------------------------
  Loss                       : 0.026841
  Generator gradient coverage: 22/22
  Generator non-zero params : 22/22
  Generator gradient norm   : 5.305276e-02
  Critic gradient coverage  : 6/6
  Critic non-zero params    : 6/6
  Critic gradient norm      : 1.298006e+00
  Latent gradient norm      : 8.562352e-04
  Generator gradients      : PASS
  Critic gradients         : PASS
  Finite gradients         : PASS
  Latent gradient          : PASS
  Gradient validation      : PASS

----------------------------------------------------------------------------------------------------
Dataset : bank_marketing
----------------------------------------------------------------------------------------------------
  Loss                       : 0.108904
  

In [61]:
# ==================================================================================================
# 22. SAVE & VERIFY ARCHITECTURE ARTIFACTS
# ==================================================================================================

print("=" * 100)
print("22. SAVE & VERIFY ARCHITECTURE ARTIFACTS")
print("=" * 100)

ARCHITECTURE_ARTIFACTS = []

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

assert set(SPPGAN_ARCHITECTURES.keys()) == set(EXPECTED_DATASETS), (
    "SPPGAN_ARCHITECTURES dataset coverage mismatch."
)

assert set(SPPGAN_MODELS.keys()) == set(EXPECTED_DATASETS), (
    "SPPGAN_MODELS dataset coverage mismatch."
)


# -----------------------------------------------------------------------------------------------
# 1. SHA-256 helper
# -----------------------------------------------------------------------------------------------

def sha256_file(path):

    path = Path(path)

    assert path.exists(), (
        f"Cannot hash missing file: {path}"
    )

    sha256 = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            sha256.update(chunk)

    return sha256.hexdigest()


# -----------------------------------------------------------------------------------------------
# 2. State-dictionary validation helper
# -----------------------------------------------------------------------------------------------

def validate_state_dict(
    model,
    state_dict,
    model_name,
    dataset_name,
):

    current_state = model.state_dict()

    assert set(state_dict.keys()) == set(current_state.keys()), (
        f"{dataset_name}: {model_name} state-dict parameter keys do not match."
    )

    for parameter_name in current_state:

        saved_tensor = state_dict[parameter_name]
        current_tensor = current_state[parameter_name]

        assert tuple(saved_tensor.shape) == tuple(current_tensor.shape), (
            f"{dataset_name}: {model_name} parameter shape mismatch "
            f"for '{parameter_name}'."
        )

        assert torch.isfinite(saved_tensor).all().item(), (
            f"{dataset_name}: {model_name} persisted parameter "
            f"'{parameter_name}' contains non-finite values."
        )

        assert torch.equal(
            saved_tensor.cpu(),
            current_tensor.detach().cpu(),
        ), (
            f"{dataset_name}: {model_name} persisted parameter "
            f"'{parameter_name}' does not match the current model."
        )

    return True


# -----------------------------------------------------------------------------------------------
# 3. Architecture metadata validation helper
# -----------------------------------------------------------------------------------------------

def validate_architecture_metadata(
    metadata,
    architecture,
    dataset_name,
):

    # Only fields that are authoritative in the Section 17
    # architecture registry are required here.
    required_keys = [
        "dataset",
        "num_numerical",
        "num_categorical",
        "categorical_cardinalities",
        "transformed_dim",
        "latent_dim",
        "conditional_dim",
    ]

    missing_keys = [
        key
        for key in required_keys
        if key not in metadata
    ]

    assert not missing_keys, (
        f"{dataset_name}: architecture metadata "
        f"is missing keys: {missing_keys}"
    )

    # -------------------------------------------------------------------------------------------
    # Dataset identity
    # -------------------------------------------------------------------------------------------

    assert metadata["dataset"] == dataset_name, (
        f"{dataset_name}: metadata dataset identity mismatch."
    )

    # -------------------------------------------------------------------------------------------
    # Numerical dimensions
    # -------------------------------------------------------------------------------------------

    assert int(metadata["num_numerical"]) == int(
        architecture["num_numerical"]
    ), (
        f"{dataset_name}: numerical feature count mismatch."
    )

    # -------------------------------------------------------------------------------------------
    # Categorical dimensions
    # -------------------------------------------------------------------------------------------

    assert int(metadata["num_categorical"]) == int(
        architecture["num_categorical"]
    ), (
        f"{dataset_name}: categorical feature count mismatch."
    )

    # -------------------------------------------------------------------------------------------
    # Categorical cardinalities
    # -------------------------------------------------------------------------------------------

    persisted_cardinalities = [
        int(value)
        for value in metadata["categorical_cardinalities"]
    ]

    authoritative_cardinalities = [
        int(value)
        for value in architecture["categorical_cardinalities"]
    ]

    assert persisted_cardinalities == authoritative_cardinalities, (
        f"{dataset_name}: categorical cardinalities mismatch."
    )

    # -------------------------------------------------------------------------------------------
    # Transformed dimension
    # -------------------------------------------------------------------------------------------

    assert int(metadata["transformed_dim"]) == int(
        architecture["transformed_dim"]
    ), (
        f"{dataset_name}: transformed dimension mismatch."
    )

    # -------------------------------------------------------------------------------------------
    # Latent dimension
    # -------------------------------------------------------------------------------------------

    assert int(metadata["latent_dim"]) == int(
        architecture["latent_dim"]
    ), (
        f"{dataset_name}: latent dimension mismatch."
    )

    # -------------------------------------------------------------------------------------------
    # Conditional dimension
    # -------------------------------------------------------------------------------------------

    assert int(
        metadata.get("conditional_dim", 0)
    ) == int(
        architecture.get("conditional_dim", 0)
    ), (
        f"{dataset_name}: conditional dimension mismatch."
    )

    # -------------------------------------------------------------------------------------------
    # Framework / notebook metadata
    # -------------------------------------------------------------------------------------------

    assert metadata["framework"] == FRAMEWORK_NAME, (
        f"{dataset_name}: framework metadata mismatch."
    )

    assert metadata["notebook"] == NOTEBOOK_ID, (
        f"{dataset_name}: notebook metadata mismatch."
    )

    assert metadata["research_title"] == RESEARCH_TITLE, (
        f"{dataset_name}: research title metadata mismatch."
    )

    # -------------------------------------------------------------------------------------------
    # Summary objects
    # -------------------------------------------------------------------------------------------

    summary_fields = [
        "config",
        "input_representation",
        "latent_space",
        "numerical_output",
        "categorical_output",
        "loss_interface",
        "statistical_guidance",
        "privacy_interface",
    ]

    for field in summary_fields:

        assert isinstance(
            metadata[field],
            dict,
        ), (
            f"{dataset_name}: persisted '{field}' "
            "summary must be a dictionary."
        )

    # -------------------------------------------------------------------------------------------
    # Creation timestamp
    # -------------------------------------------------------------------------------------------

    assert isinstance(
        metadata["created_utc"],
        str,
    ) and metadata["created_utc"], (
        f"{dataset_name}: invalid creation timestamp."
    )

    return True


# -----------------------------------------------------------------------------------------------
# 4. Save and verify artifacts
# -----------------------------------------------------------------------------------------------

for dataset_name in EXPECTED_DATASETS:

    print("\n" + "-" * 100)
    print(f"Dataset : {dataset_name}")
    print("-" * 100)

    architecture = SPPGAN_ARCHITECTURES[dataset_name]
    models = SPPGAN_MODELS[dataset_name]

    generator = models["generator"]
    critic = models["critic"]

    # -------------------------------------------------------------------------------------------
    # 4.1 Dataset model directory
    # -------------------------------------------------------------------------------------------

    dataset_model_dir = (
        DIRS["models"] / dataset_name
    )

    dataset_model_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    assert dataset_model_dir.exists(), (
        f"{dataset_name}: model directory could not be created."
    )

    # -------------------------------------------------------------------------------------------
    # 4.2 Artifact paths
    # -------------------------------------------------------------------------------------------

    generator_path = (
        dataset_model_dir / "sppgan_generator_state.pt"
    )

    critic_path = (
        dataset_model_dir / "sppgan_critic_state.pt"
    )

    architecture_path = (
        dataset_model_dir / "sppgan_architecture.json"
    )

    # -------------------------------------------------------------------------------------------
    # 4.3 Model validation
    # -------------------------------------------------------------------------------------------

    assert isinstance(
        generator,
        torch.nn.Module,
    ), (
        f"{dataset_name}: generator is not a torch.nn.Module."
    )

    assert isinstance(
        critic,
        torch.nn.Module,
    ), (
        f"{dataset_name}: critic is not a torch.nn.Module."
    )

    # -------------------------------------------------------------------------------------------
    # 4.4 Save state dictionaries
    # -------------------------------------------------------------------------------------------

    torch.save(
        generator.state_dict(),
        generator_path,
    )

    torch.save(
        critic.state_dict(),
        critic_path,
    )

    # -------------------------------------------------------------------------------------------
    # 4.5 Verify files exist and are non-empty
    # -------------------------------------------------------------------------------------------

    assert generator_path.exists(), (
        f"{dataset_name}: generator state file was not created."
    )

    assert critic_path.exists(), (
        f"{dataset_name}: critic state file was not created."
    )

    assert generator_path.stat().st_size > 0, (
        f"{dataset_name}: generator state file is empty."
    )

    assert critic_path.stat().st_size > 0, (
        f"{dataset_name}: critic state file is empty."
    )

    # -------------------------------------------------------------------------------------------
    # 4.6 Reload state dictionaries
    # -------------------------------------------------------------------------------------------

    generator_state = torch.load(
        generator_path,
        map_location="cpu",
    )

    critic_state = torch.load(
        critic_path,
        map_location="cpu",
    )

    assert isinstance(
        generator_state,
        dict,
    ), (
        f"{dataset_name}: persisted generator state is not a dictionary."
    )

    assert isinstance(
        critic_state,
        dict,
    ), (
        f"{dataset_name}: persisted critic state is not a dictionary."
    )

    # -------------------------------------------------------------------------------------------
    # 4.7 State-dictionary round-trip verification
    # -------------------------------------------------------------------------------------------

    validate_state_dict(
        model=generator,
        state_dict=generator_state,
        model_name="generator",
        dataset_name=dataset_name,
    )

    validate_state_dict(
        model=critic,
        state_dict=critic_state,
        model_name="critic",
        dataset_name=dataset_name,
    )

    # -------------------------------------------------------------------------------------------
    # 4.8 Create architecture metadata
    # -------------------------------------------------------------------------------------------

    architecture_metadata = {
        **architecture,

        "framework": FRAMEWORK_NAME,

        "notebook": NOTEBOOK_ID,

        "research_title": RESEARCH_TITLE,

        "config": SPPGAN_CONFIG,

        "input_representation": (
            SPPGANInputRepresentation(
                architecture["num_numerical"],
                architecture["num_categorical"],
                architecture["categorical_cardinalities"],
            ).summary()
        ),

        "latent_space": LATENT_SPACE.summary(),

        "numerical_output": NUMERICAL_OUTPUT.summary(),

        "categorical_output": CATEGORICAL_OUTPUT.summary(),

        "loss_interface": LOSS_INTERFACE.summary(),

        "statistical_guidance": STATISTICAL_GUIDANCE.summary(),

        "privacy_interface": PRIVACY_INTERFACE.summary(),

        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),
    }

    # -------------------------------------------------------------------------------------------
    # 4.9 Validate metadata
    # -------------------------------------------------------------------------------------------

    validate_architecture_metadata(
        metadata=architecture_metadata,
        architecture=architecture,
        dataset_name=dataset_name,
    )

    # -------------------------------------------------------------------------------------------
    # 4.10 Save architecture JSON
    # -------------------------------------------------------------------------------------------

    with open(
        architecture_path,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            architecture_metadata,
            file,
            indent=2,
            ensure_ascii=False,
            default=str,
        )

    # -------------------------------------------------------------------------------------------
    # 4.11 Verify architecture JSON
    # -------------------------------------------------------------------------------------------

    assert architecture_path.exists(), (
        f"{dataset_name}: architecture JSON was not created."
    )

    assert architecture_path.stat().st_size > 0, (
        f"{dataset_name}: architecture JSON is empty."
    )

    # -------------------------------------------------------------------------------------------
    # 4.12 Reload architecture JSON
    # -------------------------------------------------------------------------------------------

    with open(
        architecture_path,
        "r",
        encoding="utf-8",
    ) as file:

        reloaded_metadata = json.load(file)

    assert isinstance(
        reloaded_metadata,
        dict,
    ), (
        f"{dataset_name}: reloaded architecture metadata "
        "is not a dictionary."
    )

    # -------------------------------------------------------------------------------------------
    # 4.13 Validate metadata round-trip
    # -------------------------------------------------------------------------------------------

    validate_architecture_metadata(
        metadata=reloaded_metadata,
        architecture=architecture,
        dataset_name=dataset_name,
    )

    # -------------------------------------------------------------------------------------------
    # 4.14 SHA-256 checksums
    # -------------------------------------------------------------------------------------------

    generator_sha256 = sha256_file(
        generator_path
    )

    critic_sha256 = sha256_file(
        critic_path
    )

    architecture_sha256 = sha256_file(
        architecture_path
    )

    assert len(generator_sha256) == 64
    assert len(critic_sha256) == 64
    assert len(architecture_sha256) == 64

    # -------------------------------------------------------------------------------------------
    # 4.15 SHA-256 recheck
    # -------------------------------------------------------------------------------------------

    assert generator_sha256 == sha256_file(
        generator_path
    ), (
        f"{dataset_name}: generator SHA-256 integrity check failed."
    )

    assert critic_sha256 == sha256_file(
        critic_path
    ), (
        f"{dataset_name}: critic SHA-256 integrity check failed."
    )

    assert architecture_sha256 == sha256_file(
        architecture_path
    ), (
        f"{dataset_name}: architecture JSON SHA-256 "
        "integrity check failed."
    )

    # -------------------------------------------------------------------------------------------
    # 4.16 Record artifacts
    # -------------------------------------------------------------------------------------------

    ARCHITECTURE_ARTIFACTS.append({
        "dataset": dataset_name,

        "generator_state_path": str(
            generator_path
        ),

        "generator_state_size_bytes": (
            generator_path.stat().st_size
        ),

        "generator_sha256": generator_sha256,

        "critic_state_path": str(
            critic_path
        ),

        "critic_state_size_bytes": (
            critic_path.stat().st_size
        ),

        "critic_sha256": critic_sha256,

        "architecture_json_path": str(
            architecture_path
        ),

        "architecture_json_size_bytes": (
            architecture_path.stat().st_size
        ),

        "architecture_json_sha256": architecture_sha256,

        "state_dict_roundtrip": "PASS",

        "metadata_roundtrip": "PASS",

        "sha256_integrity": "PASS",

        "artifact_validation": "PASS",
    })

    print(
        "  Generator state         : PASS"
    )

    print(
        "  Critic state            : PASS"
    )

    print(
        "  Generator round-trip    : PASS"
    )

    print(
        "  Critic round-trip       : PASS"
    )

    print(
        "  Architecture metadata   : PASS"
    )

    print(
        "  Metadata round-trip     : PASS"
    )

    print(
        "  SHA-256 integrity       : PASS"
    )

    print(
        f"  Generator size          : "
        f"{generator_path.stat().st_size:,} bytes"
    )

    print(
        f"  Critic size             : "
        f"{critic_path.stat().st_size:,} bytes"
    )

    print(
        f"  Metadata size           : "
        f"{architecture_path.stat().st_size:,} bytes"
    )

    print(
        "  Artifact validation     : PASS"
    )


# -----------------------------------------------------------------------------------------------
# 5. Create artifact registry
# -----------------------------------------------------------------------------------------------

ARCHITECTURE_ARTIFACTS_DF = pd.DataFrame(
    ARCHITECTURE_ARTIFACTS
)

EXPECTED_ARTIFACT_COLUMNS = [
    "dataset",
    "generator_state_path",
    "generator_state_size_bytes",
    "generator_sha256",
    "critic_state_path",
    "critic_state_size_bytes",
    "critic_sha256",
    "architecture_json_path",
    "architecture_json_size_bytes",
    "architecture_json_sha256",
    "state_dict_roundtrip",
    "metadata_roundtrip",
    "sha256_integrity",
    "artifact_validation",
]

assert list(
    ARCHITECTURE_ARTIFACTS_DF.columns
) == EXPECTED_ARTIFACT_COLUMNS, (
    "Architecture artifact registry schema mismatch."
)


# -----------------------------------------------------------------------------------------------
# 6. Validate registry coverage
# -----------------------------------------------------------------------------------------------

assert len(
    ARCHITECTURE_ARTIFACTS_DF
) == len(EXPECTED_DATASETS), (
    "Architecture artifact registry does not contain "
    "exactly one result per dataset."
)

assert set(
    ARCHITECTURE_ARTIFACTS_DF["dataset"]
) == set(EXPECTED_DATASETS), (
    "Architecture artifact registry dataset coverage mismatch."
)


# -----------------------------------------------------------------------------------------------
# 7. Validate persisted paths
# -----------------------------------------------------------------------------------------------

for column in [
    "generator_state_path",
    "critic_state_path",
    "architecture_json_path",
]:

    for artifact_path_value in ARCHITECTURE_ARTIFACTS_DF[column]:

        persisted_path = Path(
            artifact_path_value
        )

        assert persisted_path.exists(), (
            f"Persisted artifact missing: {persisted_path}"
        )

        assert persisted_path.stat().st_size > 0, (
            f"Persisted artifact is empty: {persisted_path}"
        )


# -----------------------------------------------------------------------------------------------
# 8. Validate registry statuses
# -----------------------------------------------------------------------------------------------

for column in [
    "state_dict_roundtrip",
    "metadata_roundtrip",
    "sha256_integrity",
    "artifact_validation",
]:

    assert (
        ARCHITECTURE_ARTIFACTS_DF[column] == "PASS"
    ).all(), (
        f"Architecture artifact validation failure "
        f"in column: {column}"
    )


# -----------------------------------------------------------------------------------------------
# 9. Save artifact registry
# -----------------------------------------------------------------------------------------------

artifact_path = (
    DIRS["metadata"]
    / "sppgan_architecture_artifacts.csv"
)

ARCHITECTURE_ARTIFACTS_DF.to_csv(
    artifact_path,
    index=False,
)

assert artifact_path.exists(), (
    f"Architecture artifact registry was not created: "
    f"{artifact_path}"
)

assert artifact_path.stat().st_size > 0, (
    "Architecture artifact registry is empty."
)


# -----------------------------------------------------------------------------------------------
# 10. Final validation summary
# -----------------------------------------------------------------------------------------------

dataset_count = len(
    ARCHITECTURE_ARTIFACTS_DF
)

state_roundtrip_count = int(
    (
        ARCHITECTURE_ARTIFACTS_DF[
            "state_dict_roundtrip"
        ] == "PASS"
    ).sum()
)

metadata_roundtrip_count = int(
    (
        ARCHITECTURE_ARTIFACTS_DF[
            "metadata_roundtrip"
        ] == "PASS"
    ).sum()
)

sha256_integrity_count = int(
    (
        ARCHITECTURE_ARTIFACTS_DF[
            "sha256_integrity"
        ] == "PASS"
    ).sum()
)

artifact_validation_count = int(
    (
        ARCHITECTURE_ARTIFACTS_DF[
            "artifact_validation"
        ] == "PASS"
    ).sum()
)


# -----------------------------------------------------------------------------------------------
# 11. Final Section 22 status
# -----------------------------------------------------------------------------------------------

assert dataset_count == 3
assert state_roundtrip_count == 3
assert metadata_roundtrip_count == 3
assert sha256_integrity_count == 3
assert artifact_validation_count == 3

print("\n" + "=" * 100)
print("SECTION 22 VALIDATION SUMMARY")
print("=" * 100)

print(
    f"Datasets processed            : "
    f"{dataset_count}"
)

print(
    f"State-dictionary round-trip   : "
    f"{state_roundtrip_count}/{dataset_count} PASS"
)

print(
    f"Metadata round-trip           : "
    f"{metadata_roundtrip_count}/{dataset_count} PASS"
)

print(
    f"SHA-256 integrity             : "
    f"{sha256_integrity_count}/{dataset_count} PASS"
)

print(
    f"Artifact validation           : "
    f"{artifact_validation_count}/{dataset_count} PASS"
)

print(
    f"\n✓ Artifact registry saved : "
    f"{artifact_path}"
)

print(
    "\nSECTION 22 STATUS: PASS"
)

print("=" * 100)

22. SAVE & VERIFY ARCHITECTURE ARTIFACTS

----------------------------------------------------------------------------------------------------
Dataset : adult_income
----------------------------------------------------------------------------------------------------
  Generator state         : PASS
  Critic state            : PASS
  Generator round-trip    : PASS
  Critic round-trip       : PASS
  Architecture metadata   : PASS
  Metadata round-trip     : PASS
  SHA-256 integrity       : PASS
  Generator size          : 511,365 bytes
  Critic size             : 376,037 bytes
  Metadata size           : 8,249 bytes
  Artifact validation     : PASS

----------------------------------------------------------------------------------------------------
Dataset : bank_marketing
----------------------------------------------------------------------------------------------------
  Generator state         : PASS
  Critic state            : PASS
  Generator round-trip    : PASS
  Critic round-tri

In [64]:
# ==================================================================================================
# 23. SAVE & VERIFY MODEL CONFIGURATION
# ==================================================================================================

print("=" * 100)
print("23. SAVE & VERIFY MODEL CONFIGURATION")
print("=" * 100)


# -----------------------------------------------------------------------------------------------
# 1. Configuration constants
# -----------------------------------------------------------------------------------------------

MODEL_CONFIGURATION_VERSION = "1.0"

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

EXPECTED_DOWNSTREAM_NOTEBOOKS = {
    "09": "SPP-GAN Statistical Guidance",
    "10": "SPP-GAN Differential Privacy",
    "11": "Privacy Accounting",
    "12": "SPP-GAN Training",
    "13": "Synthetic Data Generation",
    "14": "Statistical Fidelity Evaluation",
    "15": "ML Utility / TSTR Evaluation",
    "16": "Privacy & Disclosure Risk Evaluation",
    "17": "Ablation Study",
    "18": "Privacy–Fidelity–Utility Trade-off",
    "19": "Statistical Significance Testing",
    "20": "Final Comparative Analysis",
    "21": "Publication Tables & Figures",
    "22": "Reproducibility, Manifest & Final Audit",
}


# -----------------------------------------------------------------------------------------------
# 2. SHA-256 helper
# -----------------------------------------------------------------------------------------------

def sha256_file(path):

    path = Path(path)

    assert path.exists(), (
        f"Cannot calculate SHA-256: file does not exist: {path}"
    )

    sha256 = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            sha256.update(chunk)

    return sha256.hexdigest()


# -----------------------------------------------------------------------------------------------
# 3. Validate authoritative architecture registry
# -----------------------------------------------------------------------------------------------

assert isinstance(
    SPPGAN_ARCHITECTURES,
    dict,
), (
    "SPPGAN_ARCHITECTURES must be a dictionary."
)

assert set(
    SPPGAN_ARCHITECTURES.keys()
) == set(
    EXPECTED_DATASETS
), (
    "SPPGAN_ARCHITECTURES dataset coverage mismatch."
)


# -----------------------------------------------------------------------------------------------
# 4. Validate architecture fields
# -----------------------------------------------------------------------------------------------

REQUIRED_ARCHITECTURE_FIELDS = [
    "target",
    "generative_columns",
    "numerical_columns",
    "categorical_columns",
    "categorical_cardinalities",
    "num_numerical",
    "num_categorical",
    "transformed_dim",
]

for dataset_name in EXPECTED_DATASETS:

    architecture = SPPGAN_ARCHITECTURES[
        dataset_name
    ]

    missing_fields = [
        field
        for field in REQUIRED_ARCHITECTURE_FIELDS
        if field not in architecture
    ]

    assert not missing_fields, (
        f"{dataset_name}: missing architecture fields: "
        f"{missing_fields}"
    )

    # -------------------------------------------------------------------------------------------
    # Schema containers
    # -------------------------------------------------------------------------------------------

    assert isinstance(
        architecture["generative_columns"],
        list,
    ), (
        f"{dataset_name}: generative_columns must be a list."
    )

    assert isinstance(
        architecture["numerical_columns"],
        list,
    ), (
        f"{dataset_name}: numerical_columns must be a list."
    )

    assert isinstance(
        architecture["categorical_columns"],
        list,
    ), (
        f"{dataset_name}: categorical_columns must be a list."
    )

    assert isinstance(
        architecture["categorical_cardinalities"],
        list,
    ), (
        f"{dataset_name}: categorical_cardinalities must be a list."
    )

    # -------------------------------------------------------------------------------------------
    # Numerical / categorical schema validation
    # -------------------------------------------------------------------------------------------

    assert len(
        architecture["numerical_columns"]
    ) == int(
        architecture["num_numerical"]
    ), (
        f"{dataset_name}: numerical column count mismatch."
    )

    assert len(
        architecture["categorical_columns"]
    ) == int(
        architecture["num_categorical"]
    ), (
        f"{dataset_name}: categorical column count mismatch."
    )

    assert len(
        architecture["categorical_cardinalities"]
    ) == int(
        architecture["num_categorical"]
    ), (
        f"{dataset_name}: categorical cardinality count mismatch."
    )

    # -------------------------------------------------------------------------------------------
    # Cardinality validation
    # -------------------------------------------------------------------------------------------

    assert all(
        int(cardinality) > 0
        for cardinality
        in architecture[
            "categorical_cardinalities"
        ]
    ), (
        f"{dataset_name}: categorical cardinalities "
        "must all be positive."
    )

    # -------------------------------------------------------------------------------------------
    # Transformed dimension validation
    #
    # Authoritative architecture contract:
    #
    # transformed_dim =
    #     num_numerical + sum(categorical_cardinalities)
    #
    # -------------------------------------------------------------------------------------------

    expected_transformed_dim = (
        int(
            architecture["num_numerical"]
        )
        +
        sum(
            int(cardinality)
            for cardinality
            in architecture[
                "categorical_cardinalities"
            ]
        )
    )

    assert int(
        architecture["transformed_dim"]
    ) == expected_transformed_dim, (
        f"{dataset_name}: transformed dimension mismatch. "
        f"Expected {expected_transformed_dim}, "
        f"found {architecture['transformed_dim']}."
    )

    # -------------------------------------------------------------------------------------------
    # Generative-column integrity
    #
    # Do NOT impose:
    #
    # len(generative_columns)
    #     == num_numerical + num_categorical
    #
    # The dimensional contract is defined by the explicit numerical,
    # categorical and transformed-dimension fields above.
    # -------------------------------------------------------------------------------------------

    assert len(
        architecture["generative_columns"]
    ) > 0, (
        f"{dataset_name}: generative_columns cannot be empty."
    )


# -----------------------------------------------------------------------------------------------
# 5. Construct model configuration
# -----------------------------------------------------------------------------------------------

MODEL_CONFIGURATION = {
    "configuration_version": MODEL_CONFIGURATION_VERSION,

    "framework": FRAMEWORK_NAME,

    "notebook": NOTEBOOK_ID,

    "research_title": RESEARCH_TITLE,

    "project_root": str(PROJECT_ROOT),

    "master_seed": int(MASTER_SEED),

    "training_enabled": False,

    "privacy_enabled": False,

    "architecture": SPPGAN_CONFIG,

    "datasets": {
        dataset_name: {
            "target": architecture["target"],

            "generative_columns": architecture[
                "generative_columns"
            ],

            "numerical_columns": architecture[
                "numerical_columns"
            ],

            "categorical_columns": architecture[
                "categorical_columns"
            ],

            "categorical_cardinalities": [
                int(cardinality)
                for cardinality
                in architecture[
                    "categorical_cardinalities"
                ]
            ],

            "num_numerical": int(
                architecture["num_numerical"]
            ),

            "num_categorical": int(
                architecture["num_categorical"]
            ),

            "transformed_dim": int(
                architecture["transformed_dim"]
            ),
        }

        for dataset_name, architecture
        in SPPGAN_ARCHITECTURES.items()
    },

    "interfaces": {
        "loss": "SPPGANLossInterface",
        "statistical_guidance": "SPPGANStatisticalGuidance",
        "privacy": "SPPGANPrivacyInterface",
    },

    "downstream_notebooks": EXPECTED_DOWNSTREAM_NOTEBOOKS,

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}


# -----------------------------------------------------------------------------------------------
# 6. Validate top-level configuration
# -----------------------------------------------------------------------------------------------

REQUIRED_CONFIGURATION_FIELDS = [
    "configuration_version",
    "framework",
    "notebook",
    "research_title",
    "project_root",
    "master_seed",
    "training_enabled",
    "privacy_enabled",
    "architecture",
    "datasets",
    "interfaces",
    "downstream_notebooks",
    "created_utc",
]

missing_configuration_fields = [
    field
    for field in REQUIRED_CONFIGURATION_FIELDS
    if field not in MODEL_CONFIGURATION
]

assert not missing_configuration_fields, (
    "Model configuration is missing required fields: "
    f"{missing_configuration_fields}"
)


# -----------------------------------------------------------------------------------------------
# 7. Validate configuration identity
# -----------------------------------------------------------------------------------------------

assert MODEL_CONFIGURATION[
    "framework"
] == FRAMEWORK_NAME

assert MODEL_CONFIGURATION[
    "notebook"
] == NOTEBOOK_ID

assert MODEL_CONFIGURATION[
    "research_title"
] == RESEARCH_TITLE

assert MODEL_CONFIGURATION[
    "master_seed"
] == int(MASTER_SEED)

assert MODEL_CONFIGURATION[
    "training_enabled"
] is False

assert MODEL_CONFIGURATION[
    "privacy_enabled"
] is False


# -----------------------------------------------------------------------------------------------
# 8. Validate dataset configuration
# -----------------------------------------------------------------------------------------------

assert set(
    MODEL_CONFIGURATION["datasets"].keys()
) == set(
    EXPECTED_DATASETS
), (
    "Model configuration dataset coverage mismatch."
)


for dataset_name in EXPECTED_DATASETS:

    config = MODEL_CONFIGURATION[
        "datasets"
    ][dataset_name]

    architecture = SPPGAN_ARCHITECTURES[
        dataset_name
    ]

    assert config["target"] == architecture[
        "target"
    ]

    assert config[
        "generative_columns"
    ] == architecture[
        "generative_columns"
    ]

    assert config[
        "numerical_columns"
    ] == architecture[
        "numerical_columns"
    ]

    assert config[
        "categorical_columns"
    ] == architecture[
        "categorical_columns"
    ]

    assert config[
        "categorical_cardinalities"
    ] == [
        int(cardinality)
        for cardinality
        in architecture[
            "categorical_cardinalities"
        ]
    ]

    assert config[
        "num_numerical"
    ] == int(
        architecture["num_numerical"]
    )

    assert config[
        "num_categorical"
    ] == int(
        architecture["num_categorical"]
    )

    assert config[
        "transformed_dim"
    ] == int(
        architecture["transformed_dim"]
    )


# -----------------------------------------------------------------------------------------------
# 9. Validate interfaces
# -----------------------------------------------------------------------------------------------

EXPECTED_INTERFACES = {
    "loss": "SPPGANLossInterface",
    "statistical_guidance": "SPPGANStatisticalGuidance",
    "privacy": "SPPGANPrivacyInterface",
}

assert MODEL_CONFIGURATION[
    "interfaces"
] == EXPECTED_INTERFACES, (
    "Model interface registry mismatch."
)


# -----------------------------------------------------------------------------------------------
# 10. Validate downstream notebook registry
# -----------------------------------------------------------------------------------------------

assert MODEL_CONFIGURATION[
    "downstream_notebooks"
] == EXPECTED_DOWNSTREAM_NOTEBOOKS, (
    "Downstream notebook registry mismatch."
)


# -----------------------------------------------------------------------------------------------
# 11. Validate JSON serializability
# -----------------------------------------------------------------------------------------------

try:

    configuration_json = json.dumps(
        MODEL_CONFIGURATION,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

except Exception as error:

    raise AssertionError(
        "Model configuration is not JSON serializable."
    ) from error

assert len(
    configuration_json
) > 0


# -----------------------------------------------------------------------------------------------
# 12. Save configuration
# -----------------------------------------------------------------------------------------------

MODEL_CONFIG_PATH = (
    DIRS["config"]
    / "sppgan_model_configuration.json"
)

MODEL_CONFIG_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with open(
    MODEL_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        MODEL_CONFIGURATION,
        file,
        indent=2,
        ensure_ascii=False,
        default=str,
    )


# -----------------------------------------------------------------------------------------------
# 13. Verify persisted configuration
# -----------------------------------------------------------------------------------------------

assert MODEL_CONFIG_PATH.exists(), (
    f"Model configuration was not created: "
    f"{MODEL_CONFIG_PATH}"
)

assert MODEL_CONFIG_PATH.stat().st_size > 0, (
    "Persisted model configuration is empty."
)


# -----------------------------------------------------------------------------------------------
# 14. Reload configuration
# -----------------------------------------------------------------------------------------------

with open(
    MODEL_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as file:

    RELOADED_MODEL_CONFIGURATION = json.load(
        file
    )

assert isinstance(
    RELOADED_MODEL_CONFIGURATION,
    dict,
), (
    "Reloaded model configuration is not a dictionary."
)


# -----------------------------------------------------------------------------------------------
# 15. Round-trip validation
# -----------------------------------------------------------------------------------------------

assert RELOADED_MODEL_CONFIGURATION[
    "configuration_version"
] == MODEL_CONFIGURATION[
    "configuration_version"
]

assert RELOADED_MODEL_CONFIGURATION[
    "framework"
] == MODEL_CONFIGURATION[
    "framework"
]

assert RELOADED_MODEL_CONFIGURATION[
    "notebook"
] == MODEL_CONFIGURATION[
    "notebook"
]

assert RELOADED_MODEL_CONFIGURATION[
    "research_title"
] == MODEL_CONFIGURATION[
    "research_title"
]

assert int(
    RELOADED_MODEL_CONFIGURATION[
        "master_seed"
    ]
) == int(
    MODEL_CONFIGURATION[
        "master_seed"
    ]
)

assert RELOADED_MODEL_CONFIGURATION[
    "training_enabled"
] is False

assert RELOADED_MODEL_CONFIGURATION[
    "privacy_enabled"
] is False

assert set(
    RELOADED_MODEL_CONFIGURATION[
        "datasets"
    ].keys()
) == set(
    EXPECTED_DATASETS
)

assert RELOADED_MODEL_CONFIGURATION[
    "interfaces"
] == EXPECTED_INTERFACES

assert RELOADED_MODEL_CONFIGURATION[
    "downstream_notebooks"
] == EXPECTED_DOWNSTREAM_NOTEBOOKS


# -----------------------------------------------------------------------------------------------
# 16. SHA-256 integrity
# -----------------------------------------------------------------------------------------------

MODEL_CONFIG_SHA256 = sha256_file(
    MODEL_CONFIG_PATH
)

assert len(
    MODEL_CONFIG_SHA256
) == 64

assert MODEL_CONFIG_SHA256 == sha256_file(
    MODEL_CONFIG_PATH
), (
    "Model configuration SHA-256 integrity check failed."
)


# -----------------------------------------------------------------------------------------------
# 17. Create configuration artifact registry
# -----------------------------------------------------------------------------------------------

MODEL_CONFIGURATION_ARTIFACT = {
    "configuration_version": MODEL_CONFIGURATION_VERSION,

    "framework": FRAMEWORK_NAME,

    "notebook": NOTEBOOK_ID,

    "configuration_path": str(
        MODEL_CONFIG_PATH
    ),

    "configuration_size_bytes": (
        MODEL_CONFIG_PATH.stat().st_size
    ),

    "configuration_sha256": MODEL_CONFIG_SHA256,

    "dataset_count": len(
        EXPECTED_DATASETS
    ),

    "dataset_coverage": "PASS",

    "schema_validation": "PASS",

    "json_roundtrip": "PASS",

    "sha256_integrity": "PASS",

    "artifact_validation": "PASS",

    "created_utc": MODEL_CONFIGURATION[
        "created_utc"
    ],
}


MODEL_CONFIGURATION_ARTIFACT_DF = pd.DataFrame([
    MODEL_CONFIGURATION_ARTIFACT
])


# -----------------------------------------------------------------------------------------------
# 18. Validate artifact registry
# -----------------------------------------------------------------------------------------------

EXPECTED_ARTIFACT_COLUMNS = [
    "configuration_version",
    "framework",
    "notebook",
    "configuration_path",
    "configuration_size_bytes",
    "configuration_sha256",
    "dataset_count",
    "dataset_coverage",
    "schema_validation",
    "json_roundtrip",
    "sha256_integrity",
    "artifact_validation",
    "created_utc",
]

assert list(
    MODEL_CONFIGURATION_ARTIFACT_DF.columns
) == EXPECTED_ARTIFACT_COLUMNS, (
    "Model configuration artifact registry schema mismatch."
)

assert len(
    MODEL_CONFIGURATION_ARTIFACT_DF
) == 1

assert int(
    MODEL_CONFIGURATION_ARTIFACT_DF.loc[
        0,
        "dataset_count"
    ]
) == 3

for column in [
    "dataset_coverage",
    "schema_validation",
    "json_roundtrip",
    "sha256_integrity",
    "artifact_validation",
]:

    assert (
        MODEL_CONFIGURATION_ARTIFACT_DF.loc[
            0,
            column
        ] == "PASS"
    ), (
        f"Model configuration validation failed: {column}"
    )


# -----------------------------------------------------------------------------------------------
# 19. Save artifact registry
# -----------------------------------------------------------------------------------------------

MODEL_CONFIGURATION_ARTIFACT_PATH = (
    DIRS["metadata"]
    / "sppgan_model_configuration_artifact.csv"
)

MODEL_CONFIGURATION_ARTIFACT_DF.to_csv(
    MODEL_CONFIGURATION_ARTIFACT_PATH,
    index=False,
)

assert MODEL_CONFIGURATION_ARTIFACT_PATH.exists()

assert MODEL_CONFIGURATION_ARTIFACT_PATH.stat().st_size > 0


# -----------------------------------------------------------------------------------------------
# 20. Final validation summary
# -----------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("MODEL CONFIGURATION VALIDATION SUMMARY")
print("-" * 100)

print(
    f"Configuration version       : "
    f"{MODEL_CONFIGURATION_VERSION}"
)

print(
    f"Datasets registered         : "
    f"{len(EXPECTED_DATASETS)}"
)

print(
    "Dataset coverage            : PASS"
)

print(
    "Schema validation           : PASS"
)

print(
    "JSON round-trip             : PASS"
)

print(
    "SHA-256 integrity           : PASS"
)

print(
    f"Configuration size          : "
    f"{MODEL_CONFIG_PATH.stat().st_size:,} bytes"
)

print(
    f"Configuration SHA-256       : "
    f"{MODEL_CONFIG_SHA256}"
)

print(
    "\n✓ Model configuration saved:"
)

print(
    f"  {MODEL_CONFIG_PATH}"
)

print(
    "\n✓ Configuration registry saved:"
)

print(
    f"  {MODEL_CONFIGURATION_ARTIFACT_PATH}"
)

print("\n" + "=" * 100)
print("SECTION 23 STATUS: PASS")
print("=" * 100)

23. SAVE & VERIFY MODEL CONFIGURATION

----------------------------------------------------------------------------------------------------
MODEL CONFIGURATION VALIDATION SUMMARY
----------------------------------------------------------------------------------------------------
Configuration version       : 1.0
Datasets registered         : 3
Dataset coverage            : PASS
Schema validation           : PASS
JSON round-trip             : PASS
SHA-256 integrity           : PASS
Configuration size          : 7,040 bytes
Configuration SHA-256       : 06cf9f70af552708f0896ce96a566b9bcca87bfa3dcfb5a590c5dd636748de97

✓ Model configuration saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/config/sppgan_model_configuration.json

✓ Configuration registry saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/metadata/sppgan_model_configuration_artifact.csv

SECTION 23 STATUS: PASS


In [65]:
# ==================================================================================================
# 24. COMPLETION SUMMARY
# ==================================================================================================

print("=" * 100)
print("24. COMPLETION SUMMARY")
print("=" * 100)

# -----------------------------------------------------------------------------------------------
# Validation status
# -----------------------------------------------------------------------------------------------

shape_pass = (
    SHAPE_TEST_DF["shape_test"]
    .eq("PASS")
    .all()
)

forward_pass = (
    FORWARD_TEST_DF["forward_test"]
    .eq("PASS")
    .all()
)

backward_pass = (
    BACKWARD_TEST_DF["backward_test"]
    .eq("PASS")
    .all()
)

gradient_pass = (
    GRADIENT_VALIDATION_DF["gradient_test"]
    .eq("PASS")
    .all()
)

all_validation_pass = all([
    shape_pass,
    forward_pass,
    backward_pass,
    gradient_pass,
])

if not all_validation_pass:
    raise RuntimeError(
        "Notebook 08 validation failed."
    )

# -----------------------------------------------------------------------------------------------
# Completion manifest
# -----------------------------------------------------------------------------------------------

COMPLETION_MANIFEST = {
    "notebook": NOTEBOOK_ID,
    "name": NOTEBOOK_NAME,
    "framework": FRAMEWORK_NAME,

    "status": "PASS",

    "project_root": str(PROJECT_ROOT),

    "datasets_registered": len(
        SPPGAN_ARCHITECTURES
    ),

    "datasets": list(
        SPPGAN_ARCHITECTURES.keys()
    ),

    "architecture_validation": {
        "tensor_shape_tests": "PASS",
        "forward_pass_tests": "PASS",
        "backward_pass_tests": "PASS",
        "gradient_validation": "PASS",
    },

    "training_performed": False,
    "privacy_enabled": False,
    "synthetic_generation_performed": False,

    "artifacts": {
        "architecture_summary": str(
            summary_path
        ),
        "parameter_count": str(
            parameter_path
        ),
        "shape_tests": str(
            shape_path
        ),
        "forward_tests": str(
            forward_path
        ),
        "backward_tests": str(
            backward_path
        ),
        "gradient_validation": str(
            gradient_path
        ),
        "artifact_registry": str(
            artifact_path
        ),
        "model_configuration": str(
            MODEL_CONFIG_PATH
        ),
    },

    "next_notebook": {
        "number": "09",
        "name": "SPP-GAN Statistical Guidance",
    },

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

COMPLETION_PATH = (
    DIRS["validation"]
    / "sppgan_notebook_08_completion.json"
)

with open(
    COMPLETION_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        COMPLETION_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

# -----------------------------------------------------------------------------------------------
# Final display
# -----------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOK 08 — FINAL STATUS")
print("=" * 100)

print(f"Framework                 : {FRAMEWORK_NAME}")
print(f"Datasets                  : {len(SPPGAN_ARCHITECTURES)}")
print(f"Tensor shape tests        : PASS")
print(f"Forward-pass tests        : PASS")
print(f"Backward-pass tests       : PASS")
print(f"Gradient validation       : PASS")
print(f"Training                  : NOT PERFORMED")
print(f"Privacy                   : NOT ENABLED")
print(f"Synthetic generation      : NOT PERFORMED")
print(f"Overall status            : PASS")

print("\nArtifacts:")
print(f"  Architecture : {DIRS['architecture']}")
print(f"  Configuration: {DIRS['config']}")
print(f"  Models       : {DIRS['models']}")
print(f"  Metadata     : {DIRS['metadata']}")
print(f"  Validation   : {DIRS['validation']}")

print("\nNext:")
print("  Notebook 09 — SPP-GAN Statistical Guidance")

print("=" * 100)

print("\n✓ NOTEBOOK 08 COMPLETED SUCCESSFULLY.")

24. COMPLETION SUMMARY

NOTEBOOK 08 — FINAL STATUS
Framework                 : SPP-GAN
Datasets                  : 3
Tensor shape tests        : PASS
Forward-pass tests        : PASS
Backward-pass tests       : PASS
Gradient validation       : PASS
Training                  : NOT PERFORMED
Privacy                   : NOT ENABLED
Synthetic generation      : NOT PERFORMED
Overall status            : PASS

Artifacts:
  Architecture : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/architecture
  Configuration: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/config
  Models       : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/models
  Metadata     : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/metadata
  Validation   : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/validation

Next:
  Notebook 09 — SPP-GAN Statistical Guidance

✓ NOTEBOOK 08 COMPLETED SUCCESSFULLY.
